# AR Pipeline Skripsi — Notebook Induk

Gabungan seluruh notebook Colab final (Tahap 0–14) menjadi satu berkas berurutan. Kode sel **tidak diubah**; keluaran dihapus; sel yang hanya memasang Google Drive dihapus.

**Peringatan**
- Jangan **Run All**: beberapa tahap memanggil LLM berkuota (Groq/Gemini) dan mengubah artefak di Drive.
- Jalankan per bagian, urut, dan periksa keluaran sebelum lanjut.
- Notebook induk ini **belum diuji menyeluruh** sebagai satu berkas; tiap bagian berasal dari notebook yang pernah berjalan terpisah.
- Folder kerja di Drive: `Skripsi_RAG` (`00_raw` → `01_interim` → `02_processed` → … → `05_artifacts`).
- Tahap 12b, 13b, dan 14 memuat modul di `src/` (`tahap12b_rating.py`, `tahap13_softskill.py`, `tahap14_freeze.py`); salin ke `Skripsi_RAG/src/`.
- Kunci API dibaca dari Colab Secrets, bukan ditulis di notebook.

---
# Tahap 0 — Penyiapan proyek
*Sumber: `Tahap0_Skripsi.ipynb`*

# Cell 1 — Mount Google Drive

# Cell 2 — Konfigurasi path

In [ ]:
from pathlib import Path

MYDRIVE = Path("/content/drive/MyDrive")

# Cari semua file .zip yang namanya mengandung "DataSet" (tidak peka huruf besar/kecil)
found = [p for p in MYDRIVE.rglob("*.zip") if "dataset" in p.name.lower()]

if not found:
    print("Tidak ada ZIP dataset di Drive. Cek apakah upload sudah selesai.")
else:
    for i, p in enumerate(found):
        print(i, p)

In [ ]:
from pathlib import Path

# Sesuaikan jika nama folder/ZIP di Drive berbeda
PROJECT_DIR = Path("/content/drive/MyDrive/Skripsi_RAG")
ZIP_PATH    = PROJECT_DIR / "1. 261010_DataSetSkripsi.zip"

RAW_DIR       = PROJECT_DIR / "00_raw"        # hasil ekstrak ZIP, TIDAK BOLEH diubah
INTERIM_DIR   = PROJECT_DIR / "01_interim"    # hasil ekstraksi teks mentah (Tahap 2)
PROCESSED_DIR = PROJECT_DIR / "02_processed"  # data bersih & tabel terstruktur (Tahap 3-4)
KB_DIR        = PROJECT_DIR / "03_kb"         # chunk, embedding, index (Tahap 5-6)
EVAL_DIR      = PROJECT_DIR / "04_eval"       # data uji & hasil evaluasi (Tahap 7-8)
LOG_DIR       = PROJECT_DIR / "logs"          # snapshot, manifest, catatan proses

print("ZIP ditemukan:", ZIP_PATH.exists())

# Cell 3 — Buat struktur folder

In [ ]:
for d in [RAW_DIR, INTERIM_DIR, PROCESSED_DIR, KB_DIR, EVAL_DIR, LOG_DIR]:
    d.mkdir(parents=True, exist_ok=True)

for d in sorted(PROJECT_DIR.iterdir()):
    print(("[DIR] " if d.is_dir() else "      ") + d.name)

# Cell 4 — Cek integritas ZIP

In [ ]:
import zipfile, hashlib

def sha256(path, chunk=1 << 20):
    h = hashlib.sha256()
    with open(path, "rb") as fh:
        for block in iter(lambda: fh.read(chunk), b""):
            h.update(block)
    return h.hexdigest()

with zipfile.ZipFile(ZIP_PATH) as zf:
    bad = zf.testzip()                       # None = semua file utuh
    members = [m for m in zf.infolist() if not m.is_dir()]

ZIP_SHA256 = sha256(ZIP_PATH)
print("File rusak di ZIP :", bad)
print("Jumlah file di ZIP:", len(members))   # harus 86
print("SHA-256 ZIP       :", ZIP_SHA256)

# Cell 5 — Ekstrak ZIP ke 00_raw

In [ ]:
existing = [p for p in RAW_DIR.rglob("*") if p.is_file()]

if existing:
    print(f"00_raw sudah berisi {len(existing)} file, ekstraksi dilewati.")
else:
    with zipfile.ZipFile(ZIP_PATH) as zf:
        zf.extractall(RAW_DIR)
    print("Ekstraksi selesai.")

# Cell 6 — Snapshot file mentah (daftar + hash)

In [ ]:
import pandas as pd

files = sorted(p for p in RAW_DIR.rglob("*") if p.is_file())

rows = []
for i, p in enumerate(files, 1):
    rel = p.relative_to(RAW_DIR)
    rows.append({
        "file_id":   f"F{i:03d}",
        "rel_path":  str(rel),
        "folder":    str(rel.parent),
        "file_name": p.name,
        "ext":       p.suffix.lower().lstrip("."),
        "size_kb":   round(p.stat().st_size / 1024, 1),
        "sha256":    sha256(p),
    })

raw_df = pd.DataFrame(rows)
raw_df.to_csv(LOG_DIR / "raw_snapshot.csv", index=False)

print("Total file:", len(raw_df))            # harus 86
display(raw_df["ext"].value_counts())
display(raw_df.groupby("folder").size().rename("jumlah"))

# Cell 7 — Cek duplikat persis

In [ ]:
dup = raw_df[raw_df.duplicated("sha256", keep=False)].sort_values("sha256")
print("File identik byte-per-byte:", len(dup))
display(dup[["file_id", "rel_path"]])

# Cell 8 — Catat log setup

In [ ]:
import json, datetime

setup_log = {
    "tanggal_setup":  datetime.datetime.now().isoformat(timespec="seconds"),
    "zip_file":       ZIP_PATH.name,
    "zip_sha256":     ZIP_SHA256,
    "jumlah_file":    int(len(raw_df)),
    "per_ekstensi":   raw_df["ext"].value_counts().to_dict(),
    "struktur_folder": [d.name for d in [RAW_DIR, INTERIM_DIR, PROCESSED_DIR, KB_DIR, EVAL_DIR, LOG_DIR]],
}
with open(LOG_DIR / "setup_log.json", "w") as f:
    json.dump(setup_log, f, indent=2, ensure_ascii=False)

print(json.dumps(setup_log, indent=2, ensure_ascii=False))

# Cell 9 — Instal library (jalankan ulang di setiap sesi Colab baru)

In [ ]:
!pip install -q pdfplumber pymupdf python-docx openpyxl charset-normalizer
import pdfplumber, fitz, docx, openpyxl, charset_normalizer
print("Library siap.")

---
# Tahap 1 — Inventaris dokumen mentah
*Sumber: `01_inventory.ipynb`*

# Cell A — Setup sesi

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!pip install -q pdfplumber pymupdf python-docx openpyxl

from pathlib import Path
import pandas as pd

PROJECT_DIR = Path("/content/drive/MyDrive/Skripsi_RAG")
RAW_DIR     = PROJECT_DIR / "00_raw"
INTERIM_DIR = PROJECT_DIR / "01_interim"
LOG_DIR     = PROJECT_DIR / "logs"

raw_df = pd.read_csv(LOG_DIR / "raw_snapshot.csv")
print("File di snapshot:", len(raw_df))   # harus 86

# Cell B — Konversi 4 file .doc lama ke .docx

In [ ]:
import shutil, subprocess

if shutil.which("soffice") is None:
    !apt-get -qq update && apt-get -qq install -y libreoffice-writer > /dev/null
print("LibreOffice:", shutil.which("soffice"))

DOC_CONV_DIR = INTERIM_DIR / "doc_converted"
DOC_CONV_DIR.mkdir(parents=True, exist_ok=True)

for _, r in raw_df[raw_df["ext"] == "doc"].iterrows():
    src = RAW_DIR / r["rel_path"]
    out = DOC_CONV_DIR / f"{r['file_id']}.docx"
    if out.exists():
        print("sudah ada:", out.name); continue
    tmp = Path("/content/_doc_tmp"); tmp.mkdir(exist_ok=True)
    subprocess.run(["soffice", "--headless", "--convert-to", "docx", "--outdir", str(tmp), str(src)],
                   check=True, capture_output=True)
    shutil.move(str(next(tmp.glob("*.docx"))), out)
    print(f"{r['file_id']}  {r['file_name']}  ->  {out.name}")

# Cell C — Fungsi pembaca per format

In [ ]:
import pymupdf, docx, openpyxl, warnings
warnings.filterwarnings("ignore")

def source_path(row):
    """File .doc dibaca dari hasil konversi; format lain dari file asli."""
    if row["ext"] == "doc":
        return DOC_CONV_DIR / f"{row['file_id']}.docx"
    return RAW_DIR / row["rel_path"]

def peek_pdf(p):
    with pymupdf.open(p) as d:
        n = d.page_count
        texts = [d[i].get_text() for i in range(n)]
    chars = [len(t.strip()) for t in texts]
    awal = " ".join(" ".join(texts).split())[:3000]
    return {"halaman": n, "hal_tanpa_teks": sum(c < 50 for c in chars),
            "total_karakter": sum(chars), "cuplikan": awal[:400], "teks_awal": awal}

def docx_text(p):
    d = docx.Document(p)
    parts = [para.text for para in d.paragraphs]
    for t in d.tables:                      # isi RPS sebagian besar ada di tabel
        for row in t.rows:
            seen = []
            for c in row.cells:             # sel gabungan muncul berulang, ambil sekali
                if c.text not in seen: seen.append(c.text)
            parts.append(" | ".join(seen))
    return d, "\n".join(parts)

def peek_docx(p):
    d, text = docx_text(p)
    awal = " ".join(text.split())[:3000]
    return {"jumlah_tabel": len(d.tables), "total_karakter": len(text.strip()),
            "cuplikan": awal[:400], "teks_awal": awal}

# Encoding yang sudah diverifikasi manual (deteksi otomatis salah untuk file ini)
ENCODING_OVERRIDE = {"Job Posting.csv": "mac_roman"}

def detect_encoding(p):
    if p.name in ENCODING_OVERRIDE:
        return ENCODING_OVERRIDE[p.name]
    raw = p.read_bytes()
    for enc in ("utf-8", "cp1252"):
        try:
            raw.decode(enc); return enc
        except UnicodeDecodeError:
            pass
    return "latin-1"

def detect_sep(p, enc):
    with open(p, encoding=enc) as f:
        header = f.readline()
    return max([",", "|", ";", "\t"], key=header.count)

def read_csv_robust(p):
    enc = detect_encoding(p)
    sep = detect_sep(p, enc)
    return pd.read_csv(p, encoding=enc, sep=sep, low_memory=False), enc, sep

def peek_csv(p):
    df, enc, sep = read_csv_robust(p)
    return {"encoding": enc, "delimiter": sep, "baris": len(df), "kolom": df.shape[1],
            "nama_kolom": ", ".join(map(str, df.columns))[:400]}

def peek_xlsx(p):
    sheets = pd.read_excel(p, sheet_name=None, nrows=None)
    info = [f"{name} ({df.shape[0]}x{df.shape[1]})" for name, df in sheets.items()]
    first_df = next(iter(sheets.values()))
    return {"sheet": "; ".join(info),
            "nama_kolom": ", ".join(map(str, first_df.columns))[:400]}

PEEK = {"pdf": peek_pdf, "docx": peek_docx, "doc": peek_docx, "csv": peek_csv, "xlsx": peek_xlsx}
print("Fungsi siap.")

# Cell D — Profiling semua file

In [ ]:
profiles = []
for _, r in raw_df.iterrows():
    try:
        info = PEEK[r["ext"]](source_path(r)); info["error"] = ""
    except Exception as e:
        info = {"error": f"{type(e).__name__}: {e}"}
    profiles.append({"file_id": r["file_id"], **info})
    print(f"{r['file_id']} ok" if not info["error"] else f"{r['file_id']} ERROR {info['error']}", end=" | ")

prof_df = raw_df.merge(pd.DataFrame(profiles), on="file_id")
prof_df.to_csv(LOG_DIR / "file_profile.csv", index=False)
print("\n\nError:", (prof_df["error"] != "").sum())     # harus 0

pdfs = prof_df[prof_df["ext"] == "pdf"]
display(pdfs[pdfs["hal_tanpa_teks"] > 0][["file_id", "file_name", "halaman", "hal_tanpa_teks", "total_karakter"]])

cek = prof_df[prof_df["file_name"].str.contains("roadmap|tracer|skillsfuture", case=False)
              | prof_df["folder"].str.contains("Roadmap")]
display(cek[["file_id", "file_name", "halaman", "total_karakter", "sheet", "cuplikan"]])

# Cell E — Kelompok dan status awal

In [ ]:
def kelompok(row):
    f, n = row["folder"], row["file_name"].lower()
    if "Demand Kaggle" in f: return "demand_lowongan"
    if "Industri" in f:      return "skkni"
    if "Okupasi" in f:       return "okupasi"
    if "Roadmap" in f:       return "roadmap"
    if "RPS" in f:           return "rps"
    if "tracer" in n:        return "tracer_study"
    if "skillsfuture" in n:  return "skill_taxonomy"
    if "kompetensi lulusan" in n:   return "profil_lulusan"
    if "capaian pembelajaran" in n: return "cpl"
    return "kurikulum"

MK_UMUM = ["agama", "pancasila", "b.indonesia", "english", "kewirausahaan", "metpen"]

def status_awal(row):
    n, k = row["file_name"].lower(), row["kelompok"]
    if n.startswith(("1b.", "21a.", "31a.")):
        return "salinan", "Versi PDF dari dokumen DOCX yang sama; DOCX dipakai"
    if "gabunganrps" in n:
        return "salinan", "Gabungan RPS; dipakai RPS individual agar tidak dobel di knowledge base"
    if k == "demand_lowongan" and "remote-jobs" in row["folder"]:
        return "tolak_bukti", "Kamus data menyebut synthetic; hanya untuk uji pipeline"
    if k == "demand_lowongan":
        return "pakai_dengan_filter", "Perlu filter lokasi Indonesia + bidang TI"
    if k == "tracer_study" and row["ext"] == "xlsx":
        return "referensi_privat", "Berisi data pribadi alumni; hanya agregat, tidak masuk RAG"
    if k == "tracer_study":
        return "referensi", "Statistik agregat untuk latar belakang"
    if k == "rps" and any(w in n for w in MK_UMUM):
        return "referensi", "Mata kuliah umum/non-teknis; tidak dipakai untuk pemetaan skill teknis"
    if k in ("skill_taxonomy", "roadmap"):
        return "referensi", "Pelengkap kosakata skill; bukan bukti demand Indonesia"
    return "pakai", ""

prof_df["kelompok"] = prof_df.apply(kelompok, axis=1)
prof_df[["status", "alasan"]] = prof_df.apply(lambda r: pd.Series(status_awal(r)), axis=1)
display(pd.crosstab(prof_df["kelompok"], prof_df["status"]))

# Cell F — Baca identitas SKKNI dan okupasi dari isi dokumen

In [ ]:
import re

def identitas(row):
    t = str(row.get("teks_awal", ""))
    if row["kelompok"] == "skkni":
        nomor  = re.search(r"NOMOR\s+(\d+)\s+TAHUN\s+(\d{4})", t)
        bidang = re.search(r"(BIDANG.*?)\s+BAB I", t)
        return pd.Series({"isi_nomor_tahun": f"{nomor.group(1)}/{nomor.group(2)}" if nomor else "",
                          "isi_judul": bidang.group(1) if bidang else ""})
    if row["kelompok"] == "okupasi":
        nama   = re.search(r"Nama Okupasi\s+(.*?)\s+Area Fungsi", t)
        level  = re.search(r"Level Kualifikasi\s+(\d+)", t)
        status = re.search(r"Status Okupasi\s+(.*?)(?:\s+(?:Definisi|Tanggung|Persyaratan|Lingkup))", t)
        return pd.Series({"isi_nomor_tahun": f"Level {level.group(1)}" if level else "",
                          "isi_judul": (nama.group(1) if nama else "") + (" | " + status.group(1) if status else "")})
    return pd.Series({"isi_nomor_tahun": "", "isi_judul": ""})

prof_df[["isi_nomor_tahun", "isi_judul"]] = prof_df.apply(identitas, axis=1)
pd.set_option("display.max_colwidth", 200)
display(prof_df[prof_df["kelompok"].isin(["skkni", "okupasi"])][["file_id", "file_name", "isi_nomor_tahun", "isi_judul"]])

# Cell G — Bandingkan pasangan file yang dicurigai duplikat isi

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

def full_text(row):
    p = source_path(row)
    if row["ext"] == "pdf":
        with pymupdf.open(p) as d: return " ".join(pg.get_text() for pg in d)
    if row["ext"] in ("docx", "doc"): return docx_text(p)[1]
    return ""

def fid(pattern):
    return prof_df[prof_df["file_name"].str.contains(pattern, regex=False)]["file_id"].iloc[0]

PASANGAN = [
    ("25. RPS_Sistem_Basis_Data", "25A. RPS_Sistem_Basis_Data"),
    ("1A. List Mata Kuliah", "1B. List Mata Kuliah"),
    ("21. KOMPETENSI", "21A. KOMPETENSI"),
    ("31. Capaian", "31A. Capaian"),
    ("Data & Database Management 2020-268", "Data & Database Management 2020-299"),
    ("Programming 2016-282", "Programming 2019-022"),
    ("43. Hasil Tracer", "44. TRACER STUDY"),
]
texts, hasil = {}, []
for a, b in PASANGAN:
    ia, ib = fid(a), fid(b)
    for i in (ia, ib):
        if i not in texts:
            texts[i] = full_text(prof_df[prof_df["file_id"] == i].iloc[0])
    tf = TfidfVectorizer().fit([texts[ia], texts[ib]])
    sim = cosine_similarity(tf.transform([texts[ia]]), tf.transform([texts[ib]]))[0, 0]
    hasil.append({"file_a": ia, "nama_a": a, "file_b": ib, "nama_b": b, "kemiripan": round(sim, 3)})

pair_df = pd.DataFrame(hasil)
pair_df.to_csv(LOG_DIR / "pair_similarity.csv", index=False)
display(pair_df)

# Cell H — Koreksi berdasarkan hasil verifikasi isi

In [ ]:
KOREKSI = {
    "25A. RPS_Sistem_Basis_Data":
        ("salinan", "Teks identik dengan 25. RPS_Sistem_Basis_Data (kemiripan 1.0)"),
    "SKKNI System Analyst & Requirement Engineering 2019-019":
        ("tolak", "SALAH FILE: isi Kepmenaker 19/2019 = bidang Perhiasan Logam Mulia; cari SKKNI analis sistem yang benar"),
    "SKKNI Web & Mobile Development 2020-391":
        ("perlu_keputusan", "SALAH LABEL: isi = bidang Security Operations Center (SOC); pakai hanya jika profesi keamanan siber masuk target"),
    "SKKNI Data & Database Management 2020-299":
        ("pakai", "SALAH LABEL: isi = bidang AI subbidang Data Science; bukan Database Management"),
    "SKKNI Software Engineering Programming 2019-022":
        ("pakai_selektif", "1.249 halaman, cakupan sangat luas (programming, network, operation); ambil unit yang relevan saja"),
    "TIK.DEV0701-MANAJER PENGEMBANGAN SISTEM":
        ("pakai", "SALAH LABEL: isi = okupasi 020801 Senior Programmer, Level 7"),
    "TIK.DEV0615-PENGEMBANG SISTEM PERANGKAT LUNAK":
        ("pakai", "Catatan: dokumen menyebut ada UK yang sudah dicabut; cek UK yang masih berlaku"),
    "65. RPS Kewirausahaan bidang TIK":
        ("referensi", "Mata kuliah umum; kop dokumen salah template (Fisika Terapan, Jurusan Teknik Sipil), isi utama Kewirausahaan TIF6404"),
}
for prefix, (status, alasan) in KOREKSI.items():
    mask = prof_df["file_name"].str.startswith(prefix)
    assert mask.sum() == 1, f"Tidak ketemu / lebih dari satu: {prefix}"
    prof_df.loc[mask, ["status", "alasan"]] = [status, alasan]

display(prof_df[prof_df["file_name"].str.startswith(tuple(KOREKSI))][["file_id", "file_name", "status", "alasan"]])

# Cell I — Simpan manifest

In [ ]:
manifest = prof_df[["file_id", "rel_path", "file_name", "ext", "kelompok", "status", "alasan",
                    "halaman", "hal_tanpa_teks", "baris", "kolom", "encoding", "delimiter", "sheet",
                    "isi_nomor_tahun", "isi_judul", "total_karakter", "cuplikan", "nama_kolom",
                    "error", "sha256"]].copy()
for c in ["sumber_penerbit", "nomor_tahun", "url_sumber", "catatan_manual"]:
    manifest[c] = ""                      # diisi manual saat review

manifest.to_csv(LOG_DIR / "manifest.csv", index=False)
manifest.to_excel(LOG_DIR / "manifest_review.xlsx", index=False)
print("Tersimpan:", LOG_DIR / "manifest.csv")
display(manifest["status"].value_counts())

#

---
# Tahap 2 — Ekstraksi teks (batch 1)
*Sumber: `02_extract.ipynb`*

# Cell 1 — Mount dan instal

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!pip install -q pymupdf python-docx openpyxl

# Cell 2 — Simpan fungsi baca sebagai modul (cukup sekali)

In [ ]:
!mkdir -p /content/drive/MyDrive/Skripsi_RAG/src
!ls /content/drive/MyDrive/Skripsi_RAG

In [ ]:
%%writefile /content/drive/MyDrive/Skripsi_RAG/src/io_utils.py
"""Fungsi baca file dataset skripsi (dipakai ulang di semua notebook)."""
from pathlib import Path
import pandas as pd
import pymupdf
import docx
from docx.table import Table

# Encoding yang sudah diverifikasi manual di Tahap 1
ENCODING_OVERRIDE = {"Job Posting.csv": "mac_roman"}


def source_path(row, raw_dir, doc_conv_dir):
    """File .doc dibaca dari hasil konversi; format lain dari file asli."""
    if row["ext"] == "doc":
        return Path(doc_conv_dir) / f"{row['file_id']}.docx"
    return Path(raw_dir) / row["rel_path"]


# ---------- CSV ----------
def detect_encoding(p):
    p = Path(p)
    if p.name in ENCODING_OVERRIDE:
        return ENCODING_OVERRIDE[p.name]
    raw = p.read_bytes()
    for enc in ("utf-8", "cp1252"):
        try:
            raw.decode(enc)
            return enc
        except UnicodeDecodeError:
            pass
    return "latin-1"


def detect_sep(p, enc):
    with open(p, encoding=enc) as f:
        header = f.readline()
    return max([",", "|", ";", "\t"], key=header.count)


def read_csv_robust(p):
    enc = detect_encoding(p)
    sep = detect_sep(p, enc)
    return pd.read_csv(p, encoding=enc, sep=sep, low_memory=False), enc, sep


# ---------- PDF ----------
def pdf_pages(p):
    """List teks per halaman (indeks 0 = halaman 1)."""
    with pymupdf.open(p) as d:
        return [pg.get_text() for pg in d]


# ---------- DOCX ----------
def _table_rows(tbl):
    rows = []
    for row in tbl.rows:
        seen = []
        for c in row.cells:                      # sel gabungan muncul berulang
            t = " ".join(c.text.split())
            if t and t not in seen:
                seen.append(t)
        if seen:
            rows.append(" | ".join(seen))
    return rows


def docx_blocks(p):
    """Paragraf dan tabel sesuai URUTAN di dokumen."""
    d = docx.Document(p)
    blocks = []
    for item in d.iter_inner_content():
        if isinstance(item, Table):
            rows = _table_rows(item)
            if rows:
                blocks.append({"type": "table", "text": "\n".join(rows)})
        else:
            t = item.text.strip()
            if t:
                blocks.append({"type": "paragraph", "text": t})
    return blocks


def docx_text(p):
    return "\n".join(b["text"] for b in docx_blocks(p))

# Cell 3 — Path dan import modul

In [ ]:
import sys, importlib
from pathlib import Path
import pandas as pd

PROJECT_DIR  = Path("/content/drive/MyDrive/Skripsi_RAG")
RAW_DIR      = PROJECT_DIR / "00_raw"
INTERIM_DIR  = PROJECT_DIR / "01_interim"
LOG_DIR      = PROJECT_DIR / "logs"
DOC_CONV_DIR = INTERIM_DIR / "doc_converted"

sys.path.insert(0, str(PROJECT_DIR / "src"))
import io_utils; importlib.reload(io_utils)
from io_utils import source_path, pdf_pages, docx_blocks, read_csv_robust

manifest = pd.read_csv(LOG_DIR / "manifest.csv")
print(manifest.shape)                                   # (86, 25)
print("File manual:", sorted(p.name for p in (INTERIM_DIR / "manual").glob("*.csv")))

# Cell 4 — Pilih file yang diekstrak

In [ ]:
STATUS_DIEKSTRAK = ["pakai", "pakai_selektif", "pakai_dengan_filter", "referensi", "perlu_keputusan"]
target = manifest[manifest["status"].isin(STATUS_DIEKSTRAK)].copy()
docs   = target[target["ext"].isin(["pdf", "docx", "doc"])]
tables = target[target["ext"].isin(["csv", "xlsx"])]

print("Dokumen teks:", len(docs), "| File tabel:", len(tables))      # 70 | 4
print("Tidak diekstrak:", len(manifest) - len(target))               # 12
display(manifest[~manifest["status"].isin(STATUS_DIEKSTRAK)][["file_id", "file_name", "status"]])

# Cell 5 — Ekstrak dokumen ke JSONL

In [ ]:
import json

OUT_JSONL = INTERIM_DIR / "extracted_docs.jsonl"
report = []

with open(OUT_JSONL, "w", encoding="utf-8") as f:
    for _, r in docs.iterrows():
        p = source_path(r, RAW_DIR, DOC_CONV_DIR)
        meta = {"file_id": r["file_id"], "file_name": r["file_name"],
                "kelompok": r["kelompok"], "status": r["status"]}
        n_unit, n_char, n_kosong = 0, 0, 0
        if r["ext"] == "pdf":
            for i, text in enumerate(pdf_pages(p), start=1):       # 1 unit = 1 halaman
                if len(text.strip()) < 50:
                    n_kosong += 1
                    continue
                rec = {**meta, "unit_type": "page", "unit_no": i, "text": text}
                f.write(json.dumps(rec, ensure_ascii=False) + "\n")
                n_unit += 1; n_char += len(text)
        else:
            for i, b in enumerate(docx_blocks(p), start=1):         # 1 unit = 1 paragraf/tabel
                rec = {**meta, "unit_type": b["type"], "unit_no": i, "text": b["text"]}
                f.write(json.dumps(rec, ensure_ascii=False) + "\n")
                n_unit += 1; n_char += len(b["text"])
        report.append({"file_id": r["file_id"], "file_name": r["file_name"], "kelompok": r["kelompok"],
                       "unit": n_unit, "unit_kosong_dilewati": n_kosong, "karakter": n_char})
        print(f"{r['file_id']}: {n_unit} unit, {n_char:,} karakter")

doc_report = pd.DataFrame(report)
print("\nTotal unit:", doc_report["unit"].sum(), "| Total karakter:", f"{doc_report['karakter'].sum():,}")

# Cell 6 — Tambahkan transkripsi manual F031

In [ ]:
MANUAL_DIR = INTERIM_DIR / "manual"
meta = manifest[manifest["file_id"] == "F031"].iloc[0]
base = {"file_id": "F031", "file_name": meta["file_name"],
        "kelompok": meta["kelompok"], "status": meta["status"]}

# Daftar mata kuliah sudah tercakup di 1A (F076), jadi hanya profil lulusan (hal. 1) yang ditambahkan
pl = pd.read_csv(MANUAL_DIR / "F031_profil_lulusan_kurikulum2021.csv", dtype=str)

n_added = 0
with open(OUT_JSONL, "a", encoding="utf-8") as f:
    for profil, g in pl.groupby("profil", sort=False):
        cp = "; ".join(f"{row.kode_cp_kk_prodi} ({row.kode_cp_aptikom_d4ti}): {row.cp_keterampilan_khusus}"
                       for row in g.itertuples())
        text = (f"Profil lulusan {profil}. Deskripsi: {g['deskripsi'].iloc[0]} "
                f"Capaian pembelajaran keterampilan khusus: {cp}")
        rec = {**base, "unit_type": "manual_row_profil", "unit_no": 1, "text": text}
        f.write(json.dumps(rec, ensure_ascii=False) + "\n"); n_added += 1

doc_report.loc[doc_report["file_id"] == "F031", "catatan"] = \
    "Hal. 1 (profil lulusan) ditranskripsi manual; daftar MK hal. 4-5 sudah tercakup di F076 (1A)"
print("Record profil lulusan ditambahkan:", n_added)      # 4

# Cell 7 — Simpan tabel CSV/XLSX sebagai UTF-8 seragam

In [ ]:
TABLE_DIR = INTERIM_DIR / "tables"
TABLE_DIR.mkdir(exist_ok=True)
tab_report = []

for _, r in tables.iterrows():
    p = source_path(r, RAW_DIR, DOC_CONV_DIR)
    if r["ext"] == "csv":
        df, enc, sep = read_csv_robust(p)
        frames = {"": df}
    else:
        frames = pd.read_excel(p, sheet_name=None)
        enc, sep = "xlsx", ""
    for sheet, df in frames.items():
        name = f"{r['file_id']}" + (f"__{sheet.replace(' ', '_')}" if sheet else "") + ".csv"
        df.to_csv(TABLE_DIR / name, index=False, encoding="utf-8")
        tab_report.append({"file_id": r["file_id"], "file_name": r["file_name"], "sheet": sheet,
                           "output": name, "baris": len(df), "kolom": df.shape[1],
                           "encoding_asal": enc, "delimiter_asal": sep})
tab_report = pd.DataFrame(tab_report)
display(tab_report)

# Cell 8 — Laporan dan pemeriksaan

In [ ]:
doc_report.to_csv(LOG_DIR / "extraction_report_docs.csv", index=False)
tab_report.to_csv(LOG_DIR / "extraction_report_tables.csv", index=False)

recs = pd.read_json(OUT_JSONL, lines=True)
print("Record di JSONL:", len(recs))                     # 2864
display(recs.groupby("kelompok").agg(file=("file_id", "nunique"), unit=("text", "size"),
                                     karakter=("text", lambda s: s.str.len().sum())))
display(doc_report[doc_report["unit_kosong_dilewati"] > 0])

---
# Tahap 2b — Ekstraksi teks (batch 2)
*Sumber: `02b_batch2ipynb.ipynb`*

# Cell 1 — Mount dan instal

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!pip install -q pymupdf python-docx openpyxl

# Cell 2 — Path, modul, dan manifest

In [ ]:
import sys, importlib, json, zipfile, hashlib, datetime
from pathlib import Path
import pandas as pd

PROJECT_DIR = Path("/content/drive/MyDrive/Skripsi_RAG")
RAW_DIR     = PROJECT_DIR / "00_raw"
INTERIM_DIR = PROJECT_DIR / "01_interim"
LOG_DIR     = PROJECT_DIR / "logs"
TABLE_DIR   = INTERIM_DIR / "tables"

sys.path.insert(0, str(PROJECT_DIR / "src"))
import io_utils; importlib.reload(io_utils)
from io_utils import read_csv_robust

manifest = pd.read_csv(LOG_DIR / "manifest.csv")
raw_df   = pd.read_csv(LOG_DIR / "raw_snapshot.csv")
print("Manifest:", manifest.shape, "| Snapshot:", raw_df.shape)   # (86, 25) | (86, 7)

# Cell 3 — Perbarui status F002

In [ ]:
mask = manifest["file_name"] == "Job Posting.csv"
manifest.loc[mask, ["status", "alasan"]] = [
    "tolak_bukti", "Hanya 10 dari 9.919 baris berlokasi di Indonesia; tidak ada yang bidang TI"]
manifest.to_csv(LOG_DIR / "manifest.csv", index=False)
display(manifest.loc[mask, ["file_id", "file_name", "status", "alasan"]])

# Cell 4 — Ekstrak ZIP batch kedua ke 00_raw

In [ ]:
zips = sorted(PROJECT_DIR.glob("*Rekomen*.zip"))
print(zips)
ZIP2 = zips[0]

def sha256(path, chunk=1 << 20):
    h = hashlib.sha256()
    with open(path, "rb") as fh:
        for block in iter(lambda: fh.read(chunk), b""):
            h.update(block)
    return h.hexdigest()

with zipfile.ZipFile(ZIP2) as zf:
    print("File rusak:", zf.testzip())                        # None = utuh
    members = [m for m in zf.infolist() if not m.is_dir()]
    root = members[0].filename.split("/")[0]
    print("Jumlah file:", len(members), "| Folder induk:", root)   # 248 | 0. Dataset+ Rekomen Claude
    if (RAW_DIR / root).exists():
        print("Sudah diekstrak sebelumnya, dilewati.")
    else:
        zf.extractall(RAW_DIR)
        print("Ekstraksi selesai.")
ZIP2_SHA = sha256(ZIP2)

# Cell 5 — Tambahkan ke snapshot (F087 dan seterusnya)

In [ ]:
if raw_df["rel_path"].str.startswith(root).any():
    print("Batch 2 sudah ada di snapshot, dilewati.")
else:
    start = int(raw_df["file_id"].str[1:].astype(int).max()) + 1
    files = sorted(p for p in (RAW_DIR / root).rglob("*") if p.is_file())
    rows = []
    for i, p in enumerate(files, start):
        rel = p.relative_to(RAW_DIR)
        rows.append({"file_id": f"F{i:03d}", "rel_path": str(rel), "folder": str(rel.parent),
                     "file_name": p.name, "ext": p.suffix.lower().lstrip("."),
                     "size_kb": round(p.stat().st_size / 1024, 1), "sha256": sha256(p)})
    batch2 = pd.DataFrame(rows)
    raw_df = pd.concat([raw_df, batch2], ignore_index=True)
    raw_df.to_csv(LOG_DIR / "raw_snapshot.csv", index=False)
    print(f"Ditambahkan {len(batch2)} file: {batch2['file_id'].iloc[0]} – {batch2['file_id'].iloc[-1]}")

b2 = raw_df[raw_df["rel_path"].str.startswith(root)].copy()
display(b2.groupby("folder").agg(jumlah=("file_id", "size"), dari=("file_id", "min"), sampai=("file_id", "max")))

# Cell 6 — Tambahkan ke manifest beserta sumber, periode, dan lisensi

In [ ]:
SUMBER = {
    "Kaggle Jobstreet": dict(
        kelompok="demand_lowongan", sumber_penerbit="Kaggle – azizainunnajib (Jobstreet Indonesia Dataset)",
        nomor_tahun="Nov 2021 – Ags 2022", url_sumber="https://www.kaggle.com/datasets/azizainunnajib/jobs-crawling",
        lisensi="Tidak tercantum"),
    "Github indonesia-job-vacancy-dataset": dict(
        kelompok="demand_lowongan", sumber_penerbit="Simaremare dkk. (IT Del), IJERCSE Vol 10 Issue 8, 2023",
        nomor_tahun="2015 – 2022", url_sumber="https://github.com/simaremare/indonesia-job-vacancy-dataset",
        lisensi="CC0-1.0"),
    "Zenodo": dict(
        kelompok="demand_lowongan", sumber_penerbit="Zainuddin & Indria (Binus), Zenodo 2026",
        nomor_tahun="17 Mar – 23 Apr 2025", url_sumber="https://zenodo.org/records/18162371",
        lisensi="CC BY 4.0"),
    "Mendeley": dict(
        kelompok="demand_lowongan", sumber_penerbit="Handayani, Idris & Shuib, Mendeley Data v2 (DOI 10.17632/zzy8v7ggkz.2)",
        nomor_tahun="Jul – Sep 2025 (Loker.id)", url_sumber="https://data.mendeley.com/datasets/zzy8v7ggkz",
        lisensi="Tidak tercantum"),
}

def info_sumber(folder):
    for key, val in SUMBER.items():
        if key in folder:
            return val
    raise ValueError(folder)

def status_batch2(row):
    n, f = row["file_name"], row["folder"]
    if "Kaggle Jobstreet" in f:
        if n == "mergeFile.csv":
            return "pakai", "Volume demand: 297.805 lowongan unik, 32.942 kategori TI; deskripsi terpotong ~150 karakter; dedup per id"
        return "salinan", "Snapshot harian; seluruh id tercakup di mergeFile.csv"
    if "Github" in f:
        return "pakai_dengan_filter", "Deskripsi lengkap + kolom requirement; filter judul TI; 18.374 baris tanggal 01/01/1970 = tidak diketahui"
    if "Zenodo" in f:
        return "pakai_dengan_filter", "Data 2025 khusus profesi data; buang duplikat link Glints dan lowongan data entry/annotator"
    if "Mendeley" in f and n.endswith(".csv"):
        return "pakai", "776 kalimat kebutuhan skill TI unik (2025); tanpa judul/ID lowongan sehingga tidak bisa dihitung per profesi"
    if "Mendeley" in f:
        return "referensi", "Codebook dataset Mendeley (dokumentasi sumber dan praproses)"
    raise ValueError(n)

new_rows = []
for _, r in b2.iterrows():
    if r["file_id"] in set(manifest["file_id"]):
        continue
    status, alasan = status_batch2(r)
    src = info_sumber(r["folder"])
    new_rows.append({"file_id": r["file_id"], "rel_path": r["rel_path"], "file_name": r["file_name"],
                     "ext": r["ext"], "status": status, "alasan": alasan, "sha256": r["sha256"],
                     "catatan_manual": f"Lisensi: {src['lisensi']}",
                     **{k: v for k, v in src.items() if k != "lisensi"}})
if new_rows:
    manifest = pd.concat([manifest, pd.DataFrame(new_rows)], ignore_index=True)
    manifest.to_csv(LOG_DIR / "manifest.csv", index=False)

print("Total manifest:", len(manifest))                     # 334
b2m = manifest[manifest["file_id"].isin(b2["file_id"])]
display(b2m.groupby("status").size())
display(b2m[b2m["status"] != "salinan"][["file_id", "file_name", "status", "nomor_tahun"]])

# Cell 7 — Seragamkan ke 01_interim/tables

In [ ]:
def fid(name):
    return b2m.loc[b2m["file_name"] == name, "file_id"].iloc[0]

def path_of(file_id):
    return RAW_DIR / manifest.loc[manifest["file_id"] == file_id, "rel_path"].iloc[0]

tab_report = []
def simpan(df, file_id, suffix, catatan=""):
    name = f"{file_id}{suffix}.csv"
    df.to_csv(TABLE_DIR / name, index=False, encoding="utf-8")
    tab_report.append({"file_id": file_id, "output": name, "baris": len(df), "kolom": df.shape[1], "catatan": catatan})
    print(f"{name}: {len(df):,} baris")

# 1) Kaggle Jobstreet: deduplikasi per id lowongan
f = fid("mergeFile.csv")
js = pd.read_csv(path_of(f), low_memory=False)
js = js.drop(columns=[c for c in js.columns if c.startswith("Unnamed")])
n0 = len(js)
js["postedAt"] = pd.to_datetime(js["postedAt"], errors="coerce", utc=True)
js = js.sort_values("postedAt").drop_duplicates("id", keep="first")
simpan(js, f, "", f"Dedup per id: {n0:,} -> {len(js):,}")
del js

# 2) GitHub: tanggal placeholder 1970 dijadikan kosong
f = fid("job-vacancy-dataset.csv")
gh, enc, sep = read_csv_robust(path_of(f))
gh["time"] = gh["time"].replace("01/01/1970 00:00:00", pd.NA)
simpan(gh, f, "", "Tanggal 01/01/1970 diganti kosong")

# 3) Zenodo: gabung per sumber, lalu buang duplikat
z = b2m[b2m["rel_path"].str.contains("Zenodo")]
glints = pd.concat([pd.read_csv(path_of(i)).assign(file_asal=n)
                    for i, n in zip(z["file_id"], z["file_name"]) if n.startswith("glints")], ignore_index=True)
n0 = len(glints)
glints["job_uuid"] = glints["link"].str.extract(r"jobs/[^/]+/([0-9a-f-]{36})")
glints = glints.drop_duplicates("job_uuid")
simpan(glints, z["file_id"].min(), "__glints_gabungan", f"3 file Glints digabung, dedup per UUID: {n0} -> {len(glints)}")
jsz = pd.concat([pd.read_csv(path_of(i)).assign(file_asal=n)
                 for i, n in zip(z["file_id"], z["file_name"]) if n.startswith("job_list")], ignore_index=True)
n0 = len(jsz)
jsz = jsz.drop_duplicates("Link")
simpan(jsz, z[z["file_name"].str.startswith("job_list")]["file_id"].min(), "__jobstreet_gabungan",
       f"2 file JobStreet digabung, dedup per link: {n0} -> {len(jsz)}")

# 4) Mendeley: 1 kalimat per baris (sebagian kalimat mengandung koma, jadi tidak bisa dibaca sebagai CSV biasa)
f = fid("CSVFINAL2JobDataOneRow.csv")
lines = path_of(f).read_text(encoding="utf-8-sig").splitlines()
kal = [l.strip() for l in lines[1:] if l.strip()]
md = pd.DataFrame({"text": kal}).drop_duplicates()
simpan(md, f, "", f"Header '{lines[0].strip()}' dibuang; {len(kal)} baris -> {len(md)} unik")

tab_report = pd.DataFrame(tab_report)
old = pd.read_csv(LOG_DIR / "extraction_report_tables.csv")
old = old[~old["file_id"].isin(tab_report["file_id"])]          # aman bila cell dijalankan ulang
pd.concat([old, tab_report], ignore_index=True).to_csv(LOG_DIR / "extraction_report_tables.csv", index=False)
display(tab_report)

# Cell 8 — Log batch kedua

In [ ]:
log = {"tanggal": datetime.datetime.now().isoformat(timespec="seconds"), "zip_file": ZIP2.name,
       "zip_sha256": ZIP2_SHA, "jumlah_file": int(len(b2)),
       "file_id": f"{b2['file_id'].min()} – {b2['file_id'].max()}"}
with open(LOG_DIR / "setup_log_batch2.json", "w") as fp:
    json.dump(log, fp, indent=2, ensure_ascii=False)
print(json.dumps(log, indent=2, ensure_ascii=False))
display(manifest["status"].value_counts())

---
# Tahap 2c — Ekstraksi teks (batch 3)
*Sumber: `02c_batch3.ipynb`*

# Cell 1 — Mount dan instal

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!pip install -q pymupdf python-docx openpyxl

# Cell 2 — Path, modul, dan manifest

In [ ]:
import sys, importlib, json, zipfile, hashlib, datetime, re
from pathlib import Path
import pandas as pd

PROJECT_DIR = Path("/content/drive/MyDrive/Skripsi_RAG")
RAW_DIR     = PROJECT_DIR / "00_raw"
INTERIM_DIR = PROJECT_DIR / "01_interim"
LOG_DIR     = PROJECT_DIR / "logs"

sys.path.insert(0, str(PROJECT_DIR / "src"))
import io_utils; importlib.reload(io_utils)
from io_utils import pdf_pages

manifest = pd.read_csv(LOG_DIR / "manifest.csv")
raw_df   = pd.read_csv(LOG_DIR / "raw_snapshot.csv")
print("Manifest:", manifest.shape, "| Snapshot:", raw_df.shape)   # (334, ...) | (334, 7)

# Cell 3 — Ambil hanya file yang isinya baru

In [ ]:
zips = sorted(PROJECT_DIR.glob("*261013*.zip"))
print(zips)
ZIP3 = zips[0]
BATCH3_DIR = RAW_DIR / "batch3_261013"

known = set(raw_df["sha256"])            # hash semua file yang sudah ada di 00_raw
baru, sama, dilewati = [], 0, []
with zipfile.ZipFile(ZIP3) as zf:
    print("File rusak:", zf.testzip())
    for m in zf.infolist():
        if m.is_dir():
            continue
        if m.filename.lower().endswith(".zip"):          # ZIP di dalam ZIP tidak diproses
            dilewati.append(m.filename); continue
        data = zf.read(m)
        h = hashlib.sha256(data).hexdigest()
        if h in known:                                    # isi identik dengan file lama
            sama += 1; continue
        known.add(h)
        out = BATCH3_DIR / m.filename
        if not out.exists():
            out.parent.mkdir(parents=True, exist_ok=True)
            out.write_bytes(data)
        baru.append(m.filename)

print(f"Sudah ada di 00_raw (isi identik): {sama}")              # 337
print(f"ZIP di dalam ZIP (dilewati): {dilewati}")
print(f"File baru disimpan ke {BATCH3_DIR.name}: {len(baru)}")   # 40

# Cell 4 — Tambahkan ke snapshot (F335–F374)

In [ ]:
def sha256(path, chunk=1 << 20):
    h = hashlib.sha256()
    with open(path, "rb") as fh:
        for block in iter(lambda: fh.read(chunk), b""):
            h.update(block)
    return h.hexdigest()

if raw_df["rel_path"].str.startswith(BATCH3_DIR.name).any():
    print("Batch 3 sudah ada di snapshot, dilewati.")
else:
    start = int(raw_df["file_id"].str[1:].astype(int).max()) + 1
    files = sorted(p for p in BATCH3_DIR.rglob("*") if p.is_file())
    rows = []
    for i, p in enumerate(files, start):
        rel = p.relative_to(RAW_DIR)
        rows.append({"file_id": f"F{i:03d}", "rel_path": str(rel), "folder": str(rel.parent),
                     "file_name": p.name, "ext": p.suffix.lower().lstrip("."),
                     "size_kb": round(p.stat().st_size / 1024, 1), "sha256": sha256(p)})
    raw_df = pd.concat([raw_df, pd.DataFrame(rows)], ignore_index=True)
    raw_df.to_csv(LOG_DIR / "raw_snapshot.csv", index=False)
    print(f"Ditambahkan {len(rows)} file: {rows[0]['file_id']} – {rows[-1]['file_id']}")

b3 = raw_df[raw_df["rel_path"].str.startswith(BATCH3_DIR.name)].copy()
display(b3.groupby("folder").agg(jumlah=("file_id", "size"), dari=("file_id", "min"), sampai=("file_id", "max")))

# Cell 5 — Baca identitas dari isi dokumen dan tentukan status

In [ ]:
def teks_awal(p, n=4000):
    t = ""
    for pg in pdf_pages(p):
        t += pg
        if len(t) > n:
            break
    return " ".join(t.split())

def identitas_b3(row):
    p = RAW_DIR / row["rel_path"]
    if row["ext"] != "pdf":
        return pd.Series({"kelompok": "kurikulum", "isi_nomor_tahun": "", "isi_judul": ""})
    t = teks_awal(p)
    if "SKKNI" in row["folder"]:
        no = re.search(r"NOMOR\s+(\d+)\s+TAHUN\s+(\d{4})", t)
        bd = re.search(r"(BIDANG.{0,160}?)\s+BAB I", t, re.I)
        return pd.Series({"kelompok": "skkni",
                          "isi_nomor_tahun": f"{no.group(1)}/{no.group(2)}" if no else "",
                          "isi_judul": bd.group(1) if bd else ""})
    nm = re.search(r"Nama Okupasi\s+(.*?)\s+Area Fungsi\s+(.*?)\s+Level Kualifikasi\s+(\d+)", t)
    return pd.Series({"kelompok": "okupasi",
                      "isi_nomor_tahun": f"Level {nm.group(3)}" if nm else "",
                      "isi_judul": f"{nm.group(1)} | {nm.group(2)}" if nm else ""})

b3[["kelompok", "isi_nomor_tahun", "isi_judul"]] = b3.apply(identitas_b3, axis=1)

def status_b3(row):
    n, k = row["file_name"], row["kelompok"]
    if n.startswith("F031_"):
        return "salinan", "Sama dengan transkripsi manual di 01_interim/manual/"
    if n.endswith("(1).pdf"):
        return "salinan", "Unduhan ganda; isi identik kecuali timestamp"
    if k == "skkni":
        return "pakai", ""
    lvl = int(row["isi_nomor_tahun"].split()[-1])
    if 4 <= lvl <= 6:
        return "pakai", f"Level {lvl}: sesuai jenjang lulusan D4/baru"
    if lvl >= 7:
        return "referensi", f"Level {lvl}: di atas jenjang lulusan baru"
    return "referensi", f"Level {lvl}: di bawah jenjang lulusan D4"

b3[["status", "alasan"]] = b3.apply(lambda r: pd.Series(status_b3(r)), axis=1)

KOREKSI_B3 = {
    "TIK.INF0405-": ("referensi", "SALAH LABEL: isi = okupasi 140405 Supervisor Jaringan, Level 4 (bukan Asisten Jaringan)"),
    "TIK.ITG0702-": ("referensi", "Level 7, area Tata Kelola TI; tidak terkait profesi target"),
    "SKKNI UI UX Designer 2018-107": ("pakai", "Rujukan utama UI/UX: memuat unit user interface & user experience"),
    "SKKNI UI UX Designer 2023-126": ("pakai", "Pelengkap UI/UX: desain grafis & DKV; UI hanya disebut sebagai jenis karya"),
}
for prefix, (st, al) in KOREKSI_B3.items():
    mask = b3["file_name"].str.startswith(prefix)
    b3.loc[mask, ["status", "alasan"]] = [st, al]

pd.set_option("display.max_colwidth", 90)
display(b3[["file_id", "file_name", "kelompok", "isi_nomor_tahun", "isi_judul", "status"]])

# Cell 6 — Tambahkan ke manifest

In [ ]:
new_rows = b3[~b3["file_id"].isin(manifest["file_id"])][
    ["file_id", "rel_path", "file_name", "ext", "kelompok", "status", "alasan",
     "isi_nomor_tahun", "isi_judul", "sha256"]].copy()
new_rows["sumber_penerbit"] = new_rows["kelompok"].map(
    {"skkni": "JDIH Kemnaker", "okupasi": "Stankomdigi KOMDIGI (standar.sdmdigital.id)",
     "kurikulum": "Transkripsi manual"})
new_rows["url_sumber"] = new_rows["kelompok"].map(
    {"skkni": "https://jdih.kemnaker.go.id/peraturan",
     "okupasi": "https://standar.sdmdigital.id/peta-okupasi/"})
if len(new_rows):
    manifest = pd.concat([manifest, new_rows], ignore_index=True)
    manifest.to_csv(LOG_DIR / "manifest.csv", index=False)

print("Total manifest:", len(manifest))                     # 374
display(manifest[manifest["file_id"].isin(b3["file_id"])].groupby(["kelompok", "status"]).size())

# Cell 7 — Ekstrak teks PDF baru ke extracted_docs.jsonl

In [ ]:
OUT_JSONL = INTERIM_DIR / "extracted_docs.jsonl"
target = manifest[manifest["file_id"].isin(b3["file_id"])
                  & manifest["status"].isin(["pakai", "referensi"])
                  & (manifest["ext"] == "pdf")]

lama = pd.read_json(OUT_JSONL, lines=True)
lama = lama[~lama["file_id"].isin(target["file_id"])]       # aman bila dijalankan ulang
lama.to_json(OUT_JSONL, orient="records", lines=True, force_ascii=False)
n_lama = len(lama)

report = []
with open(OUT_JSONL, "a", encoding="utf-8") as f:
    for _, r in target.iterrows():
        meta = {"file_id": r["file_id"], "file_name": r["file_name"],
                "kelompok": r["kelompok"], "status": r["status"]}
        n_unit = n_char = n_kosong = 0
        for i, text in enumerate(pdf_pages(RAW_DIR / r["rel_path"]), start=1):
            if len(text.strip()) < 50:
                n_kosong += 1; continue
            f.write(json.dumps({**meta, "unit_type": "page", "unit_no": i, "text": text}, ensure_ascii=False) + "\n")
            n_unit += 1; n_char += len(text)
        report.append({"file_id": r["file_id"], "file_name": r["file_name"], "kelompok": r["kelompok"],
                       "unit": n_unit, "unit_kosong_dilewati": n_kosong, "karakter": n_char})

rep = pd.DataFrame(report)
old = pd.read_csv(LOG_DIR / "extraction_report_docs.csv")
old = old[~old["file_id"].isin(rep["file_id"])]
pd.concat([old, rep], ignore_index=True).to_csv(LOG_DIR / "extraction_report_docs.csv", index=False)

recs = pd.read_json(OUT_JSONL, lines=True)
print("Dokumen diekstrak:", len(rep), "| unit baru:", rep["unit"].sum())      # 37 | 1389
print("Record JSONL:", n_lama, "->", len(recs))                                # 2797 -> 4186
display(recs.groupby("kelompok").agg(file=("file_id", "nunique"), unit=("text", "size")))

# Cell 8 — Log batch ketiga

In [ ]:
log = {"tanggal": datetime.datetime.now().isoformat(timespec="seconds"), "zip_file": ZIP3.name,
       "zip_sha256": sha256(ZIP3), "file_identik_dilewati": sama, "file_baru": len(b3),
       "file_id": f"{b3['file_id'].min()} – {b3['file_id'].max()}"}
with open(LOG_DIR / "setup_log_batch3.json", "w") as fp:
    json.dump(log, fp, indent=2, ensure_ascii=False)
print(json.dumps(log, indent=2, ensure_ascii=False))
display(manifest["status"].value_counts())

---
# Tahap 3 — Demand industri (iklan lowongan)
*Sumber: `03_demand.ipynb`*

# Cell 1 — Mount

# Cell 2 — Path

In [ ]:
import re, json, datetime
from pathlib import Path
import pandas as pd

PROJECT_DIR = Path("/content/drive/MyDrive/Skripsi_RAG")
TABLE_DIR   = PROJECT_DIR / "01_interim" / "tables"
OUT_DIR     = PROJECT_DIR / "02_processed"
LOG_DIR     = PROJECT_DIR / "logs"
OUT_DIR.mkdir(exist_ok=True)

# Cell 3 — Daftar profesi (file konfigurasi)

In [ ]:
PROFESI_CSV = OUT_DIR / "profesi.csv"
PROFESI_DEFAULT = [
 # Urutan baris = urutan pengecekan (pola yang lebih spesifik dicek lebih dulu)
 # id, nama, pola judul (semua sumber), pola tambahan (hanya sumber berkategori TI), aktif, catatan
 ("P01", "UI/UX Designer", r"ui\s*/\s*ux|\bux\b|ui designer|user interface|user experience|product designer", r"\bui\b", 1, ""),
 ("P08", "Cyber Security", r"cyber|information security|\bit security|security (engineer|analyst|specialist|architect)|keamanan (informasi|siber|jaringan)|pentest|penetration|soc analyst", r"security", 1, ""),
 ("P02", "QA / Software Tester", r"qa engineer|qa tester|software qa|\bsqa\b|software tester|\btester\b|test engineer|automation test|qa automation|quality assurance (engineer|software|tester|analyst)", r"\bqa\b|quality assurance|quality control", 1, ""),
 ("P03", "Data Scientist / ML / AI Engineer", r"data scien|machine learning|\bml engineer|\bai engineer|artificial intelligence|deep learning|computer vision|\bnlp\b", "", 1, ""),
 ("P04", "Data Engineer", r"data engineer|big data|\betl\b|data warehouse", "", 1, ""),
 ("P05", "Data Analyst / BI", r"data analy|analis data|business intelligence|\bbi (analyst|developer)|reporting analyst", "", 1, ""),
 ("P06", "Database Administrator", r"database|\bdba\b|basis data", "", 1, "Profil lulusan prodi"),
 ("P07", "DevOps / Cloud Engineer", r"devops|dev ops|cloud (engineer|architect|admin)|site reliability|\bsre\b|platform engineer", r"cloud", 1, ""),
 ("P09", "Network / System Administrator", r"network (engineer|admin|administrator|technician|specialist|support)|jaringan|system admin|sys ?admin|administrator sistem|server admin|infrastructure engineer|\bit infrastructure|noc ", r"network|infrastructure|infrastruktur|server|system engineer", 1, ""),
 ("P10", "System Analyst", r"system analyst|systems analyst|analis sistem|sistem analis|\bit analyst|\bit business analyst|business system analyst", r"business analyst", 1, "Profil lulusan prodi"),
 ("P11", "Mobile Developer", r"android|\bios (developer|engineer|programmer)|flutter|mobile (developer|programmer|engineer|apps)|react native|kotlin", r"mobile|\bios\b", 1, ""),
 ("P12", "Software / Web Developer", r"developer|programmer|software engineer|web (developer|programmer|designer|master)|backend|back end|back-end|frontend|front end|front-end|full ?stack|pemrogram|\bphp\b|java (developer|programmer|engineer)|\.net|golang|laravel|software development", r"\bweb\b|software|java|\bit development", 1, "Profil lulusan prodi (Software Developer, Pemrogram Senior)"),
 ("P13", "IT Support / Helpdesk", r"\bit support|helpdesk|help desk|technical support|desktop support|\bit staff|staff it\b|staf it\b|\bit officer|teknisi (komputer|it|jaringan)|computer technician|\bit technician|\bit helpdesk", r"support|teknisi|technician|\bit supervisor|\bit specialist|\bit engineer", 1, ""),
 ("P14", "IT Project/Product Manager", r"\bit project manager|project manager it\b|\bit manager|head of it\b|\bit lead|scrum master|\bit product manager", r"project manager|product manager|product owner|technical lead", 1, "Jabatan manajerial; jarang terbuka untuk lulusan baru"),
]
if PROFESI_CSV.exists():
    profesi = pd.read_csv(PROFESI_CSV, keep_default_na=False)
    print("Memakai profesi.csv yang sudah ada (edit file itu untuk mengubah daftar).")
else:
    profesi = pd.DataFrame(PROFESI_DEFAULT, columns=["id", "nama", "pola", "pola_kategori_ti", "aktif", "catatan"])
    profesi.to_csv(PROFESI_CSV, index=False)
    print("profesi.csv dibuat.")
display(profesi[["id", "nama", "aktif", "catatan"]])

# Cell 4 — Fungsi klasifikasi judul ke profesi

In [ ]:
# Judul yang jelas bukan TI meski mengandung kata seperti "security", "java", "network"
EXCLUDE = re.compile(r"\b(sales|marketing|satpam|guard|security officer|keamanan gedung|sipil|civil|restaurant|waiter|"
                     r"driver|kurir|teller|accounting|akuntan|telemarketing|admin gudang|warehouse|psikolog\w*|recruit\w*|alat tes|"
                     r"interior|lecturer|dosen|tutor|pengajar|instruktur|trainer|account manager|presales|pre-sales|"
                     r"area java|jawa (timur|barat|tengah)|east java|west java|central java)\b", re.I)

aktif = profesi[profesi["aktif"].astype(int) == 1]
POLA = [(r["nama"], re.compile(r["pola"], re.I),
         re.compile(r["pola_kategori_ti"], re.I) if r["pola_kategori_ti"] else None) for _, r in aktif.iterrows()]

def klasifikasi(judul, kategori_ti):
    j = str(judul)
    if EXCLUDE.search(j):
        return "Lainnya"
    for nama, pola, tambahan in POLA:             # 1) pola ketat, semua sumber
        if pola.search(j):
            return nama
    if kategori_ti:                               # 2) pola longgar, hanya sumber yang sudah pasti kategori TI
        for nama, pola, tambahan in POLA:
            if tambahan is not None and tambahan.search(j):
                return nama
    return "Lainnya"

# Cell 5 — Gabungkan lowongan dari semua sumber

In [ ]:
frames = []

# F327 – Kaggle Jobstreet 2021–2022 (kategori resmi JobStreet; deskripsi terpotong)
js = pd.read_csv(TABLE_DIR / "F327.csv", low_memory=False,
                 usecols=["id", "jobTitle", "categoriesName", "description", "postedAt", "locations", "companyName"])
js = js[js["categoriesName"].astype(str).str.startswith("Komputer")]
frames.append(pd.DataFrame({"sumber": "F327_jobstreet_2122", "id_lowongan": js["id"].astype(str),
    "judul": js["jobTitle"], "perusahaan": js["companyName"], "teks": js["description"].fillna(""),
    "tanggal": js["postedAt"], "lokasi": js["locations"], "level": "", "kategori_ti": True,
    "teks_lengkap": False}))

# F003 – all.csv (fungsi Komputer/TI; deskripsi lengkap)
al = pd.read_csv(TABLE_DIR / "F003.csv")
al = al[al["job_function"].astype(str).str.startswith("Komputer")]
al = al.drop_duplicates(["job_title", "job_description"])
frames.append(pd.DataFrame({"sumber": "F003_jobdesc_id", "id_lowongan": al["id"].astype(str),
    "judul": al["job_title"], "perusahaan": "", "teks": al["job_description"].fillna(""),
    "tanggal": "", "lokasi": al["location"], "level": al["career_level"], "kategori_ti": True,
    "teks_lengkap": True}))

# F087 – GitHub multi-portal 2015–2022 (tanpa kategori; deskripsi + requirement lengkap)
gh = pd.read_csv(TABLE_DIR / "F087.csv", low_memory=False)
gh = gh.drop_duplicates("link")
frames.append(pd.DataFrame({"sumber": "F087_multiportal_1522", "id_lowongan": gh["link"],
    "judul": gh["title"], "perusahaan": gh["company"],
    "teks": gh["requirement"].fillna("") + " \n " + gh["description"].fillna(""),
    "tanggal": gh["time"], "lokasi": gh["location"], "level": gh["job_level"], "kategori_ti": False,
    "teks_lengkap": True}))

# Zenodo 2025 – profesi data (Glints: tag skill; JobStreet: deskripsi)
zg = pd.read_csv(TABLE_DIR / "F330__glints_gabungan.csv")
frames.append(pd.DataFrame({"sumber": "F330_glints_2025", "id_lowongan": zg["job_uuid"],
    "judul": zg["title"], "perusahaan": zg["company"], "teks": zg["skills"].fillna(""),
    "tanggal": "2025-03/04", "lokasi": zg["Location"], "level": zg["Experience"], "kategori_ti": True,
    "teks_lengkap": True}))
zj = pd.read_csv(TABLE_DIR / "F333__jobstreet_gabungan.csv")
frames.append(pd.DataFrame({"sumber": "F333_jobstreet_2025", "id_lowongan": zj["Link"],
    "judul": zj["Job Title"], "perusahaan": zj["Company Name"], "teks": zj["Job Description"].fillna(""),
    "tanggal": "2025-03/04", "lokasi": zj["Location"], "level": "", "kategori_ti": True,
    "teks_lengkap": True}))

low = pd.concat(frames, ignore_index=True)
low["profesi"] = [klasifikasi(j, k) for j, k in zip(low["judul"], low["kategori_ti"])]
# Zenodo: buang data entry / annotator (bukan profesi target)
low.loc[low["judul"].astype(str).str.contains(r"data entry|annotat|anotasi", case=False), "profesi"] = "Lainnya"

ti = low[low["profesi"] != "Lainnya"].copy()
ti.to_csv(OUT_DIR / "lowongan_ti.csv", index=False)
print("Total baris dibaca:", f"{len(low):,}", "| lowongan profesi target:", f"{len(ti):,}")
display(low.groupby("sumber")["profesi"].apply(lambda s: (s != "Lainnya").sum()).rename("lowongan_target"))

# Cell 6 — Volume lowongan per profesi

In [ ]:
vol = pd.crosstab(ti["profesi"], ti["sumber"])
vol["Total (bisa tumpang tindih)"] = vol.sum(axis=1)
vol["% F327"] = (vol["F327_jobstreet_2122"] / vol["F327_jobstreet_2122"].sum() * 100).round(1)
vol = vol.sort_values("F327_jobstreet_2122", ascending=False)
vol.to_csv(OUT_DIR / "demand_volume_profesi.csv")
display(vol)

# Cell 7 — Kamus skill (file konfigurasi)

In [ ]:
SKILL_CSV = OUT_DIR / "kamus_skill.csv"
SKILL_DEFAULT = [
 # skill, kategori, pola regex
 ("Python","Bahasa pemrograman",r"python"),("Java","Bahasa pemrograman",r"\bjava\b(?!script)"),
 ("JavaScript","Bahasa pemrograman",r"javascript|\bjs\b"),("TypeScript","Bahasa pemrograman",r"typescript"),
 ("PHP","Bahasa pemrograman",r"\bphp"),("Go","Bahasa pemrograman",r"golang|\bgo lang"),
 ("C#","Bahasa pemrograman",r"c#|c sharp"),("C/C++","Bahasa pemrograman",r"c\+\+"),
 ("Kotlin","Bahasa pemrograman",r"kotlin"),("Swift","Bahasa pemrograman",r"\bswift\b"),
 ("Dart","Bahasa pemrograman",r"\bdart\b"),("R","Bahasa pemrograman",r"\br programming|\bbahasa r\b|\br studio|rstudio"),
 ("Ruby","Bahasa pemrograman",r"\bruby"),("Scala","Bahasa pemrograman",r"\bscala\b"),
 ("HTML/CSS","Web",r"html|css"),("Laravel","Web",r"laravel"),("CodeIgniter","Web",r"codeigniter|\bci3\b"),
 ("React","Web",r"react(?!\s*native)|reactjs"),("Vue.js","Web",r"\bvue"),("Angular","Web",r"angular"),
 ("Node.js","Web",r"node\.?js|nodejs|\bnode\b"),("Express.js","Web",r"express\.?js"),
 ("Spring Boot","Web",r"spring ?boot|\bspring\b"),("Django/Flask","Web",r"django|flask"),
 (".NET","Web",r"\.net|asp\.net|dotnet"),("Next.js","Web",r"next\.?js"),("WordPress","Web",r"wordpress"),
 ("REST API","Web",r"rest ?api|restful|\bapi\b"),("GraphQL","Web",r"graphql"),("Microservices","Web",r"microservice"),
 ("Flutter","Mobile",r"flutter"),("React Native","Mobile",r"react ?native"),("Android","Mobile",r"android"),
 ("iOS","Mobile",r"\bios\b"),("Firebase","Mobile",r"firebase"),
 ("SQL","Basis data",r"\bsql\b|t-sql|pl/sql"),("MySQL","Basis data",r"mysql"),("PostgreSQL","Basis data",r"postgre"),
 ("Oracle DB","Basis data",r"oracle"),("SQL Server","Basis data",r"sql server|mssql|ms sql"),
 ("MongoDB","Basis data",r"mongo"),("NoSQL","Basis data",r"nosql"),("Redis","Basis data",r"redis"),
 ("Backup & Recovery","Basis data",r"backup|recovery"),("Database design","Basis data",r"database design|desain database|erd\b|normali[sz]"),
 ("Git","Tools & DevOps",r"\bgit\b|github|gitlab|bitbucket"),("Docker","Tools & DevOps",r"docker"),
 ("Kubernetes","Tools & DevOps",r"kubernetes|\bk8s\b"),("CI/CD","Tools & DevOps",r"ci/cd|ci cd|jenkins|continuous integration"),
 ("Linux","Tools & DevOps",r"linux|ubuntu|centos|debian|red ?hat"),("AWS","Cloud",r"\baws\b|amazon web services"),
 ("GCP","Cloud",r"\bgcp\b|google cloud"),("Azure","Cloud",r"azure"),("Terraform/Ansible","Tools & DevOps",r"terraform|ansible"),
 ("Machine Learning","Data & AI",r"machine learning"),("Deep Learning","Data & AI",r"deep learning|neural network"),
 ("TensorFlow/PyTorch","Data & AI",r"tensorflow|pytorch"),("Pandas/NumPy","Data & AI",r"pandas|numpy"),
 ("Statistics","Data & AI",r"statisti"),("Data Visualization","Data & AI",r"visuali[sz]|dashboard"),
 ("Tableau","Data & AI",r"tableau"),("Power BI","Data & AI",r"power ?bi"),("Excel","Data & AI",r"\bexcel\b"),
 ("ETL","Data & AI",r"\betl\b|data pipeline"),("Spark/Hadoop","Data & AI",r"spark|hadoop|hive"),
 ("NLP","Data & AI",r"\bnlp\b|natural language"),("Computer Vision","Data & AI",r"computer vision"),
 ("Networking (TCP/IP)","Jaringan",r"tcp/ip|tcp ip|networking|jaringan komputer|\blan\b|\bwan\b"),
 ("Routing & Switching","Jaringan",r"routing|switching|router|switch"),("Cisco (CCNA)","Jaringan",r"cisco|ccna|ccnp"),
 ("Mikrotik","Jaringan",r"mikrotik|mtcna"),("Firewall","Keamanan",r"firewall|fortigate|palo alto"),
 ("VPN","Jaringan",r"\bvpn\b"),("Windows Server","Jaringan",r"windows server|active directory"),
 ("Virtualization","Jaringan",r"vmware|virtuali[sz]|hyper-v|proxmox"),
 ("Network Security","Keamanan",r"network security|keamanan jaringan"),("Penetration Testing","Keamanan",r"penetration|pentest|vulnerability|kerentanan"),
 ("SIEM/SOC","Keamanan",r"siem|\bsoc\b|security operation"),("ISO 27001","Keamanan",r"iso ?27001|iso/iec 27001"),
 ("Manual Testing","Testing",r"manual test|test case|test scenario|test plan"),("Automation Testing","Testing",r"automation test|selenium|cypress|appium|katalon"),
 ("API Testing","Testing",r"postman|api test"),("Performance Testing","Testing",r"jmeter|performance test|load test"),
 ("Figma","Desain",r"figma"),("Adobe XD/Sketch","Desain",r"adobe xd|\bsketch\b"),("Wireframe/Prototype","Desain",r"wireframe|prototyp|mockup|mock-up"),
 ("User Research","Desain",r"user research|usability|user testing"),("Adobe Photoshop/Illustrator","Desain",r"photoshop|illustrator"),
 ("UML","Analisis sistem",r"\buml\b|use case|activity diagram|sequence diagram"),("Requirement Analysis","Analisis sistem",r"requirements? (analysis|gathering|engineering|elicitation)|analisa kebutuhan|analisis kebutuhan|kebutuhan sistem|business process|proses bisnis|\bbrd\b|\bfsd\b|user stor"),
 ("ERP/SAP","Analisis sistem",r"\berp\b|\bsap\b|odoo"),
 ("Hardware Troubleshooting","IT Support",r"troubleshoot|hardware|perangkat keras"),("Helpdesk/Ticketing","IT Support",r"helpdesk|help desk|ticketing|service desk"),
 ("Windows OS","IT Support",r"windows(?! server)"),("Microsoft Office","IT Support",r"ms office|microsoft office"),
 ("ITIL","IT Support",r"\bitil\b"),
 ("Agile/Scrum","Metodologi",r"agile|scrum|kanban|sprint"),("SDLC","Metodologi",r"sdlc|software development life ?cycle"),
 ("OOP","Metodologi",r"\boop\b|object[- ]oriented|berorientasi objek"),("Design Pattern/Clean Code","Metodologi",r"design pattern|clean code|solid principle"),
 ("Project Management","Metodologi",r"project management|manajemen proyek|\bpmp\b|prince2"),
 ("Komunikasi","Soft skill",r"communication|komunikasi"),("Kerja tim","Soft skill",r"team ?work|team player|kerja sama|kerjasama|bekerja dalam tim"),
 ("Problem solving","Soft skill",r"problem solving|pemecahan masalah|analytical|analitis"),("Bahasa Inggris","Soft skill",r"english|bahasa inggris"),
]
if SKILL_CSV.exists():
    kamus = pd.read_csv(SKILL_CSV, keep_default_na=False)
    print("Memakai kamus_skill.csv yang sudah ada.")
else:
    kamus = pd.DataFrame(SKILL_DEFAULT, columns=["skill", "kategori", "pola"])
    kamus.to_csv(SKILL_CSV, index=False)
    print("kamus_skill.csv dibuat:", len(kamus), "skill")       # 105
display(kamus.groupby("kategori").size().rename("jumlah_skill"))

# Cell 8 — Hitung frekuensi skill per profesi

In [ ]:
POLA_SKILL = [(r["skill"], r["kategori"], re.compile(r["pola"], re.I)) for _, r in kamus.iterrows()]

teks_df = ti[ti["teks_lengkap"]].copy()                  # F327 dikecualikan: deskripsinya terpotong
teks_df["teks_norm"] = teks_df["teks"].astype(str).str.replace(r"([a-z])([A-Z])", r"\1 \2", regex=True)  # pisahkan kata menempel

rows = []
for (prof, src), g in teks_df.groupby(["profesi", "sumber"]):
    n = len(g)
    for skill, kat, pola in POLA_SKILL:
        k = int(g["teks_norm"].str.contains(pola).sum())
        if k:
            rows.append({"profesi": prof, "sumber": src, "skill": skill, "kategori": kat, "n_lowongan": k, "n_total": n})
ds = pd.DataFrame(rows)

agg = ds.groupby(["profesi", "skill", "kategori"], as_index=False)[["n_lowongan"]].sum()
tot = teks_df.groupby("profesi").size().rename("n_total")
agg = agg.merge(tot, left_on="profesi", right_index=True)
agg["persen"] = (agg["n_lowongan"] / agg["n_total"] * 100).round(1)
agg = agg.sort_values(["profesi", "persen"], ascending=[True, False])
agg.to_csv(OUT_DIR / "demand_skill_profesi.csv", index=False)
ds.to_csv(OUT_DIR / "demand_skill_profesi_per_sumber.csv", index=False)
print("Lowongan berteks lengkap yang dianalisis:", f"{len(teks_df):,}")    # 8.272
display(tot.rename("lowongan_berteks_lengkap").to_frame())

# Cell 9 — Top-10 skill teknis dan soft skill per profesi

In [ ]:
TOP = 10
top = (agg[agg["kategori"] != "Soft skill"].groupby("profesi").head(TOP)
       .assign(teks=lambda d: d["skill"] + " (" + d["persen"].astype(str) + "%)")
       .groupby("profesi")["teks"].apply(lambda s: ", ".join(s)))
pd.set_option("display.max_colwidth", 400)
display(top.to_frame(f"Top-{TOP} skill teknis"))
soft = agg[agg["kategori"] == "Soft skill"].pivot_table(index="profesi", columns="skill", values="persen").fillna(0)
display(soft)

# Cell 10 — Sampel validasi manual dan log

In [ ]:
# Sampel acak untuk validasi manual ketepatan klasifikasi judul -> profesi
sampel = (ti.sample(frac=1, random_state=42).groupby("profesi").head(20)
            .sort_values("profesi")[["profesi", "sumber", "judul"]])
sampel["benar_1_salah_0"] = ""
sampel.to_csv(OUT_DIR / "validasi_klasifikasi_sampel.csv", index=False)
print("Sampel validasi:", len(sampel), "baris ->", OUT_DIR / "validasi_klasifikasi_sampel.csv")

log = {"tanggal": datetime.datetime.now().isoformat(timespec="seconds"),
       "profesi_aktif": aktif["nama"].tolist(),
       "lowongan_target_per_sumber": ti["sumber"].value_counts().to_dict(),
       "lowongan_berteks_lengkap": int(len(teks_df)), "jumlah_skill_kamus": int(len(kamus))}
with open(LOG_DIR / "tahap3_log.json", "w") as fp:
    json.dump(log, fp, indent=2, ensure_ascii=False)
print(json.dumps(log, indent=2, ensure_ascii=False))

---
# Tahap 4 — Pemetaan kompetensi ke profesi
*Sumber: `04_mapping.ipynb`*

# Cell 1 — Mount

# Cell 2 — Path dan data

In [ ]:
import re, json, datetime, shutil
from pathlib import Path
import pandas as pd

PROJECT_DIR = Path("/content/drive/MyDrive/Skripsi_RAG")
INTERIM_DIR = PROJECT_DIR / "01_interim"
OUT_DIR     = PROJECT_DIR / "02_processed"
LOG_DIR     = PROJECT_DIR / "logs"

manifest = pd.read_csv(LOG_DIR / "manifest.csv")
docs     = pd.read_json(INTERIM_DIR / "extracted_docs.jsonl", lines=True)
print("Record JSONL:", len(docs))

# Cell 3 — Kamus skill versi 2

In [ ]:
SKILL_CSV = OUT_DIR / "kamus_skill.csv"
if not (OUT_DIR / "kamus_skill_v1.csv").exists():
    shutil.copy(SKILL_CSV, OUT_DIR / "kamus_skill_v1.csv")       # simpan versi lama
kamus = pd.read_csv(OUT_DIR / "kamus_skill_v1.csv", keep_default_na=False)

UBAH = {   # pola baru: tambah istilah Indonesia, buang pola yang terlalu umum
 "Routing & Switching": r"routing|switching|\brouter\b|\bvlan\b",
 "Penetration Testing": r"penetration|pentest|uji penetrasi|vulnerability assessment",
 "Manual Testing": r"manual test|test case|test scenario|test plan|black ?box|white ?box|kasus uji|skenario (uji|pengujian)|pengujian perangkat lunak",
 "Automation Testing": r"automation test|pengujian otomatis|selenium|cypress|appium|katalon",
 "Requirement Analysis": r"requirements? (analysis|gathering|engineering|elicitation)|analisa kebutuhan|analisis kebutuhan|elisitasi|spesifikasi kebutuhan|\bsrs\b|kebutuhan sistem|business process|proses bisnis|\bbrd\b|\bfsd\b|user stor",
 "UML": r"\buml\b|use case|activity diagram|sequence diagram|class diagram|diagram (kelas|aktivitas|sekuen)",
 "Database design": r"database design|desain database|perancangan basis data|\berd\b|entity relationship|normali[sz]",
 "SQL": r"\bsql\b|t-sql|pl/sql|\bddl\b|\bdml\b",
 "Networking (TCP/IP)": r"tcp/ip|tcp ip|networking|jaringan komputer|\blan\b|\bwan\b|model osi|\bosi layer|subnet|pengalamatan ip|ip address",
 "Machine Learning": r"machine learning|pembelajaran mesin",
 "Deep Learning": r"deep learning|neural network|jaringan saraf|pembelajaran mendalam",
 "Data Visualization": r"visuali[sz]|dashboard",
 "Wireframe/Prototype": r"wireframe|prototyp|prototipe|purwarupa|mockup|mock-up",
 "User Research": r"user research|usability|user testing|pengujian kegunaan|\bpersona\b",
 "Hardware Troubleshooting": r"troubleshoot|perbaikan perangkat|instalasi perangkat",
}
TAMBAH = [
 ("Cloud computing (konsep)", "Cloud", r"cloud computing|komputasi awan|\biaas\b|\bpaas\b|\bsaas\b"),
 ("Information Security", "Keamanan", r"information security|keamanan informasi|kriptografi|cryptograph|enkripsi|encryption"),
 ("Data Mining", "Data & AI", r"data mining|penambangan data|clustering|\bk-?means\b"),
 ("IoT", "Jaringan", r"internet of things|\biot\b|arduino|raspberry|mikrokontroler|microcontroller"),
]
for s, p in UBAH.items():
    assert (kamus["skill"] == s).any(), s
    kamus.loc[kamus["skill"] == s, "pola"] = p
for s, k, p in TAMBAH:
    if not (kamus["skill"] == s).any():
        kamus.loc[len(kamus)] = [s, k, p]
kamus.to_csv(SKILL_CSV, index=False)
print("kamus_skill.csv versi 2:", len(kamus), "skill")       # 109
POLA_SKILL = [(r["skill"], r["kategori"], re.compile(r["pola"], re.I)) for _, r in kamus.iterrows()]

# Cell 4 — Hitung ulang demand skill dengan kamus versi 2

In [ ]:
ti = pd.read_csv(OUT_DIR / "lowongan_ti.csv", low_memory=False)
teks_df = ti[ti["teks_lengkap"]].copy()
teks_df["teks_norm"] = teks_df["teks"].astype(str).str.replace(r"([a-z])([A-Z])", r"\1 \2", regex=True)

rows = []
for (prof, src), g in teks_df.groupby(["profesi", "sumber"]):
    for skill, kat, pola in POLA_SKILL:
        k = int(g["teks_norm"].str.contains(pola).sum())
        if k:
            rows.append({"profesi": prof, "sumber": src, "skill": skill, "kategori": kat, "n_lowongan": k, "n_total": len(g)})
ds = pd.DataFrame(rows)
agg = ds.groupby(["profesi", "skill", "kategori"], as_index=False)[["n_lowongan"]].sum()
agg = agg.merge(teks_df.groupby("profesi").size().rename("n_total"), left_on="profesi", right_index=True)
agg["persen"] = (agg["n_lowongan"] / agg["n_total"] * 100).round(1)
agg = agg.sort_values(["profesi", "persen"], ascending=[True, False])
agg.to_csv(OUT_DIR / "demand_skill_profesi.csv", index=False)
ds.to_csv(OUT_DIR / "demand_skill_profesi_per_sumber.csv", index=False)
print("demand_skill_profesi.csv diperbarui:", len(agg), "baris")

# Cell 5 — Petakan RPS ke kode mata kuliah

In [ ]:
RPS_KE_KODE = {   # nomor depan nama file RPS -> kode mata kuliah (Kurikulum MBKM 2021)
 "11": "TIK2105", "12": "TIK2103", "13": "TI210101", "14": "TI210102", "16": "TIK2107", "17": "TI210104",
 "21": "TI210201", "22": "TI210202", "23": "TI210203", "24": "TI210204", "25": "TI210205", "27": "TIK2106",
 "28": "TI210207", "29": "TIK2104", "30": "TI210310", "33": "TI210303", "34": "TI210409", "35": "TI210305",
 "36": "TI210306", "38": "TI210308", "39": "TI210309", "41": "TI210401", "42": "TI210402", "43": "TI210403",
 "44": "TI210404", "45": "TI210405", "47": "TI210407", "49": "TI210304", "50": "TI210410", "61": "TI210601",
 "62": "TI210602", "64": "TI210604", "67": "TIK2108",
}
mk = pd.read_csv(INTERIM_DIR / "manual" / "F031_mata_kuliah_kurikulum2021.csv", dtype=str)
mk = mk[mk["induk"].isna()].drop_duplicates("kode")          # mata kuliah utama saja

rps = docs[(docs["kelompok"] == "rps") & (docs["status"] == "pakai")]
rps_txt = rps.groupby(["file_id", "file_name"])["text"].apply(lambda s: "\n".join(s)).reset_index()
rps_txt["kode"] = rps_txt["file_name"].str.extract(r"^(\d+)")[0].map(RPS_KE_KODE)
rps_txt = rps_txt.merge(mk[["kode", "nama_mata_kuliah", "semester"]], on="kode", how="left")
assert rps_txt["kode"].notna().all(), rps_txt[rps_txt["kode"].isna()]
display(rps_txt[["file_id", "file_name", "kode", "nama_mata_kuliah", "semester"]])

teknis = mk[~mk["kode"].str.startswith("PNJ") & ~mk["nama_mata_kuliah"].str.contains(
    "Bahasa|English|Kepemimpinan|Kewirausahaan|Metodologi|Merdeka|Magang|Skripsi|Etika", case=False)]
tanpa_rps = teknis[~teknis["kode"].isin(rps_txt["kode"])]
print("\nMata kuliah teknis TANPA RPS di dataset:", len(tanpa_rps))      # 5
display(tanpa_rps[["semester", "kode", "nama_mata_kuliah"]])

# Cell 6 — Kandidat skill per mata kuliah beserta bukti

In [ ]:
def bukti(teks, pola, lebar=70):
    m = pola.search(teks)
    if not m:
        return ""
    s = max(0, m.start() - lebar); e = min(len(teks), m.end() + lebar)
    return " ".join(teks[s:e].split())

rows = []
for _, r in rps_txt.iterrows():
    t = r["text"]
    for skill, kat, pola in POLA_SKILL:
        n = len(pola.findall(t))
        if n:
            rows.append({"kode": r["kode"], "nama_mata_kuliah": r["nama_mata_kuliah"], "semester": r["semester"],
                         "file_id": r["file_id"], "skill": skill, "kategori": kat, "n_sebutan": n,
                         "bukti": bukti(t, pola), "valid": 1, "catatan": ""})
kand = pd.DataFrame(rows).sort_values(["semester", "kode", "n_sebutan"], ascending=[True, True, False])
kand.to_csv(OUT_DIR / "mk_skill_kandidat.csv", index=False)
print("Kandidat pasangan MK–skill:", len(kand), "dari", kand["kode"].nunique(), "mata kuliah")    # 134 dari 31
display(kand.groupby("nama_mata_kuliah")["skill"].apply(lambda s: ", ".join(s)).to_frame("kandidat skill"))

# Cell 7 — Cek skill di dokumen standar kompetensi tiap profesi

In [ ]:
STANDAR = {   # potongan nama file SKKNI/okupasi per profesi
 "Software / Web Developer": ["2016-282", "2019-022", "TIK.DEV0615"],
 "Mobile Developer": ["2015-458"],
 "System Analyst": ["2017-044", "2015-090", "TIK.DEV0613"],
 "Database Administrator": ["2020-268", "TIK.DSC0604"],
 "QA / Software Tester": ["2017-047"],
 "UI/UX Designer": ["2018-107", "2023-126", "TIK.SRV0506"],
 "Network / System Administrator": ["2016-321", "2017-082", "TIK.INF0402", "TIK.INF0403", "TIK.INF0502",
                                    "TIK.INF0504", "TIK.INF0506", "TIK.INF0602", "TIK.INF0605", "TIK.INF0609",
                                    "TIK.INF0604-DESAINER JARINGAN (NETWORK DESIGNER).pdf"],
 "IT Support / Helpdesk": ["2016-285", "kmnaker030"],
 "Cyber Security": ["kmnaker191", "kmnaker120", "2022-023", "Web & Mobile Development 2020-391", "TIK.INF0611", "TIK.SEC0603"],
 "DevOps / Cloud Engineer": ["2023-102"],
 "Data Analyst / BI": ["SKKNI Data & Database Management 2020-299"],
 "Data Engineer": ["2020-268", "SKKNI Data & Database Management 2020-299"],
 "Data Scientist / ML / AI Engineer": ["SKKNI Data & Database Management 2020-299", "kmnaker103"],
 "IT Project/Product Manager": ["kmnaker200"],
}
# SKKNI SOC 391/2020 kini dipakai untuk Cyber Security (sebelumnya "perlu_keputusan")
mask = manifest["file_name"].str.contains("Web & Mobile Development 2020-391", regex=False)
manifest.loc[mask, ["status", "alasan"]] = ["pakai", "Isi = SKKNI SOC 391/2020; dipakai untuk profesi Cyber Security"]
manifest.to_csv(LOG_DIR / "manifest.csv", index=False)

teks_file = docs.groupby("file_id")["text"].apply(lambda s: "\n".join(s))
std_rows = []
for prof, kunci in STANDAR.items():
    fids = []
    for k in kunci:
        hit = manifest[manifest["file_name"].str.contains(k, regex=False)
                       & manifest["kelompok"].isin(["skkni", "okupasi"])
                       & (manifest["status"] != "salinan")]["file_id"].tolist()
        assert hit, f"Tidak ketemu: {k}"
        fids += hit
    fids = sorted(set(fids))
    t = "\n".join(teks_file.get(f, "") for f in fids)
    for skill, kat, pola in POLA_SKILL:
        std_rows.append({"profesi": prof, "skill": skill, "di_standar": int(bool(pola.search(t)))})
    print(f"{prof:35} {len(fids):2} dokumen: {', '.join(fids)}")
std = pd.DataFrame(std_rows)

# Cell 8 — Matriks skill–profesi dan cakupan kurikulum

In [ ]:
AMBANG = 10          # skill dianggap "diminta" bila muncul di >= 10% lowongan profesi tsb

verif_xlsx = OUT_DIR / "mk_skill_verifikasi.xlsx"
if verif_xlsx.exists():
    v1 = pd.read_excel(verif_xlsx, sheet_name="Verifikasi", dtype=str).fillna("")
    v2 = pd.read_excel(verif_xlsx, sheet_name="Tambahan", dtype=str).fillna("")
    v2 = v2[v2["nama_mata_kuliah"].str.strip() != ""]
    cols = ["kode", "nama_mata_kuliah", "skill", "valid", "sumber_bukti", "catatan"]
    mks_all = pd.concat([v1[cols].assign(asal="kandidat"), v2[cols].assign(asal="tambahan")], ignore_index=True)
    mks_all["valid"] = mks_all["valid"].str.replace(".0", "", regex=False)
    belum = (~mks_all["valid"].isin(["0", "1"])).sum()
    salah_nama = set(mks_all["skill"]) - set(kamus["skill"])
    assert not salah_nama, f"Nama skill tidak ada di kamus: {salah_nama}"
    print(f"Memakai {verif_xlsx.name}: {len(v1)} kandidat + {len(v2)} tambahan | belum diisi: {belum}")
    mks_all.to_csv(OUT_DIR / "mk_skill_terverifikasi.csv", index=False)
    mks = mks_all[mks_all["valid"] == "1"]
    print(mks.groupby(["asal", "sumber_bukti"]).size().rename("pasangan_valid"))
else:
    mks = kand.assign(sumber_bukti="RPS")
    print("PERINGATAN: belum ada mk_skill_verifikasi.xlsx; memakai kandidat otomatis (belum diverifikasi).")

def peta_mk(df):
    return df.groupby("skill")["nama_mata_kuliah"].apply(lambda s: "; ".join(sorted(set(s))))
mk_rps = peta_mk(mks[mks["sumber_bukti"].str.startswith("RPS") | (mks["sumber_bukti"] == "Kurikulum")])   # kurikulum formal (RPS/dokumen kurikulum)
mk_all = peta_mk(mks)                                                # formal + pengalaman praktikum

matriks = agg[agg["kategori"] != "Soft skill"].merge(std, on=["profesi", "skill"], how="left")
matriks["di_kurikulum_rps"] = matriks["skill"].isin(mk_rps.index).astype(int)
matriks["di_kurikulum_aktual"] = matriks["skill"].isin(mk_all.index).astype(int)
matriks["mk_pendukung"] = matriks["skill"].map(mk_all).fillna("")
matriks["diminta"] = (matriks["persen"] >= AMBANG).astype(int)
matriks = matriks.sort_values(["profesi", "persen"], ascending=[True, False])
matriks.to_csv(OUT_DIR / "skill_profesi_matriks.csv", index=False)

def cakupan(g, kol):
    return round((g["persen"] * g[kol]).sum() / g["persen"].sum() * 100, 1)
d = matriks[matriks["diminta"] == 1]
cov = d.groupby("profesi").apply(lambda g: pd.Series({
    "skill_diminta": len(g),
    "cakupan_RPS_%": cakupan(g, "di_kurikulum_rps"),
    "cakupan_aktual_%": cakupan(g, "di_kurikulum_aktual"),
    "ada_di_standar": int(g["di_standar"].sum()),
    "gap_skill": ", ".join(g.loc[g["di_kurikulum_aktual"] == 0, "skill"]),
}), include_groups=False).sort_values("cakupan_aktual_%", ascending=False)
cov.to_csv(OUT_DIR / "cakupan_kurikulum_profesi.csv")
pd.set_option("display.max_colwidth", 200)
display(cov)

# Cell 9 — Log

In [ ]:
log = {"tanggal": datetime.datetime.now().isoformat(timespec="seconds"),
       "kamus_versi": 2, "jumlah_skill": int(len(kamus)), "ambang_diminta_%": AMBANG,
       "rps_dipetakan": int(len(rps_txt)), "mk_teknis_tanpa_rps": tanpa_rps["nama_mata_kuliah"].tolist(),
              "pasangan_mk_skill_valid": int(len(mks)), "terverifikasi": verif_xlsx.exists()}
with open(LOG_DIR / "tahap4_log.json", "w") as fp:
    json.dump(log, fp, indent=2, ensure_ascii=False)
print(json.dumps(log, indent=2, ensure_ascii=False))

---
# Tahap 5 — Knowledge base
*Sumber: `05_kb.ipynb`*

# Cell 1 — Mount

# Cell 2 — Path dan data

In [ ]:
import re, json, datetime
from pathlib import Path
import pandas as pd

PROJECT_DIR = Path("/content/drive/MyDrive/Skripsi_RAG")
INTERIM_DIR = PROJECT_DIR / "01_interim"
PROC_DIR    = PROJECT_DIR / "02_processed"
KB_DIR      = PROJECT_DIR / "03_kb"
LOG_DIR     = PROJECT_DIR / "logs"
KB_DIR.mkdir(exist_ok=True)

manifest = pd.read_csv(LOG_DIR / "manifest.csv")
docs     = pd.read_json(INTERIM_DIR / "extracted_docs.jsonl", lines=True)
matriks  = pd.read_csv(PROC_DIR / "skill_profesi_matriks.csv", keep_default_na=False)
vol      = pd.read_csv(PROC_DIR / "demand_volume_profesi.csv")
mks      = pd.read_csv(PROC_DIR / "mk_skill_terverifikasi.csv", dtype=str, keep_default_na=False)
mks      = mks[mks["valid"] == "1"]
mk_list  = pd.read_csv(INTERIM_DIR / "manual" / "F031_mata_kuliah_kurikulum2021.csv", dtype=str)
mk_list  = mk_list[mk_list["induk"].isna()].drop_duplicates("kode")
profil_l = pd.read_csv(INTERIM_DIR / "manual" / "F031_profil_lulusan_kurikulum2021.csv", dtype=str)
print("docs:", len(docs), "| matriks:", len(matriks), "| MK-skill valid:", len(mks))     # MK-skill valid: 62

# Cell 3 — Fungsi pembersih dan pemotong

In [ ]:
def bersih(t):
    """Satukan baris terpotong dan rapikan spasi (teks PDF sering terpecah per kata)."""
    return re.sub(r"\s+", " ", str(t)).strip()

def potong(teks, maks=1500, overlap=200):
    """Potong teks panjang di batas kalimat/kata; tiap potongan <= maks karakter."""
    teks = bersih(teks)
    if len(teks) <= maks:
        return [teks]
    out, i = [], 0
    while i < len(teks):
        j = min(i + maks, len(teks))
        if j < len(teks):
            k = max(teks.rfind(". ", i, j), teks.rfind("; ", i, j))
            j = k + 1 if k > i + maks * 0.5 else (teks.rfind(" ", i, j) if teks.rfind(" ", i, j) > i else j)
        out.append(teks[i:j].strip())
        if j >= len(teks):
            break
        i = max(j - overlap, i + 1)
    return out

CHUNKS = []
def tambah(jenis, judul, teks, profesi, file_id, sumber, halaman="", maks=1500):
    for n, p in enumerate(potong(teks, maks=maks), 1):
        CHUNKS.append({"chunk_id": f"{jenis}-{len(CHUNKS)+1:05d}", "jenis": jenis, "judul": judul,
                       "teks": f"{judul}\n{p}", "profesi": profesi, "file_id": file_id,
                       "sumber": sumber, "halaman": str(halaman), "bagian": n})

# Cell 4 — Pemetaan profesi ke dokumen standar

In [ ]:
STANDAR = {
 "Software / Web Developer": ["2016-282", "2019-022", "TIK.DEV0615"],
 "Mobile Developer": ["2015-458"],
 "System Analyst": ["2017-044", "2015-090", "TIK.DEV0613"],
 "Database Administrator": ["2020-268", "TIK.DSC0604"],
 "QA / Software Tester": ["2017-047"],
 "UI/UX Designer": ["2018-107", "2023-126", "TIK.SRV0506"],
 "Network / System Administrator": ["2016-321", "2017-082", "TIK.INF0402", "TIK.INF0403", "TIK.INF0502",
                                    "TIK.INF0504", "TIK.INF0506", "TIK.INF0602", "TIK.INF0605", "TIK.INF0609",
                                    "TIK.INF0604-DESAINER JARINGAN (NETWORK DESIGNER).pdf"],
 "IT Support / Helpdesk": ["2016-285", "kmnaker030"],
 "Cyber Security": ["kmnaker191", "kmnaker120", "2022-023", "Web & Mobile Development 2020-391", "TIK.INF0611", "TIK.SEC0603"],
 "DevOps / Cloud Engineer": ["2023-102"],
 "Data Analyst / BI": ["SKKNI Data & Database Management 2020-299"],
 "Data Engineer": ["2020-268", "SKKNI Data & Database Management 2020-299"],
 "Data Scientist / ML / AI Engineer": ["SKKNI Data & Database Management 2020-299", "kmnaker103"],
 "IT Project/Product Manager": ["kmnaker200"],
}
rows = []
for prof, kunci in STANDAR.items():
    for k in kunci:
        hit = manifest[manifest["file_name"].str.contains(k, regex=False)
                       & manifest["kelompok"].isin(["skkni", "okupasi"]) & (manifest["status"] != "salinan")]
        for _, r in hit.iterrows():
            rows.append({"profesi": prof, "file_id": r["file_id"], "kelompok": r["kelompok"],
                         "file_name": r["file_name"], "isi_nomor_tahun": r["isi_nomor_tahun"], "isi_judul": r["isi_judul"]})
prof_std = pd.DataFrame(rows).drop_duplicates(["profesi", "file_id"])
prof_std.to_csv(PROC_DIR / "profesi_standar.csv", index=False)
FILE_PROFESI = prof_std.groupby("file_id")["profesi"].apply(list).to_dict()

def label_skkni(fid):
    r = manifest[manifest["file_id"] == fid].iloc[0]
    bidang = re.sub(r"(?i)^bidang( keahlian)?\s*", "", str(r["isi_judul"])).title()
    return f"SKKNI {r['isi_nomor_tahun']} – {bidang}"
display(prof_std.groupby("profesi").size().rename("dokumen_standar"))

# Cell 5 — A. Profil profesi

In [ ]:
okup_teks = docs[docs["kelompok"] == "okupasi"].groupby("file_id")["text"].apply(lambda s: bersih(" ".join(s)))
PROFIL_PRODI = {"Software / Web Developer": ["Software Developer", "PEMROGRAM SENIOR"],
                "Database Administrator": ["DATABASE ADMINISTRATOR"], "System Analyst": ["System Analyst"]}
v = vol.set_index("profesi")
for prof in matriks["profesi"].unique():
    m = matriks[(matriks["profesi"] == prof)]
    dm = m[m["diminta"] == 1]
    bagian = [f"Profesi: {prof}."]
    if prof in v.index:
        bagian.append(f"Kebutuhan industri: {int(v.loc[prof, 'F327_jobstreet_2122']):,} lowongan pada data JobStreet Indonesia "
                      f"Nov 2021–Ags 2022 ({v.loc[prof, '% F327']}% dari lowongan TI yang dipetakan).")
    for nama in PROFIL_PRODI.get(prof, []):
        pl = profil_l[profil_l["profil"] == nama]
        if len(pl):
            bagian.append(f"Profil lulusan Prodi D4 TI PNJ ({nama}): {pl['deskripsi'].iloc[0]}")
    okp = []
    for fid in prof_std[(prof_std["profesi"] == prof) & (prof_std["kelompok"] == "okupasi")]["file_id"]:
        t = okup_teks.get(fid, "")
        nm = re.search(r"Nama Okupasi\s+(.*?)\s+Area Fungsi", t)
        lv = re.search(r"Level Kualifikasi\s+(\d+)", t)
        if nm:
            okp.append(f"{re.sub(r'^[0-9A-Z.]+ - ', '', nm.group(1)).title()} (Level KKNI {lv.group(1) if lv else '-'})")
    if okp:
        bagian.append("Okupasi terkait di Peta Okupasi Nasional KOMDIGI: " + "; ".join(okp) + ".")
    bagian.append("Skill yang paling banyak diminta lowongan: " +
                  "; ".join(f"{r.skill} ({r.persen}%)" for r in dm.head(15).itertuples()) + ".")
    tc = dm[dm["di_kurikulum_rps"] == 1]
    if len(tc):
        bagian.append("Skill yang dibekali kurikulum PNJ: " +
                      "; ".join(f"{r.skill} (MK: {r.mk_pendukung})" for r in tc.itertuples()) + ".")
    gap = dm[dm["di_kurikulum_aktual"] == 0]
    if len(gap):
        bagian.append("Skill yang diminta industri tetapi tidak tercantum di kurikulum (perlu belajar mandiri/sertifikasi): " +
                      ", ".join(gap["skill"]) + ".")
    sk = prof_std[(prof_std["profesi"] == prof) & (prof_std["kelompok"] == "skkni")]["file_id"]
    if len(sk):
        bagian.append("Standar kompetensi acuan: " + "; ".join(label_skkni(f) for f in sk) + ".")
    tambah("profil_profesi", f"[Profil Profesi] {prof}", " ".join(bagian), [prof], "",
           "Disusun dari skill_profesi_matriks.csv, demand_volume_profesi.csv, profil lulusan, okupasi, SKKNI", maks=3000)
print("Profil profesi:", sum(c["jenis"] == "profil_profesi" for c in CHUNKS), "chunk")     # 14

# Cell 6 — B. Mata kuliah dan RPS

In [ ]:
skill_diminta = matriks[matriks["diminta"] == 1].groupby("skill")["profesi"].apply(lambda s: sorted(set(s))).to_dict()
rps = docs[(docs["kelompok"] == "rps") & (docs["status"] == "pakai")]
rps_txt = rps.groupby(["file_id", "file_name"])["text"].apply(lambda s: bersih(" ".join(s))).reset_index()
rps_txt["kode"] = rps_txt["file_name"].str.extract(r"^(\d+)")[0].map({
 "11": "TIK2105", "12": "TIK2103", "13": "TI210101", "14": "TI210102", "16": "TIK2107", "17": "TI210104",
 "21": "TI210201", "22": "TI210202", "23": "TI210203", "24": "TI210204", "25": "TI210205", "27": "TIK2106",
 "28": "TI210207", "29": "TIK2104", "30": "TI210310", "33": "TI210303", "34": "TI210409", "35": "TI210305",
 "36": "TI210306", "38": "TI210308", "39": "TI210309", "41": "TI210401", "42": "TI210402", "43": "TI210403",
 "44": "TI210404", "45": "TI210405", "47": "TI210407", "49": "TI210304", "50": "TI210410", "61": "TI210601",
 "62": "TI210602", "64": "TI210604", "67": "TIK2108"})
rps_by_kode = rps_txt.set_index("kode")

teknis = mk_list[~mk_list["kode"].str.startswith("PNJ") & ~mk_list["nama_mata_kuliah"].str.contains(
    "Bahasa|English|Kepemimpinan|Kewirausahaan|Metodologi|Merdeka|Magang|Skripsi|Etika", case=False)]
for _, r in teknis.iterrows():
    s = mks[mks["kode"] == r["kode"]]["skill"].unique().tolist()
    profs = sorted({p for x in s for p in skill_diminta.get(x, [])})
    desk = ""
    if r["kode"] in rps_by_kode.index:
        t = rps_by_kode.loc[r["kode"], "text"]
        md = re.search(r"Deskripsi Mata Kuliah\s*\|?\s*:?\s*\|?\s*(.{50,700}?)(?:\s*\|\s*[A-Z][a-z]+ [A-Z]|CAPAIAN|Capaian|$)", t)
        desk = md.group(1).strip(" |:") if md else ""
    teks = (f"Mata kuliah {r['nama_mata_kuliah']} (kode {r['kode']}), semester {r['semester']}, {r['sks_total']} SKS, "
            f"Kurikulum MBKM D4 Teknik Informatika PNJ 2021. "
            + (f"Deskripsi: {desk.rstrip('. ')}. " if desk else "")
            + (f"Skill yang dibekali (terverifikasi): {', '.join(s)}. " if s else "Tidak ada skill teknis industri yang terpetakan. ")
            + (f"Relevan untuk profesi: {', '.join(profs)}." if profs else ""))
    tambah("mata_kuliah", f"[Mata Kuliah] {r['nama_mata_kuliah']}", teks, profs,
           rps_by_kode.loc[r["kode"], "file_id"] if r["kode"] in rps_by_kode.index else "F031",
           f"Kurikulum MBKM 2021; RPS {r['nama_mata_kuliah']}" if r["kode"] in rps_by_kode.index else "Kurikulum MBKM 2021 (RPS tidak tersedia)")

def isi_rps(t):
    m = re.search(r"(Deskripsi Mata Kuliah|DESKRIPSI MATA KULIAH|Capaian Pembelajaran|CAPAIAN PEMBELAJARAN)", t)
    return t[m.start():] if m else t                       # buang bagian administratif (SPMI, tanda tangan)

for _, r in rps_txt.iterrows():
    nama = teknis.set_index("kode")["nama_mata_kuliah"].get(r["kode"], r["file_name"])
    s = mks[mks["kode"] == r["kode"]]["skill"].unique().tolist()
    profs = sorted({p for x in s for p in skill_diminta.get(x, [])})
    tambah("rps", f"[RPS] {nama}", isi_rps(r["text"]), profs, r["file_id"], f"RPS {nama} ({r['file_name']})")
print("Mata kuliah:", sum(c["jenis"] == "mata_kuliah" for c in CHUNKS), "| RPS:", sum(c["jenis"] == "rps" for c in CHUNKS))   # 38 | 332

# Cell 7 — C. SKKNI per unit kompetensi

In [ ]:
UNIT = re.compile(r"KODE UNIT\s*:\s*(\S+)\s*JUDUL UNIT\s*:\s*(.{5,250}?)\s*DESKRIPSI(?: UNIT)?\s*:", re.S)
skkni_ids = prof_std[prof_std["kelompok"] == "skkni"]["file_id"].unique()
fid22 = manifest.loc[manifest["file_name"].str.contains("2019-022", regex=False), "file_id"].iloc[0]
n_unit = {}
for fid in skkni_ids:
    if fid == fid22:
        continue                                                    # format berbeda, ditangani di bawah
    t = bersih(" ".join(docs[docs["file_id"] == fid].sort_values("unit_no")["text"]))
    starts = [m for m in UNIT.finditer(t)]
    lab = label_skkni(fid)
    for i, m in enumerate(starts):
        end = starts[i + 1].start() if i + 1 < len(starts) else len(t)
        judul = f"[{lab}] Unit {m.group(1)} – {bersih(m.group(2))}"
        tambah("skkni", judul, t[m.start():end], FILE_PROFESI.get(fid, []), fid, f"{lab}, unit {m.group(1)}")
    n_unit[lab] = len(starts)

# SKKNI 22/2019 (adopsi standar Australia): hanya unit berkode TI (ICA*/ICT*)
pg = docs[docs["file_id"] == fid22].sort_values("unit_no")
HEAD = re.compile(r"^\s*\d*\s*((?:ICA|ICT)[A-Z]{3}\d{3}[A-Z]?)\s+(.{5,120}?)\s{2,}")
cur, buf, start_pg, n22 = None, [], None, 0
def flush():
    global n22
    if cur:
        prof22 = (["Database Administrator", "Data Engineer"] if "DBS" in cur[0] else
                  ["Network / System Administrator"] if re.search(r"NWK|TEN|SAS", cur[0]) else ["Software / Web Developer"])
        tambah("skkni", f"[SKKNI 22/2019 – adopsi] Unit {cur[0]} – {cur[1]}", " ".join(buf)[:6000], prof22, fid22,
               f"SKKNI 22/2019, unit {cur[0]}", start_pg)
        n22 += 1
for _, r in pg.iterrows():
    first = r["text"][:250].replace("\n", "  ")
    m = HEAD.match(first)
    if m:
        flush(); cur, buf, start_pg = (m.group(1), bersih(m.group(2))), [], r["unit_no"]
    elif re.match(r"^\s*\d*\s*(BSB|HLT|CUS|SIR|TAE|CPC|FNS)[A-Z]{3}\d{3}", first):
        flush(); cur = None; buf = []                              # unit non-TI: berhenti mengumpulkan
    if cur:
        buf.append(bersih(r["text"]))
flush()
n_unit["SKKNI 22/2019 – unit TI saja"] = n22
display(pd.Series(n_unit, name="unit"))
print("SKKNI:", sum(c["jenis"] == "skkni" for c in CHUNKS), "chunk")      # 2056

# Cell 8 — D–F. Okupasi, kurikulum prodi, roadmap.sh

In [ ]:
for fid in prof_std[prof_std["kelompok"] == "okupasi"]["file_id"].unique():
    t = okup_teks.get(fid, "")
    t = re.sub(r"Platform Stankomdigi by KOMDIGI.*?Indonesia \d{4}-\s*\d{2}-\d{2} \d{2}:\d{2}:\d{2}", "", t)
    nm = re.search(r"Nama Okupasi\s+(.*?)\s+Area Fungsi", t)
    tambah("okupasi", f"[Okupasi KOMDIGI] {nm.group(1) if nm else fid}", t, FILE_PROFESI.get(fid, []), fid,
           "Peta Okupasi Nasional KOMDIGI (Stankomdigi)")

for fid, jenis_doc in [("F080", "CPL Prodi D4 TI PNJ"), ("F078", "Kompetensi Lulusan D4 TI PNJ")]:
    t = " ".join(docs[docs["file_id"] == fid]["text"])
    if t:
        tambah("kurikulum", f"[{jenis_doc}]", t, [], fid, jenis_doc)
for _, r in profil_l.drop_duplicates("profil").iterrows():
    cp = "; ".join(profil_l[profil_l["profil"] == r["profil"]]["cp_keterampilan_khusus"])
    tambah("kurikulum", f"[Profil Lulusan D4 TI PNJ] {r['profil']}", f"{r['deskripsi']} Capaian keterampilan khusus: {cp}",
           [], "F031", "Kurikulum MBKM 2021, tabel profil lulusan (transkripsi manual)")

ROADMAP = {"backend": ["Software / Web Developer"], "frontend": ["Software / Web Developer"],
           "full-stack": ["Software / Web Developer"], "software-architect": ["Software / Web Developer"],
           "software-design-architecture": ["Software / Web Developer"], "android": ["Mobile Developer"],
           "qa": ["QA / Software Tester"], "devops": ["DevOps / Cloud Engineer"],
           "mlops": ["DevOps / Cloud Engineer", "Data Scientist / ML / AI Engineer"],
           "cyber-security": ["Cyber Security"], "data-analyst": ["Data Analyst / BI"], "data-engineer": ["Data Engineer"],
           "ai-data-scientist": ["Data Scientist / ML / AI Engineer"]}
for _, r in docs[docs["kelompok"] == "roadmap"].iterrows():
    key = r["file_name"].replace(".pdf", "")
    tambah("roadmap", f"[Roadmap.sh] {key}", r["text"], ROADMAP.get(key, []), r["file_id"],
           f"roadmap.sh – {key} (referensi jalur belajar komunitas, bukan bukti demand)")

# Cell 9 — Simpan KB dan ringkasan

In [ ]:
kb = pd.DataFrame(CHUNKS)
kb["n_char"] = kb["teks"].str.len()
with open(KB_DIR / "kb_chunks.jsonl", "w", encoding="utf-8") as f:
    for rec in kb.to_dict("records"):
        f.write(json.dumps(rec, ensure_ascii=False) + "\n")
print("Total chunk:", len(kb), "| rata-rata karakter:", int(kb["n_char"].mean()), "| maks:", kb["n_char"].max())
display(kb.groupby("jenis").agg(chunk=("chunk_id", "size"), karakter=("n_char", "sum")))
display(kb.explode("profesi").groupby("profesi").size().rename("chunk_bertag").sort_values(ascending=False))

with open(KB_DIR / "profil_profesi.md", "w", encoding="utf-8") as f:
    for r in kb[kb["jenis"] == "profil_profesi"].itertuples():
        f.write(f"## {r.judul}\n\n{r.teks.split(chr(10), 1)[1]}\n\n")
print("Profil profesi untuk dibaca manusia:", KB_DIR / "profil_profesi.md")

log = {"tanggal": datetime.datetime.now().isoformat(timespec="seconds"), "total_chunk": int(len(kb)),
       "per_jenis": kb["jenis"].value_counts().to_dict(), "maks_karakter_chunk": 1500, "overlap": 200}
with open(LOG_DIR / "tahap5_log.json", "w") as fp:
    json.dump(log, fp, indent=2, ensure_ascii=False)

---
# Tahap 6 — Indeks retrieval
*Sumber: `06_index.ipynb`*

# Cell 1 — Mount dan instal

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!pip install -q sentence-transformers faiss-cpu

# Cell 2 — Path dan KB

In [ ]:
import json, datetime, time
from pathlib import Path
import numpy as np
import pandas as pd
import faiss

PROJECT_DIR = Path("/content/drive/MyDrive/Skripsi_RAG")
KB_DIR  = PROJECT_DIR / "03_kb"
LOG_DIR = PROJECT_DIR / "logs"

kb = pd.read_json(KB_DIR / "kb_chunks.jsonl", lines=True)
kb["profesi"] = kb["profesi"].apply(lambda x: x if isinstance(x, list) else [])
print("Chunk:", len(kb))          # 2514

# Cell 3 — Muat model embedding

In [ ]:
from sentence_transformers import SentenceTransformer
import torch

MODEL_NAME = "intfloat/multilingual-e5-base"
device = "cuda" if torch.cuda.is_available() else "cpu"
model = SentenceTransformer(MODEL_NAME, device=device)
model.max_seq_length = 512
print("Model:", MODEL_NAME, "| device:", device, "| dimensi:", model.get_sentence_embedding_dimension())

def embed_passages(texts, bs=64):
    # model keluarga E5 mensyaratkan awalan "passage: " untuk dokumen
    return model.encode(["passage: " + t for t in texts], batch_size=bs, normalize_embeddings=True,
                        show_progress_bar=True, convert_to_numpy=True).astype("float32")

def embed_query(q):
    # ... dan awalan "query: " untuk pertanyaan/profil pengguna
    return model.encode(["query: " + q], normalize_embeddings=True, convert_to_numpy=True).astype("float32")

# Cell 4 — Embed semua chunk dan simpan index

In [ ]:
t0 = time.time()
emb = embed_passages(kb["teks"].tolist())
print(f"Embedding selesai: {emb.shape} dalam {time.time() - t0:.0f} detik")

index = faiss.IndexFlatIP(emb.shape[1])          # inner product = cosine (vektor sudah dinormalisasi)
index.add(emb)
faiss.write_index(index, str(KB_DIR / "kb_index.faiss"))
np.save(KB_DIR / "kb_embeddings.npy", emb)
kb[["chunk_id", "jenis", "judul", "profesi", "file_id", "sumber", "halaman", "bagian", "teks"]].to_json(
    KB_DIR / "kb_meta.jsonl", orient="records", lines=True, force_ascii=False)
print("Tersimpan: kb_index.faiss, kb_embeddings.npy, kb_meta.jsonl | jumlah vektor:", index.ntotal)

# Cell 5 — Fungsi pencarian

In [ ]:
def cari(query, k=5, jenis=None, profesi=None):
    """Cari chunk paling mirip; opsional filter jenis (str/list) dan profesi."""
    q = embed_query(query)[0]
    mask = np.ones(len(kb), dtype=bool)
    if jenis is not None:
        mask &= kb["jenis"].isin([jenis] if isinstance(jenis, str) else jenis).to_numpy()
    if profesi is not None:
        mask &= kb["profesi"].apply(lambda ps: profesi in ps).to_numpy()
    idx = np.where(mask)[0]
    if len(idx) == 0:
        return pd.DataFrame()
    skor = emb[idx] @ q
    urut = np.argsort(-skor)
    hasil = kb.iloc[idx[urut]][["chunk_id", "jenis", "judul", "sumber", "teks"]].copy()
    hasil.insert(0, "skor", skor[urut].round(4))
    hasil = hasil.drop_duplicates("judul")           # satu unit/dokumen cukup muncul sekali
    return hasil.head(k).reset_index(drop=True)

def cari_bertingkat(profil_pengguna, n_profesi=3, k_standar=3, k_mk=3):
    """1) peringkat profesi dari profil profesi, 2) bukti SKKNI/okupasi per profesi, 3) mata kuliah terkait."""
    prof = cari(profil_pengguna, k=n_profesi, jenis="profil_profesi")
    prof["profesi"] = prof["judul"].str.replace("[Profil Profesi] ", "", regex=False)
    bukti = {p: cari(profil_pengguna, k=k_standar, jenis=["skkni", "okupasi"], profesi=p) for p in prof["profesi"]}
    mk = cari(profil_pengguna, k=k_mk, jenis="mata_kuliah")
    return prof, bukti, mk

# Cell 6 — Uji kewajaran (10 kueri)

In [ ]:
UJI = [
 ("Saya suka mendesain tampilan aplikasi, menguasai Figma dan pernah membuat wireframe serta prototipe.", "UI/UX Designer"),
 ("Saya senang mengonfigurasi router Mikrotik, mengatur VLAN, dan memelihara jaringan kantor.", "Network / System Administrator"),
 ("Saya suka menulis test case, mencari bug, dan mencoba Selenium untuk pengujian otomatis.", "QA / Software Tester"),
 ("Saya membuat aplikasi Android dengan Kotlin dan Flutter untuk tugas akhir semester.", "Mobile Developer"),
 ("Saya suka mengolah data dengan SQL dan Excel lalu membuat dashboard Power BI untuk laporan.", "Data Analyst / BI"),
 ("Saya tertarik membangun model machine learning dengan Python dan TensorFlow.", "Data Scientist / ML / AI Engineer"),
 ("Saya mengelola basis data MySQL, membuat backup, dan mengoptimalkan query.", "Database Administrator"),
 ("Saya membuat website dengan Laravel, React, dan REST API.", "Software / Web Developer"),
 ("Saya tertarik dengan penetration testing, firewall, dan keamanan jaringan.", "Cyber Security"),
 ("Saya terbiasa menggunakan Docker, Kubernetes, dan pipeline CI/CD di AWS.", "DevOps / Cloud Engineer"),
]
rows = []
for q, harapan in UJI:
    prof = cari(q, k=3, jenis="profil_profesi")
    nama = prof["judul"].str.replace("[Profil Profesi] ", "", regex=False).tolist()
    rows.append({"kueri": q[:60] + "...", "harapan": harapan, "top1": nama[0], "top3": ", ".join(nama),
                 "hit@1": int(nama[0] == harapan), "hit@3": int(harapan in nama)})
uji = pd.DataFrame(rows)
pd.set_option("display.max_colwidth", 120)
display(uji)
print(f"Hit@1: {uji['hit@1'].mean():.0%} | Hit@3: {uji['hit@3'].mean():.0%}  (uji kewajaran awal, bukan evaluasi final)")

# Cell 7 — Contoh pencarian bertingkat

In [ ]:
prof, bukti, mk = cari_bertingkat(UJI[0][0])
print("PROFESI TERATAS"); display(prof[["skor", "profesi"]])
for p, b in bukti.items():
    print(f"\nBukti standar untuk {p}:"); display(b[["skor", "jenis", "judul"]])
print("\nMATA KULIAH TERKAIT"); display(mk[["skor", "judul"]])

# Cell 8 — Log

In [ ]:
log = {"tanggal": datetime.datetime.now().isoformat(timespec="seconds"), "model": MODEL_NAME,
       "dimensi": int(emb.shape[1]), "jumlah_vektor": int(index.ntotal), "index": "FAISS IndexFlatIP (cosine)",
       "uji_kewajaran_hit@1": float(uji["hit@1"].mean()), "uji_kewajaran_hit@3": float(uji["hit@3"].mean())}
with open(LOG_DIR / "tahap6_log.json", "w") as fp:
    json.dump(log, fp, indent=2, ensure_ascii=False)
print(json.dumps(log, indent=2, ensure_ascii=False))

# Cell 9 — baseline skill dan peringkat hybrid

In [ ]:
import re
PROC_DIR = PROJECT_DIR / "02_processed"
if "emb" not in globals():                       # bila runtime sempat restart: cukup jalankan Cell 1-3 dan 5
    emb = np.load(KB_DIR / "kb_embeddings.npy")
kamus  = pd.read_csv(PROC_DIR / "kamus_skill.csv")
demand = pd.read_csv(PROC_DIR / "demand_skill_profesi.csv")
demand = demand[demand["kategori"] != "Soft skill"]
# matriks profesi x skill: proporsi lowongan profesi tsb yang meminta skill tsb
M = demand.pivot_table(index="profesi", columns="skill", values="persen", fill_value=0) / 100
POLA = [(r.skill, re.compile(r.pola, re.I)) for r in kamus.itertuples() if r.skill in M.columns]

def ekstrak_skill(teks):
    return [s for s, p in POLA if p.search(teks)]

def skor_skill(teks):
    """Baseline klasik (content-based): jumlah persen permintaan skill yang disebut pengguna, per profesi."""
    s = ekstrak_skill(teks)
    if not s:
        return pd.Series(0.0, index=M.index), s
    return M[s].sum(axis=1), s

prof_idx  = np.where(kb["jenis"] == "profil_profesi")[0]
prof_nama = kb.iloc[prof_idx]["judul"].str.replace("[Profil Profesi] ", "", regex=False).to_numpy()

def skor_dense(teks):
    return pd.Series(emb[prof_idx] @ embed_query(teks)[0], index=prof_nama)

def minmax(s):
    rng = s.max() - s.min()
    return (s - s.min()) / rng if rng > 0 else s * 0

def peringkat_profesi(teks, metode="hybrid", alpha=0.5):
    d = skor_dense(teks)
    k, s = skor_skill(teks)
    k = k.reindex(d.index, fill_value=0)
    if metode == "dense":
        skor = d
    elif metode == "skill":
        skor = k
    elif not s:                                  # hybrid tanpa skill terdeteksi -> murni semantik
        skor = d
    else:
        skor = alpha * minmax(d) + (1 - alpha) * minmax(k)
    return skor.sort_values(ascending=False), s
print("Skill di kamus:", len(POLA), "| profesi:", len(M), "| profil:", len(prof_idx))

# Cell 10 — perbandingan pada kueri eksplisit dan implisit

In [ ]:
UJI_IMPLISIT = [   # profil tanpa nama tools; label = dugaan awal, bukan ground truth final
    ("Saya senang membantu teman memperbaiki laptop yang rusak dan menginstal ulang sistem operasi.", "IT Support / Helpdesk"),
    ("Saya suka memikirkan bagaimana pengguna merasa nyaman saat memakai aplikasi.", "UI/UX Designer"),
    ("Saya senang mencari celah pada sistem dan ikut lomba CTF.", "Cyber Security"),
    ("Saya suka mengatur jadwal tim dan memastikan proyek selesai tepat waktu.", "IT Project/Product Manager"),
    ("Saya suka menganalisis kebutuhan pengguna dan membuat diagram UML.", "System Analyst"),
]
rows = []
for jenis_uji, data in [("eksplisit", UJI), ("implisit", UJI_IMPLISIT)]:
    for q, harapan in data:
        for metode in ["skill", "dense", "hybrid"]:
            r, s = peringkat_profesi(q, metode)
            top = list(r.index[:3]) if (metode != "skill" or s) else []
            rows.append({"jenis_uji": jenis_uji, "metode": metode, "kueri": q[:50], "harapan": harapan,
                         "skill_terdeteksi": ", ".join(s), "top3": ", ".join(top),
                         "hit@1": int(bool(top) and top[0] == harapan), "hit@3": int(harapan in top)})
banding = pd.DataFrame(rows)
display(banding.pivot_table(index="jenis_uji", columns="metode", values=["hit@1", "hit@3"], aggfunc="mean").round(2))
display(banding[banding["metode"] == "hybrid"][["jenis_uji", "harapan", "skill_terdeteksi", "top3", "hit@1"]])

# Cell 11 — simpan hasil ke 04_eval/ dan log

In [ ]:
(PROJECT_DIR / "04_eval").mkdir(exist_ok=True)
banding.to_csv(PROJECT_DIR / "04_eval" / "uji_kewajaran_metode.csv", index=False)
ring = banding.groupby(["jenis_uji", "metode"])[["hit@1", "hit@3"]].mean().round(2).reset_index()
log = json.load(open(LOG_DIR / "tahap6_log.json"))
log["perbandingan_metode"] = ring.to_dict(orient="records")
log["hybrid"] = "alpha*minmax(dense) + (1-alpha)*minmax(skill), alpha=0.5; tanpa skill terdeteksi -> dense"
with open(LOG_DIR / "tahap6_log.json", "w") as fp:
    json.dump(log, fp, indent=2, ensure_ascii=False)
print(ring.to_string(index=False))

---
# Tahap 7 — Evaluasi LLM awal
*Sumber: `07_llm_eval.ipynb`*

In [ ]:
   !pip install -q -U google-genai

# Cell 1 — setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!pip install -q google-genai sentence-transformers
import sys
from pathlib import Path
PROJECT_DIR = Path("/content/drive/MyDrive/Skripsi_RAG")
!mkdir -p /content/drive/MyDrive/Skripsi_RAG/src
sys.path.insert(0, str(PROJECT_DIR / "src"))

# Cell 2 — modul retrieval. Fungsi Tahap 6 dipindah ke file .py agar bisa dipakai ulang oleh evaluasi dan backend web nanti. cari_bertingkat di sini sudah memakai peringkat hybrid.

In [ ]:
%%writefile /content/drive/MyDrive/Skripsi_RAG/src/retrieval.py
"""Modul retrieval KB karier (Tahap 6): dipakai notebook evaluasi dan nanti backend web."""
import re
from pathlib import Path
import numpy as np
import pandas as pd

MODEL_NAME = "intfloat/multilingual-e5-base"
S = {}   # state modul: kb, emb, model, matriks skill

def init(project_dir, model=None):
    p = Path(project_dir)
    kb = pd.read_json(p / "03_kb" / "kb_chunks.jsonl", lines=True)
    kb["profesi"] = kb["profesi"].apply(lambda x: x if isinstance(x, list) else [])
    S["kb"], S["emb"] = kb, np.load(p / "03_kb" / "kb_embeddings.npy")
    assert len(kb) == len(S["emb"]), "kb_chunks dan kb_embeddings tidak sinkron"
    if model is None:
        from sentence_transformers import SentenceTransformer
        model = SentenceTransformer(MODEL_NAME)
    S["model"] = model
    kamus  = pd.read_csv(p / "02_processed" / "kamus_skill.csv")
    demand = pd.read_csv(p / "02_processed" / "demand_skill_profesi.csv")
    demand = demand[demand["kategori"] != "Soft skill"]
    S["demand"] = demand
    S["M"] = demand.pivot_table(index="profesi", columns="skill", values="persen", fill_value=0) / 100
    S["pola"] = [(r.skill, re.compile(r.pola, re.I)) for r in kamus.itertuples() if r.skill in S["M"].columns]
    S["prof_idx"] = np.where(kb["jenis"] == "profil_profesi")[0]
    S["prof_nama"] = kb.iloc[S["prof_idx"]]["judul"].str.replace("[Profil Profesi] ", "", regex=False).to_numpy()
    return kb

def embed_query(q):
    return S["model"].encode(["query: " + q], normalize_embeddings=True, convert_to_numpy=True).astype("float32")[0]

def ekstrak_skill(teks):
    return [s for s, p in S["pola"] if p.search(teks)]

def skor_skill(teks):
    s = ekstrak_skill(teks)
    if not s:
        return pd.Series(0.0, index=S["M"].index), s
    return S["M"][s].sum(axis=1), s

def _minmax(x):
    r = x.max() - x.min()
    return (x - x.min()) / r if r > 0 else x * 0

def peringkat_profesi(teks, metode="hybrid", alpha=0.5, q=None):
    q = embed_query(teks) if q is None else q
    d = pd.Series(S["emb"][S["prof_idx"]] @ q, index=S["prof_nama"])
    k, s = skor_skill(teks)
    k = k.reindex(d.index, fill_value=0)
    if metode == "dense" or (metode == "hybrid" and not s):
        skor = d
    elif metode == "skill":
        skor = k
    else:
        skor = alpha * _minmax(d) + (1 - alpha) * _minmax(k)
    return skor.sort_values(ascending=False), s

def cari(teks, k=5, jenis=None, profesi=None, q=None):
    kb, emb = S["kb"], S["emb"]
    q = embed_query(teks) if q is None else q
    mask = np.ones(len(kb), dtype=bool)
    if jenis is not None:
        mask &= kb["jenis"].isin([jenis] if isinstance(jenis, str) else jenis).to_numpy()
    if profesi is not None:
        mask &= kb["profesi"].apply(lambda ps: profesi in ps).to_numpy()
    idx = np.where(mask)[0]
    if len(idx) == 0:
        return pd.DataFrame(columns=["skor", "chunk_id", "jenis", "judul", "sumber", "teks"])
    skor = emb[idx] @ q
    urut = np.argsort(-skor)
    hasil = kb.iloc[idx[urut]][["chunk_id", "jenis", "judul", "sumber", "teks"]].copy()
    hasil.insert(0, "skor", skor[urut].round(4))
    return hasil.drop_duplicates("judul").head(k).reset_index(drop=True)

def cari_bertingkat(teks, n_profesi=3, k_standar=3, k_mk=4):
    """1) peringkat profesi (hybrid), 2) bukti SKKNI/okupasi per profesi, 3) mata kuliah terkait."""
    q = embed_query(teks)
    rank, skill = peringkat_profesi(teks, "hybrid", q=q)
    top = list(rank.index[:n_profesi])
    kb = S["kb"]
    profil = {p: kb.iloc[S["prof_idx"][list(S["prof_nama"]).index(p)]]["teks"] for p in top}
    bukti = {p: cari(teks, k_standar, ["skkni", "okupasi"], p, q=q) for p in top}
    mk = cari(teks, k_mk, "mata_kuliah", q=q)
    return {"peringkat": rank, "skill_terdeteksi": skill, "profesi": top, "profil": profil, "bukti": bukti, "mk": mk}

# Cell 3a

In [ ]:
import os
from google.colab import userdata
k = userdata.get("GEMINI_API_KEY")
print("panjang:", len(k), "| awalan:", k[:4], "| akhiran:", k[-4:], "| ada spasi/enter:", k != k.strip())
print("env lain yang bisa bentrok:", [v for v in ["GOOGLE_API_KEY", "GEMINI_API_KEY", "GOOGLE_GENAI_USE_VERTEXAI"] if v in os.environ])

# Cell 3 — muat retrieval dan klien Gemini

In [ ]:
import importlib, time
import retrieval as R
importlib.reload(R)
kb = R.init(PROJECT_DIR)
print("Chunk:", len(kb), "| tes:", list(R.peringkat_profesi("Saya membuat aplikasi Android dengan Kotlin")[0].index[:3]))

from google import genai
from google.genai import types
from google.colab import userdata
client = genai.Client(api_key=userdata.get("GEMINI_API_KEY"))
MODEL_LLM = "gemini-3.8-flash"      # jika error 404, ganti dengan nama model Flash yang tercantum di AI Studio

def panggil_llm(prompt, coba=4):
    for i in range(coba):
        try:
            r = client.models.generate_content(
                model=MODEL_LLM, contents=prompt,
                config=types.GenerateContentConfig(temperature=0, response_mime_type="application/json"))
            return r.text
        except Exception as e:
            if any(c in str(e) for c in ["429", "500", "503"]):   # kuota/server sibuk -> tunggu lalu ulangi
                time.sleep(20 * (i + 1)); continue
            raise
    return None
print(panggil_llm('Balas dengan JSON {"ok": true}'))

# Cell 4 — data acuan untuk pengecekan grounding

In [ ]:
import json, re, datetime
import pandas as pd, numpy as np
EVAL_DIR = PROJECT_DIR / "04_eval"; EVAL_DIR.mkdir(exist_ok=True)
PROFESI = list(R.S["prof_nama"])
mkdf = pd.read_csv(PROJECT_DIR / "01_interim" / "manual" / "F031_mata_kuliah_kurikulum2021.csv")
def norm(t):
    return re.sub(r"[^a-z0-9]+", " ", str(t).lower().replace("&", " dan ")).strip()
MK_VALID = {norm(n): n for n in mkdf["nama_mata_kuliah"]}
std = pd.read_csv(PROJECT_DIR / "02_processed" / "profesi_standar.csv")
STD_PROFESI = std.groupby("profesi")["isi_nomor_tahun"].apply(set).to_dict()
STD_SEMUA = set(std["isi_nomor_tahun"])
DEMAND = R.S["demand"].set_index(["profesi", "skill"])["persen"].to_dict()
print("Profesi:", len(PROFESI), "| MK kurikulum:", len(MK_VALID), "| nomor SKKNI:", len(STD_SEMUA))

# Cell 5 — prompt. Kedua kondisi LLM memakai instruksi, daftar profesi, dan format JSON yang sama. Perbedaannya hanya pada ada atau tidaknya KONTEKS.

In [ ]:
INSTRUKSI = """Anda adalah asisten rekomendasi karier untuk mahasiswa D4 Teknik Informatika Politeknik Negeri Jakarta (PNJ).
Berdasarkan profil mahasiswa, pilih TEPAT 3 profesi yang paling sesuai, urut dari yang paling sesuai, HANYA dari daftar berikut:
{daftar}

Untuk setiap profesi, isi:
- "profesi": nama persis dari daftar
- "alasan": 2-3 kalimat yang mengaitkan profil mahasiswa dengan profesi
- "skill_dimiliki": skill relevan yang tampak dari profil
- "skill_perlu_dikembangkan": maksimal 5 skill yang perlu dipelajari
- "persen_permintaan": objek {{nama_skill: persen lowongan yang meminta skill tsb}} untuk skill_perlu_dikembangkan
- "mata_kuliah_pendukung": nama mata kuliah kurikulum D4 TI PNJ yang mendukung
- "standar_acuan": nomor/tahun SKKNI yang relevan, contoh "282/2016"
{aturan}
Jawab HANYA dalam JSON: {{"rekomendasi": [ {{...}}, {{...}}, {{...}} ]}}

PROFIL MAHASISWA:
{profil}
"""
ATURAN_RAG = """
ATURAN: angka persen, nama mata kuliah, dan nomor SKKNI WAJIB diambil dari KONTEKS di bawah. Jika tidak ada di KONTEKS,
kosongkan isian tersebut. Jangan mengarang.

KONTEKS:
{konteks}
"""
ATURAN_NON = """
ATURAN: gunakan pengetahuan Anda. Jika tidak yakin pada suatu angka, nama mata kuliah, atau nomor SKKNI, kosongkan isian tersebut.
"""

def susun_konteks(profil, n_profesi=5):
    h = R.cari_bertingkat(profil, n_profesi=n_profesi)
    bag = [f"Skill terdeteksi dari profil: {', '.join(h['skill_terdeteksi']) or '-'}",
           "Kandidat profesi (urut skor retrieval): " + ", ".join(h["profesi"])]
    for p in h["profesi"]:
        bag.append("\n" + h["profil"][p])
        for _, b in h["bukti"][p].iterrows():
            bag.append(f"  - Bukti standar: {b['judul']}")
    bag.append("\nMata kuliah PNJ terkait profil:")
    for _, m in h["mk"].iterrows():
        bag.append(f"  - {m['teks'][:350]}")
    return "\n".join(bag), h

def buat_prompt(profil, kondisi):
    daftar = "\n".join(f"- {p}" for p in PROFESI)
    if kondisi == "llm_rag":
        konteks, h = susun_konteks(profil)
        return INSTRUKSI.format(daftar=daftar, aturan=ATURAN_RAG.format(konteks=konteks), profil=profil), h
    return INSTRUKSI.format(daftar=daftar, aturan=ATURAN_NON, profil=profil), None

# Cell 6 — parsing, pengecekan grounding, dan metrik peringkat

In [ ]:
def parse_json(teks):
    m = re.search(r"\{.*\}", teks or "", re.S)
    try:
        return json.loads(m.group(0))["rekomendasi"]
    except Exception:
        return None

def cek_grounding(rek):
    """Hitung klaim valid / tidak valid per jenis. Isian kosong = abstain (bukan halusinasi)."""
    c = {k: 0 for k in ["mk_klaim", "mk_valid", "std_klaim", "std_valid", "std_terkait",
                        "persen_klaim", "persen_valid", "profesi_tidak_valid"]}
    for r in rek or []:
        p = r.get("profesi", "")
        if p not in PROFESI:
            c["profesi_tidak_valid"] += 1
        for mk in r.get("mata_kuliah_pendukung") or []:
            c["mk_klaim"] += 1; c["mk_valid"] += norm(mk) in MK_VALID
        for s in r.get("standar_acuan") or []:
            for nt in re.findall(r"\d{1,3}\s*/\s*\d{4}", str(s)) or [s]:
                nt = re.sub(r"\s", "", str(nt))
                c["std_klaim"] += 1; c["std_valid"] += nt in STD_SEMUA
                c["std_terkait"] += nt in STD_PROFESI.get(p, set())
        for sk, v in (r.get("persen_permintaan") or {}).items():
            try:
                v = float(str(v).replace("%", "").replace(",", "."))
            except ValueError:
                continue
            c["persen_klaim"] += 1
            ref = DEMAND.get((p, sk))
            c["persen_valid"] += ref is not None and abs(ref - v) <= 1.0
    return c

def metrik_peringkat(top, label):
    rel = [i for i, p in enumerate(top[:3]) if p in label]
    return {"hit@1": int(bool(top) and top[0] in label), "hit@3": int(bool(rel)),
            "mrr": 1 / (rel[0] + 1) if rel else 0.0}

# Cell 7 — uji coba dua profil × dua kondisi LLM

In [ ]:
uji = [("Saya senang membantu teman memperbaiki laptop dan menginstal ulang sistem operasi.", "IT Support / Helpdesk"),
       ("Saya suka mengolah data dengan SQL dan Excel lalu membuat dashboard Power BI.", "Data Analyst / BI")]
for profil, harapan in uji:
    for kondisi in ["llm_non_rag", "llm_rag"]:
        prompt, _ = buat_prompt(profil, kondisi)
        rek = parse_json(panggil_llm(prompt))
        top = [r.get("profesi") for r in rek or []]
        print(f"\n[{kondisi}] harapan={harapan}\n  top3 = {top}\n  grounding = {cek_grounding(rek)}")
        if rek:
            print("  contoh:", json.dumps(rek[0], ensure_ascii=False)[:600])
    time.sleep(6)

# Cell 8 — template set uji

In [ ]:
TEMPLATE = EVAL_DIR / "profil_uji.csv"
if not TEMPLATE.exists():
    pd.DataFrame(columns=["id", "jenis", "profil", "label", "penilai_1", "penilai_2", "catatan"]).to_csv(TEMPLATE, index=False)
    print("Template dibuat:", TEMPLATE)
else:
    print("Template sudah ada:", TEMPLATE)

# Cell 9 — fungsi evaluasi. Cell ini cukup dijalankan supaya fungsinya terdefinisi. Evaluasi penuh baru dijalankan setelah set uji terisi. Hasil disimpan di cache, jadi jika terputus karena kuota, cukup jalankan ulang dan bagian yang sudah selesai akan dilewati.

In [ ]:
def jalankan_evaluasi(path=TEMPLATE, jeda=6):
    df = pd.read_csv(path).dropna(subset=["profil", "label"])
    cache_f = EVAL_DIR / "hasil_mentah.jsonl"
    cache = {}
    if cache_f.exists():
        for l in open(cache_f, encoding="utf-8"):
            d = json.loads(l); cache[(d["id"], d["kondisi"])] = d
    rows = []
    for _, u in df.iterrows():
        label = [x.strip() for x in str(u["label"]).split(";")]
        for kondisi in ["baseline_skill", "retrieval_hybrid", "llm_non_rag", "llm_rag"]:
            key = (str(u["id"]), kondisi)
            if key not in cache:
                if kondisi in ("baseline_skill", "retrieval_hybrid"):      # tanpa LLM
                    metode = "skill" if kondisi == "baseline_skill" else "hybrid"
                    r, s = R.peringkat_profesi(u["profil"], metode)
                    d = {"top": list(r.index[:3]) if (s or metode == "hybrid") else [], "rek": None, "mentah": None}
                else:
                    prompt, _ = buat_prompt(u["profil"], kondisi)
                    mentah = panggil_llm(prompt); rek = parse_json(mentah)
                    d = {"top": [x.get("profesi") for x in rek or []], "rek": rek, "mentah": mentah}
                    time.sleep(jeda)                         # batas laju API gratis
                d.update(id=key[0], kondisi=kondisi)
                with open(cache_f, "a", encoding="utf-8") as fp:
                    fp.write(json.dumps(d, ensure_ascii=False) + "\n")
                cache[key] = d
            d = cache[key]
            rows.append({"id": key[0], "jenis": u["jenis"], "kondisi": kondisi, "top3": "; ".join(d["top"]),
                         "json_valid": int(not kondisi.startswith("llm") or d["rek"] is not None),
                         **metrik_peringkat(d["top"], label), **cek_grounding(d["rek"])})
    return pd.DataFrame(rows)

def ringkas(hasil):
    g = hasil.groupby("kondisi")
    t = g[["hit@1", "hit@3", "mrr", "json_valid"]].mean().round(3)
    s = g[["mk_klaim", "mk_valid", "std_klaim", "std_valid", "std_terkait", "persen_klaim", "persen_valid"]].sum()
    for a in ["mk", "std", "persen"]:
        t[f"{a}_valid_%"] = (100 * s[f"{a}_valid"] / s[f"{a}_klaim"].replace(0, np.nan)).round(1)
    t["std_terkait_%"] = (100 * s["std_terkait"] / s["std_klaim"].replace(0, np.nan)).round(1)
    t["n_klaim_mk/std/persen"] = s["mk_klaim"].astype(str) + "/" + s["std_klaim"].astype(str) + "/" + s["persen_klaim"].astype(str)
    return t
print("Fungsi evaluasi siap. Isi profil_uji.csv dulu sebelum menjalankan jalankan_evaluasi().")

---
# Tahap 8 — Pemisahan demand (v4)
*Sumber: `08_split_demand_v4.ipynb`*

# Tahap 8 — Pemisahan Senior Programmer/Software Developer, hold-out 70/30, demand ulang
Sesuai Bab III subbab 3.2.2 dan Persamaan 3.1–3.2. Notebook ini tidak melatih model:
70% iklan dipakai untuk menghitung profil demand, 30% disisihkan sebagai data uji ranking.

# Cell 1 — Lokasi proyek (Colab atau lokal)

In [ ]:
import os, re, json, math, datetime
from pathlib import Path
import pandas as pd

try:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_DIR = Path("/content/drive/MyDrive/Skripsi_RAG")
except ImportError:
    PROJECT_DIR = Path(os.environ.get("SKRIPSI_DIR", "."))
OUT_DIR, LOG_DIR = PROJECT_DIR / "02_processed", PROJECT_DIR / "logs"
SEED, TEST_FRAC, AMBANG = 42, 0.30, 10.0      # seed tetap, 30% uji, skill diminta jika >= 10% iklan
print("Proyek:", PROJECT_DIR)

# Cell 2 — Data

In [ ]:
ti = pd.read_csv(OUT_DIR / "lowongan_ti.csv", low_memory=False)
ti["teks_lengkap"] = ti["teks_lengkap"].astype(str).eq("True")
kamus = pd.read_csv(OUT_DIR / "kamus_skill.csv", keep_default_na=False)        # kamus v2, 109 skill
# Kamus v3: pola "Hardware Troubleshooting" v2 juga menangkap troubleshooting database/UX/software,
# jadi dipersempit ke konteks perangkat keras. Kamus v2 tetap disimpan apa adanya.
UBAH_V3 = {"Hardware Troubleshooting":
           r"hardware|perangkat keras|troubleshoot\w*\s+(?:\w+\s+){0,3}(?:pc|computer|komputer|printer|laptop|perangkat|devices?)\b|perbaikan perangkat|instalasi perangkat|rakit (pc|komputer)"}
for k_, v_ in UBAH_V3.items():
    kamus.loc[kamus["skill"] == k_, "pola"] = v_
kamus.to_csv(OUT_DIR / "kamus_skill_v3.csv", index=False)
POLA_SKILL = [(r["skill"], r["kategori"], re.compile(r["pola"], re.I)) for _, r in kamus.iterrows()]
SOFT = set(kamus.loc[kamus["kategori"] == "Soft skill", "skill"])
print("Iklan TI:", f"{len(ti):,}", "| berdeskripsi lengkap:", f"{ti['teks_lengkap'].sum():,}", "| skill:", len(kamus))

# Cell 3 — Pisahkan Senior Programmer dan Software Developer

In [ ]:
SENIOR = re.compile(r"\bsenior\b|\bsr\b\.?|\blead\b|\bprincipal\b", re.I)
sw = ti["profesi"].eq("Software / Web Developer")
ti["karier"] = ti["profesi"]
ti.loc[sw & ti["judul"].astype(str).str.contains(SENIOR), "karier"] = "Senior Programmer"
ti.loc[sw & ~ti["judul"].astype(str).str.contains(SENIOR), "karier"] = "Software Developer"
# Revisi setelah validasi label (v2): manajer TI umum dan UX writer bukan salah satu dari 15 karier,
# jadi dikeluarkan (bukan dipindah). Judul manajer yang memuat project/product/scrum tetap dipertahankan.
MGR_UMUM = re.compile(r"\bit manager\b|\bmanager it\b|\bhead (of )?it\b|\bit head\b|kepala (bagian |divisi )?it\b|\bhead of it division\b", re.I)
PROYEK = re.compile(r"project|product|scrum|program manager|\bpmo\b", re.I)
UX_WRITER = re.compile(r"\b(ux )?(copy)?writer\b", re.I)
jdl = ti["judul"].astype(str)
keluar = ((ti["karier"] == "IT Project/Product Manager") & jdl.str.contains(MGR_UMUM) & ~jdl.str.contains(PROYEK)) | \
         ((ti["karier"] == "UI/UX Designer") & jdl.str.contains(UX_WRITER))
DIKELUARKAN = ti.loc[keluar & ti["teks_lengkap"]].groupby("karier").size().to_dict()
print("Dikeluarkan (berteks lengkap):", DIKELUARKAN)
ti = ti[~keluar].copy()
print(ti.loc[ti["teks_lengkap"], "karier"].value_counts().to_string())

# Cell 4 — Katalog 15 karier: profil lulusan, area minat (SKKNI + peta okupasi), kode O*NET

In [ ]:
KATALOG = [
 # karier, profil_lulusan, area minat (dipisah ';'), kode O*NET-SOC
 ("Senior Programmer", 1, "Software development", "15-1251.00"),
 ("Software Developer", 1, "Software development", "15-1252.00"),
 ("System Analyst", 1, "Systems analysis", "15-1211.00"),
 ("Database Administrator", 1, "Databases and data management", "15-1242.00"),
 ("Mobile Developer", 0, "Software development", "15-1252.00"),
 ("QA / Software Tester", 0, "Quality assurance", "15-1253.00"),
 ("UI/UX Designer", 0, "UI/UX design", "15-1255.00"),
 ("Data Analyst / BI", 0, "Data science and AI", "15-2051.01"),
 ("Data Engineer", 0, "Databases and data management;Data science and AI", "15-1243.01"),
 ("Data Scientist / ML / AI Engineer", 0, "Data science and AI", "15-2051.00"),
 ("DevOps / Cloud Engineer", 0, "Infrastructure and cloud", "15-1299.08"),
 ("Network / System Administrator", 0, "Infrastructure and cloud", "15-1244.00"),
 ("Cyber Security", 0, "Security", "15-1212.00"),
 ("IT Support / Helpdesk", 0, "IT service and management", "15-1232.00"),
 ("IT Project/Product Manager", 0, "IT service and management", "15-1299.09"),
]
katalog = pd.DataFrame(KATALOG, columns=["karier", "profil_lulusan", "area_minat", "onet_soc"])
assert set(katalog["karier"]) == set(ti.loc[ti["teks_lengkap"], "karier"]), "nama karier tidak cocok"
katalog.to_csv(OUT_DIR / "katalog_karier.csv", index=False)
katalog

# Cell 5 — Ekstraksi skill per iklan (biner)

In [ ]:
teks = ti[ti["teks_lengkap"]].copy()
teks["teks_norm"] = teks["teks"].astype(str).str.replace(r"([a-z])([A-Z])", r"\1 \2", regex=True)
for skill, _, pola in POLA_SKILL:
    teks[skill] = teks["teks_norm"].str.contains(pola)
SKILLS = [s for s, _, _ in POLA_SKILL]
teks["n_skill_teknis"] = teks[[s for s in SKILLS if s not in SOFT]].sum(axis=1)

# Cell 6 — Hold-out 70/30 bertingkat per karier (seed tetap)

In [ ]:
teks["iid"] = teks["sumber"].astype(str) + "|" + teks["id_lowongan"].astype(str)
uji_idx = teks.groupby("karier", group_keys=False).sample(frac=TEST_FRAC, random_state=SEED).index
teks["split"] = "bangun"
teks.loc[uji_idx, "split"] = "uji"
split_tab = pd.crosstab(teks["karier"], teks["split"]).assign(total=lambda d: d.sum(axis=1))
display(split_tab) if "display" in globals() else print(split_tab)

# Cell 7 — Demand d(c,s) dari 70% iklan (Persamaan 3.1)

In [ ]:
bangun = teks[teks["split"] == "bangun"]
N = bangun.groupby("karier").size()
dem = (bangun.groupby("karier")[SKILLS].sum().stack().rename("n_lowongan").reset_index()
       .rename(columns={"level_1": "skill"}))
dem = dem[dem["n_lowongan"] > 0].merge(N.rename("n_total"), left_on="karier", right_index=True)
dem["persen"] = (dem["n_lowongan"] / dem["n_total"] * 100).round(1)
dem = dem.merge(kamus[["skill", "kategori"]], on="skill").sort_values(["karier", "persen"], ascending=[True, False])
dem.to_csv(OUT_DIR / "demand_skill_karier_v2.csv", index=False)

# Cell 8 — Himpunan syarat R(c) dan bobot w(c,s) (Persamaan 3.2)

In [ ]:
R = dem[(dem["persen"] >= AMBANG) & (~dem["skill"].isin(SOFT))].copy()
C = katalog["karier"].nunique()
m_s = R.groupby("skill")["karier"].nunique().rename("m_s")
R = R.merge(m_s, left_on="skill", right_index=True)
R["d"] = R["n_lowongan"] / R["n_total"]
R["idf"] = (1 + C / R["m_s"]).map(math.log)
R["w"] = (R["d"] * R["idf"]).round(4)
R.to_csv(OUT_DIR / "bobot_skill_karier.csv", index=False)
print(R.groupby("karier").size().rename("jumlah_skill_syarat").to_string())

# Cell 9 — Cakupan kurikulum per karier (bobot demand)

In [ ]:
mks = pd.read_csv(OUT_DIR / "mk_skill_terverifikasi.csv", dtype=str)
mks = mks[mks["valid"].str.replace(".0", "", regex=False) == "1"]
diajar = set(mks["skill"])
cak = (R.assign(diajar=R["skill"].isin(diajar))
         .groupby("karier").apply(lambda g: pd.Series({
             "skill_syarat": len(g),
             "cakupan_%": round(100 * g.loc[g["diajar"], "d"].sum() / g["d"].sum(), 1),
             "gap_skill": ", ".join(g.loc[~g["diajar"]].sort_values("d", ascending=False)["skill"])}))
         .reset_index().merge(katalog[["karier", "profil_lulusan"]], on="karier")
         .sort_values(["profil_lulusan", "cakupan_%"], ascending=[False, False]))
cak.to_csv(OUT_DIR / "cakupan_kurikulum_karier_v2.csv", index=False)
cak

# Cell 10 — Simpan data uji dan sampel validasi

In [ ]:
uji = teks[teks["split"] == "uji"]
kol = ["iid", "sumber", "id_lowongan", "judul", "karier", "n_skill_teknis"]
uji_out = uji[kol].copy()
uji_out["skill_teknis"] = uji[[s for s in SKILLS if s not in SOFT]].apply(lambda r: ";".join(r.index[r.values]), axis=1)
uji_out["layak_uji"] = uji_out["n_skill_teknis"] >= 3          # profil uji minimal 3 skill dikenali
uji_out.to_csv(OUT_DIR / "holdout_uji.csv", index=False)
teks[["iid", "karier", "split"]].to_csv(OUT_DIR / "split_iklan.csv", index=False)

# Sampel validasi sudah dilabeli manual (validasi_klasifikasi_sampel_v2_terlabel.csv); tidak dibuat ulang.
# Akurasi dihitung sebelum dan sesudah aturan pengecualian (baris yang kini dikeluarkan tidak dihitung).
lab = pd.read_csv(OUT_DIR / "validasi_klasifikasi_sampel_v2_terlabel.csv")
lab = lab.rename(columns={"label": "benar_1_salah_0"})
jl = lab["judul"].astype(str)
lab["dikeluarkan"] = ((lab["karier"] == "IT Project/Product Manager") & jl.str.contains(MGR_UMUM) & ~jl.str.contains(PROYEK)) | \
                     ((lab["karier"] == "UI/UX Designer") & jl.str.contains(UX_WRITER))
def akurasi(df):
    g = df.groupby("karier")["benar_1_salah_0"].agg(n="size", benar="sum")
    g["akurasi_%"] = (100 * g["benar"] / g["n"]).round(1)
    return g
akur_awal, akur_akhir = akurasi(lab), akurasi(lab[~lab["dikeluarkan"]])
validasi = {"awal_%": round(100 * lab["benar_1_salah_0"].mean(), 1),
            "akhir_%": round(100 * lab.loc[~lab["dikeluarkan"], "benar_1_salah_0"].mean(), 1),
            "n_akhir": int((~lab["dikeluarkan"]).sum()),
            "karier_di_bawah_80%": akur_akhir.index[akur_akhir["akurasi_%"] < 80].tolist()}
print(validasi)
akur_awal.join(akur_akhir, rsuffix="_akhir").to_csv(OUT_DIR / "validasi_klasifikasi_akurasi.csv")
akur_awal.join(akur_akhir, rsuffix="_akhir")

# Cell 11 — Log

In [ ]:
log = {"tanggal": datetime.datetime.now().isoformat(timespec="seconds"), "seed": SEED, "test_frac": TEST_FRAC,
       "ambang_%": AMBANG, "iklan_berteks_lengkap": int(len(teks)),
       "iklan_bangun": int((teks["split"] == "bangun").sum()), "iklan_uji": int(len(uji)),
       "iklan_uji_layak(>=3 skill)": int(uji_out["layak_uji"].sum()),
       "dikeluarkan_aturan_judul": DIKELUARKAN, "validasi_klasifikasi": validasi,
       "per_karier": split_tab.to_dict(orient="index"),
       "cakupan_profil_lulusan": cak[cak["profil_lulusan"] == 1].set_index("karier")["cakupan_%"].to_dict()}
with open(LOG_DIR / "tahap8_log.json", "w") as fp:
    json.dump(log, fp, indent=2, ensure_ascii=False)
print(json.dumps({k: v for k, v in log.items() if k != "per_karier"}, indent=2, ensure_ascii=False))

---
# Tahap 9 — Triangulasi sumber
*Sumber: `09_triangulasi.ipynb`*

# Tahap 9 — Triangulasi himpunan syarat R(c) dengan SKKNI/peta okupasi dan O*NET 31.0
Bab III subbab 3.2.2: persentase skill syarat tiap karier yang juga muncul di standar nasional
dan/atau di Software Skills O*NET okupasi padanannya (Appendix B). Skill yang tidak ada di
referensi mana pun ditinjau manual.
Siapkan dulu: folder 00_raw/onet_31_0/ berisi "Software Skills.xlsx" dan "Occupation Data.xlsx".

# Cell 1 — Lokasi proyek

In [ ]:
import os, re, json, datetime
from pathlib import Path
import pandas as pd

try:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_DIR = Path("/content/drive/MyDrive/Skripsi_RAG")
except ImportError:
    PROJECT_DIR = Path(os.environ.get("SKRIPSI_DIR", "."))
OUT_DIR, LOG_DIR = PROJECT_DIR / "02_processed", PROJECT_DIR / "logs"
ONET_DIR = PROJECT_DIR / "00_raw" / "onet_31_0"

# Cell 2 — Data

In [ ]:
R = pd.read_csv(OUT_DIR / "bobot_skill_karier.csv")
kat = pd.read_csv(OUT_DIR / "katalog_karier.csv")
kamus = pd.read_csv(OUT_DIR / "kamus_skill_v3.csv", keep_default_na=False)
POLA = {r["skill"]: re.compile(r["pola"], re.I) for _, r in kamus.iterrows()}
manifest = pd.read_csv(LOG_DIR / "manifest.csv")
docs = pd.read_json(PROJECT_DIR / "01_interim" / "extracted_docs.jsonl", lines=True)
teks_file = docs.groupby("file_id")["text"].apply(lambda s: "\n".join(s))
onet = pd.read_excel(ONET_DIR / "Software Skills.xlsx")

# Cell 3 — Dokumen standar per karier (sama dengan Tahap 4, ditambah okupasi Senior Programmer)

In [ ]:
STANDAR = {
 "Senior Programmer": ["2016-282", "2019-022", "TIK.DEV0701"],     # file TIK.DEV0701 berisi okupasi 020801 Senior Programmer (KKNI 7)
 "Software Developer": ["2016-282", "2019-022", "TIK.DEV0615"],
 "Mobile Developer": ["2015-458"],
 "System Analyst": ["2017-044", "2015-090", "TIK.DEV0613"],
 "Database Administrator": ["2020-268", "TIK.DSC0604"],
 "QA / Software Tester": ["2017-047"],
 "UI/UX Designer": ["2018-107", "2023-126", "TIK.SRV0506"],
 "Network / System Administrator": ["2016-321", "2017-082", "TIK.INF0402", "TIK.INF0403", "TIK.INF0502", "TIK.INF0504",
                                    "TIK.INF0506", "TIK.INF0602", "TIK.INF0605", "TIK.INF0609",
                                    "TIK.INF0604-DESAINER JARINGAN (NETWORK DESIGNER).pdf"],
 "IT Support / Helpdesk": ["2016-285", "kmnaker030"],
 "Cyber Security": ["kmnaker191", "kmnaker120", "2022-023", "Web & Mobile Development 2020-391", "TIK.INF0611", "TIK.SEC0603"],
 "DevOps / Cloud Engineer": ["2023-102"],
 "Data Analyst / BI": ["SKKNI Data & Database Management 2020-299"],
 "Data Engineer": ["2020-268", "SKKNI Data & Database Management 2020-299"],
 "Data Scientist / ML / AI Engineer": ["SKKNI Data & Database Management 2020-299", "kmnaker103"],
 "IT Project/Product Manager": ["kmnaker200"],
}
def teks_standar(karier):
    fids = set()
    for k in STANDAR[karier]:
        hit = manifest[manifest["file_name"].str.contains(k, regex=False) & manifest["kelompok"].isin(["skkni", "okupasi"])
                       & (manifest["status"] != "salinan")]["file_id"]
        assert len(hit), f"Tidak ketemu: {k}"
        fids |= set(hit)
    return "\n".join(teks_file.get(f, "") for f in sorted(fids))

# Cell 4 — Teks O*NET per okupasi (contoh software + kategori)

In [ ]:
onet["teks"] = onet["Workplace Example"].astype(str) + " | " + onet["Element Name"].astype(str)
teks_onet = onet.groupby("O*NET-SOC Code")["teks"].apply(lambda s: "\n".join(s))
hot = onet[onet["Hot Technology"].eq("Y")].groupby("O*NET-SOC Code")["Workplace Example"].apply(lambda s: "\n".join(s))

# Cell 5 — Cek tiap skill syarat

In [ ]:
rows = []
for c, g in R.groupby("karier"):
    t_std = teks_standar(c)
    soc = kat.set_index("karier").loc[c, "onet_soc"]
    t_on, t_hot = teks_onet.get(soc, ""), hot.get(soc, "")
    for _, r in g.iterrows():
        p = POLA[r["skill"]]
        rows.append({"karier": c, "skill": r["skill"], "persen_iklan": r["persen"],
                     "di_standar_nasional": bool(p.search(t_std)), "di_onet": bool(p.search(t_on)),
                     "onet_hot_technology": bool(p.search(t_hot))})
tri = pd.DataFrame(rows)
tri["di_salah_satu"] = tri["di_standar_nasional"] | tri["di_onet"]
tri.to_csv(OUT_DIR / "triangulasi_skill_karier.csv", index=False)

# Cell 6 — Ringkasan per karier

In [ ]:
rk = tri.groupby("karier").agg(skill_syarat=("skill", "size"),
        standar_nasional_pct=("di_standar_nasional", "mean"), onet_pct=("di_onet", "mean"),
        salah_satu_pct=("di_salah_satu", "mean"), onet_hot_pct=("onet_hot_technology", "mean"))
for col in [c for c in rk.columns if c.endswith("_pct")]:
    rk[col] = (rk[col] * 100).round(1)
rk["tanpa_referensi"] = tri[~tri["di_salah_satu"]].groupby("karier")["skill"].apply(", ".join)
rk = rk.fillna({"tanpa_referensi": ""}).merge(kat.set_index("karier")[["profil_lulusan"]], left_index=True, right_index=True)
rk = rk.sort_values(["profil_lulusan", "salah_satu_pct"], ascending=[False, False])
rk.to_csv(OUT_DIR / "triangulasi_ringkasan.csv")
total = {"skill_syarat": int(len(tri)), "salah_satu_%": round(100 * tri["di_salah_satu"].mean(), 1),
         "standar_nasional_%": round(100 * tri["di_standar_nasional"].mean(), 1), "onet_%": round(100 * tri["di_onet"].mean(), 1)}
print(total)
rk

# Cell 7 — Log

In [ ]:
with open(LOG_DIR / "tahap9_log.json", "w") as fp:
    json.dump({"tanggal": datetime.datetime.now().isoformat(timespec="seconds"), "total": total,
               "per_karier": rk.drop(columns="profil_lulusan").to_dict(orient="index")}, fp, indent=2, ensure_ascii=False)

---
# Tahap 10 — Evaluasi ranking (hold-out)
*Sumber: `10_ranking_holdout.ipynb`*

# Tahap 10 — Uji ranking pada iklan hold-out (RQ4)
Bab III subbab 3.6.2: profil uji = skill teknis dari iklan uji (30%); ranking 15 karier dengan
kecocokan kompetensi K(c) (Persamaan 3.4, e(s)=1 untuk skill di iklan) vs baseline cosine biner.
Metrik: Hit@1, Hit@3, MRR (makro per karier dan mikro); uji McNemar dan Wilcoxon; CI bootstrap.

# Cell 1 — Lokasi proyek

In [ ]:
import os, json, datetime
from pathlib import Path
import numpy as np, pandas as pd
from scipy.stats import binomtest, wilcoxon

try:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_DIR = Path("/content/drive/MyDrive/Skripsi_RAG")
except ImportError:
    PROJECT_DIR = Path(os.environ.get("SKRIPSI_DIR", "."))
OUT_DIR, LOG_DIR = PROJECT_DIR / "02_processed", PROJECT_DIR / "logs"
EVAL_DIR = PROJECT_DIR / "04_eval"; EVAL_DIR.mkdir(exist_ok=True)
SEED = 42

# Cell 2 — Data dari Tahap 8

In [ ]:
R = pd.read_csv(OUT_DIR / "bobot_skill_karier.csv")
kat = pd.read_csv(OUT_DIR / "katalog_karier.csv")
uji = pd.read_csv(OUT_DIR / "holdout_uji.csv", keep_default_na=False)
uji = uji[uji["layak_uji"].astype(str) == "True"].copy()
uji["S"] = uji["skill_teknis"].str.split(";").map(set)
KARIER = kat["karier"].tolist()
W = {c: dict(zip(g["skill"], g["w"])) for c, g in R.groupby("karier")}
print("Profil uji:", len(uji), "| karier:", len(KARIER))

# Cell 3 — Skor: usulan (K berbobot demand) dan baseline (cosine biner)

In [ ]:
def skor_K(S):
    return {c: sum(w for s, w in W[c].items() if s in S) / sum(W[c].values()) for c in KARIER}
def skor_cos(S):
    return {c: len(S & set(W[c])) / np.sqrt(len(S) * len(W[c])) for c in KARIER}
def peringkat(sk, benar):
    urut = sorted(KARIER, key=lambda c: (-sk[c], c))          # seri dipecah menurut nama (deterministik)
    return urut.index(benar) + 1, urut[:3]

rows = []
for _, r in uji.iterrows():
    rk, top_k = peringkat(skor_K(r["S"]), r["karier"])
    rb, top_b = peringkat(skor_cos(r["S"]), r["karier"])
    rows.append({"iid": r["iid"], "karier": r["karier"], "n_skill": len(r["S"]),
                 "rank_usulan": rk, "top1_usulan": top_k[0], "rank_baseline": rb, "top1_baseline": top_b[0]})
hasil = pd.DataFrame(rows)
hasil.to_csv(EVAL_DIR / "ranking_holdout_hasil.csv", index=False)

# Cell 4 — Metrik

In [ ]:
def metrik(df, col):
    r = df[col]
    return pd.Series({"Hit@1": (r == 1).mean(), "Hit@3": (r <= 3).mean(), "MRR": (1 / r).mean(), "n": len(r)})
per_karier = pd.concat({m: hasil.groupby("karier").apply(metrik, col=f"rank_{m}") for m in ["usulan", "baseline"]}, axis=1)
makro = {m: per_karier[m][["Hit@1", "Hit@3", "MRR"]].mean().round(3).to_dict() for m in ["usulan", "baseline"]}
mikro = {m: metrik(hasil, f"rank_{m}")[["Hit@1", "Hit@3", "MRR"]].round(3).to_dict() for m in ["usulan", "baseline"]}
print("Makro:", makro); print("Mikro:", mikro)
per_karier.round(3).to_csv(EVAL_DIR / "ranking_holdout_per_karier.csv")
per_karier.round(3)

# Cell 5 — Uji statistik berpasangan

In [ ]:
def mcnemar(a, b):                       # a, b: array boolean benar/salah pada profil yang sama
    n01, n10 = int((~a & b).sum()), int((a & ~b).sum())
    p = binomtest(n10, n01 + n10, 0.5).pvalue if n01 + n10 else 1.0
    return {"usulan_benar_baseline_salah": n10, "baseline_benar_usulan_salah": n01, "p": p}
uji_stat = {
    "McNemar_Hit@1": mcnemar((hasil.rank_usulan == 1).values, (hasil.rank_baseline == 1).values),
    "McNemar_Hit@3": mcnemar((hasil.rank_usulan <= 3).values, (hasil.rank_baseline <= 3).values),
}
rr_u, rr_b = 1 / hasil.rank_usulan, 1 / hasil.rank_baseline
uji_stat["Wilcoxon_RR"] = {"p": float(wilcoxon(rr_u, rr_b, zero_method="wilcox").pvalue)}
rng = np.random.default_rng(SEED)
diff = rr_u.values - rr_b.values
boot = [diff[rng.integers(0, len(diff), len(diff))].mean() for _ in range(2000)]
uji_stat["selisih_MRR_mikro"] = {"nilai": float(diff.mean()), "CI95": [float(np.percentile(boot, 2.5)), float(np.percentile(boot, 97.5))]}
print(json.dumps(uji_stat, indent=2))

# Cell 6 — Kebingungan antarkarier (top-1 usulan)

In [ ]:
conf = pd.crosstab(hasil["karier"], hasil["top1_usulan"])
conf.to_csv(EVAL_DIR / "ranking_holdout_confusion.csv")
sp = hasil[hasil["karier"].isin(["Senior Programmer", "Software Developer"])]
print(pd.crosstab(sp["karier"], sp["top1_usulan"]).loc[:, lambda d: d.sum() > 0])

# Cell 7 — Kriteria penerimaan (Tabel 3.13) dan log

In [ ]:
kriteria = {"Hit@3_makro>=0.80": makro["usulan"]["Hit@3"] >= 0.80,
            "MRR_usulan>baseline_p<0.05": (mikro["usulan"]["MRR"] > mikro["baseline"]["MRR"]) and uji_stat["Wilcoxon_RR"]["p"] < 0.05}
log = {"tanggal": datetime.datetime.now().isoformat(timespec="seconds"), "profil_uji": int(len(hasil)),
       "makro": makro, "mikro": mikro, "uji_statistik": uji_stat, "kriteria_terpenuhi": kriteria}
with open(LOG_DIR / "tahap10_log.json", "w") as fp:
    json.dump(log, fp, indent=2, ensure_ascii=False, default=float)
print(json.dumps(kriteria, indent=2))

---
# Tahap 11 — Knowledge base dan indeks (v3)
*Sumber: `11_kb_index_v3.ipynb`*

# Tahap 11 — Knowledge base v2, index FAISS, dan ambang τ
Bab III subbab 3.2.4 dan 3.5.4. Yang berubah dari KB Tahap 5–6:
1. Tag karier memakai 15 karier (Senior Programmer dan Software Developer dipisah).
2. 14 profil profesi lama diganti profil 15 karier dari data v2 (2 chunk per karier: kebutuhan industri; kurikulum dan standar).
3. Setiap chunk diberi label akses: **publik** (SKKNI, okupasi, profil karier) atau **privat** (RPS, deskripsi MK, kurikulum).
4. Chunk roadmap.sh tetap disimpan tetapi **tidak** ikut retrieval (bukan dokumen resmi).
5. Okupasi TIK.DEV0701 (dipakai Senior Programmer) ditambahkan.
6. Chunk profil lulusan prodi ditautkan ke empat karier profil lulusan.
7. Unit SKKNI hanya diambil dari bagian deskripsi, elemen, dan kriteria unjuk kerja (bagian batasan variabel
   dan panduan penilaian tidak ikut retrieval karena berisi teks asesmen yang sama di banyak unit).

**Bagian A** (Cell 1–9): bangun KB, embed, index, lalu buat lembar label τ (30 kueri × [top-5 + 3 pembanding], digabung per karier–passage).
**Bagian B** (Cell 10–13): setelah label diisi, hitung τ, uji mode publik, simpan konfigurasi dan artefak untuk web.

Runtime: pilih **GPU (T4)** supaya embedding ±2.500 chunk selesai dalam 1–3 menit (CPU ±15 menit).

# Cell 1 — Lokasi proyek dan instal

In [ ]:
import os, re, json, datetime, shutil
from pathlib import Path
import numpy as np, pandas as pd

try:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_DIR = Path("/content/drive/MyDrive/Skripsi_RAG")
    DI_COLAB = True
    os.system("pip install -q sentence-transformers faiss-cpu")
except ImportError:
    PROJECT_DIR = Path(os.environ.get("SKRIPSI_DIR", "."))
    DI_COLAB = False
INTERIM, PROC, KB_DIR = PROJECT_DIR / "01_interim", PROJECT_DIR / "02_processed", PROJECT_DIR / "03_kb"
EVAL_DIR, LOG_DIR, ART_DIR = PROJECT_DIR / "04_eval", PROJECT_DIR / "logs", PROJECT_DIR / "05_artifacts"
for d in (EVAL_DIR, ART_DIR):
    d.mkdir(exist_ok=True)
KB_VERSI, TOP_K, MAKS_CHAR = "v2", 5, 1500
print("Proyek:", PROJECT_DIR)

# Cell 2 — Data

In [ ]:
kb_lama = pd.read_json(KB_DIR / "kb_chunks.jsonl", lines=True)          # KB Tahap 5 (2.514 chunk), tidak diubah
kb_lama["profesi"] = kb_lama["profesi"].apply(lambda x: x if isinstance(x, list) else [])
manifest = pd.read_csv(LOG_DIR / "manifest.csv")
docs = pd.read_json(INTERIM / "extracted_docs.jsonl", lines=True)
kat = pd.read_csv(PROC / "katalog_karier.csv")
R = pd.read_csv(PROC / "bobot_skill_karier.csv")
cak = pd.read_csv(PROC / "cakupan_kurikulum_karier_v2.csv", keep_default_na=False)
tri = pd.read_csv(PROC / "triangulasi_skill_karier.csv")
split = pd.read_csv(PROC / "split_iklan.csv")
mks = pd.read_csv(PROC / "mk_skill_terverifikasi.csv", dtype=str)
mks = mks[mks["valid"].str.replace(".0", "", regex=False) == "1"]
KARIER = kat["karier"].tolist()
print("KB lama:", len(kb_lama), "| karier:", len(KARIER), "| skill syarat:", len(R), "| pasangan MK–skill:", len(mks))

# Cell 3 — Tag karier baru

In [ ]:
# 3a. Dokumen standar per karier (sama dengan Tahap 9)
STANDAR = {
 "Senior Programmer": ["2016-282", "2019-022", "TIK.DEV0701"],
 "Software Developer": ["2016-282", "2019-022", "TIK.DEV0615"],
 "Mobile Developer": ["2015-458"],
 "System Analyst": ["2017-044", "2015-090", "TIK.DEV0613"],
 "Database Administrator": ["2020-268", "TIK.DSC0604"],
 "QA / Software Tester": ["2017-047"],
 "UI/UX Designer": ["2018-107", "2023-126", "TIK.SRV0506"],
 "Network / System Administrator": ["2016-321", "2017-082", "TIK.INF0402", "TIK.INF0403", "TIK.INF0502", "TIK.INF0504",
                                    "TIK.INF0506", "TIK.INF0602", "TIK.INF0605", "TIK.INF0609",
                                    "TIK.INF0604-DESAINER JARINGAN (NETWORK DESIGNER).pdf"],
 "IT Support / Helpdesk": ["2016-285", "kmnaker030"],
 "Cyber Security": ["kmnaker191", "kmnaker120", "2022-023", "Web & Mobile Development 2020-391", "TIK.INF0611", "TIK.SEC0603"],
 "DevOps / Cloud Engineer": ["2023-102"],
 "Data Analyst / BI": ["SKKNI Data & Database Management 2020-299"],
 "Data Engineer": ["2020-268", "SKKNI Data & Database Management 2020-299"],
 "Data Scientist / ML / AI Engineer": ["SKKNI Data & Database Management 2020-299", "kmnaker103"],
 "IT Project/Product Manager": ["kmnaker200"],
}
FILE_KARIER = {}
for c, kunci in STANDAR.items():
    for k in kunci:
        hit = manifest[manifest["file_name"].str.contains(k, regex=False) & manifest["kelompok"].isin(["skkni", "okupasi"])
                       & (manifest["status"] != "salinan")]["file_id"]
        assert len(hit), f"Tidak ketemu: {k}"
        for f in hit:
            FILE_KARIER.setdefault(f, set()).add(c)
FID22 = manifest.loc[manifest["file_name"].str.contains("2019-022", regex=False), "file_id"].iloc[0]
LAMA_KE_BARU = {"Software / Web Developer": ["Senior Programmer", "Software Developer"]}

# 3b. Mata kuliah dan RPS: karier yang himpunan syarat R(c)-nya memuat skill yang diajarkan MK tersebut
syarat = R.groupby("skill")["karier"].apply(set).to_dict()
mk_skill = mks.groupby("nama_mata_kuliah")["skill"].apply(set).to_dict()
mk_karier = {mk: sorted({c for s in ss for c in syarat.get(s, set())}) for mk, ss in mk_skill.items()}

def nama_mk(judul):
    return re.sub(r"^\[(Mata Kuliah|RPS)\]\s*", "", judul).strip()

def tag_baru(r):
    if r["jenis"] in ("skkni", "okupasi"):
        if r["file_id"] == FID22:                              # unit SKKNI 22/2019: tag heuristik per unit dipertahankan
            return sorted({b for p in r["profesi"] for b in LAMA_KE_BARU.get(p, [p])})
        return sorted(FILE_KARIER.get(r["file_id"], set()))
    if r["jenis"] in ("mata_kuliah", "rps"):
        return mk_karier.get(nama_mk(r["judul"]), [])
    if r["jenis"] == "kurikulum":                              # profil lulusan prodi: terikat ke karier profil lulusan
        for pola, c in PROFIL_LULUSAN.items():
            if re.search(pola, r["judul"], re.I):
                return [c]
    return []                                                  # CPL umum dan roadmap: tidak terikat karier

PROFIL_LULUSAN = {r"PEMROGRAM SENIOR": "Senior Programmer", r"DATABASE ADMINISTRATOR": "Database Administrator",
                  r"\] Software Developer": "Software Developer", r"\] System Analyst": "System Analyst"}

kb = kb_lama[kb_lama["jenis"] != "profil_profesi"].copy()      # 14 profil lama diganti di Cell 4
kb["karier"] = kb.apply(tag_baru, axis=1)
mk_tanpa_tag = kb[kb["jenis"].isin(["mata_kuliah", "rps"]) & (kb["karier"].str.len() == 0)]["judul"].map(nama_mk).unique()
print("MK/RPS tanpa skill syarat terverifikasi (tidak ikut retrieval per karier):", len(mk_tanpa_tag))

# Teks deskripsi MK lama menyebut nama profesi lama; diganti dengan daftar 15 karier
def ganti_relevan(r):
    if r["jenis"] != "mata_kuliah":
        return r["teks"]
    t = re.sub(r"\s*Relevan untuk profesi:.*$", "", r["teks"], flags=re.S)
    return t + (f" Relevan untuk karier: {', '.join(r['karier'])}." if r["karier"] else "")
kb["teks"] = kb.apply(ganti_relevan, axis=1)

# 3c. Okupasi TIK.DEV0701 (Senior Programmer) belum ada di KB lama
def bersih(t):
    return re.sub(r"\s+", " ", str(t)).strip()
def potong(teks, maks=MAKS_CHAR, overlap=200):
    teks = bersih(teks)
    if len(teks) <= maks:
        return [teks]
    out, i = [], 0
    while i < len(teks):
        j = min(i + maks, len(teks))
        if j < len(teks):
            k = max(teks.rfind(". ", i, j), teks.rfind("; ", i, j))
            j = k + 1 if k > i + maks * 0.5 else (teks.rfind(" ", i, j) if teks.rfind(" ", i, j) > i else j)
        out.append(teks[i:j].strip())
        if j >= len(teks):
            break
        i = max(j - overlap, i + 1)
    return out

baru = []
ada = set(kb["file_id"])
for fid in sorted(f for f in FILE_KARIER if f not in ada and manifest.set_index("file_id").loc[f, "kelompok"] == "okupasi"):
    t = bersih(" ".join(docs[docs["file_id"] == fid].sort_values("unit_no")["text"]))
    t = re.sub(r"Platform Stankomdigi by KOMDIGI.*?Indonesia \d{4}-\s*\d{2}-\d{2} \d{2}:\d{2}:\d{2}", "", t)
    nm = re.search(r"Nama Okupasi\s+(.*?)\s+Area Fungsi", t)
    judul = f"[Okupasi KOMDIGI] {nm.group(1) if nm else fid}"
    for n, p in enumerate(potong(t), 1):
        baru.append({"chunk_id": f"okupasi-{fid}-{n:02d}", "jenis": "okupasi", "judul": judul, "teks": f"{judul}\n{p}",
                     "profesi": [], "karier": sorted(FILE_KARIER[fid]), "file_id": fid,
                     "sumber": "Peta Okupasi Nasional KOMDIGI (Stankomdigi)", "halaman": "", "bagian": n})
print("Chunk okupasi baru:", len(baru), "dari", sorted({b['file_id'] for b in baru}))

# Cell 4 — 15 profil karier dari data v2 (publik: hanya data iklan publik, nama MK, dan standar)

In [ ]:
manifest_i = manifest.set_index("file_id")
def label_skkni(fid):
    r = manifest_i.loc[fid]
    bidang = re.sub(r"(?i)^bidang( keahlian)?\s*", "", str(r["isi_judul"])).title()
    return f"SKKNI {r['isi_nomor_tahun']} – {bidang}"

okup_judul = kb[kb["jenis"] == "okupasi"].groupby("file_id")["judul"].first().to_dict()
okup_judul.update({b["file_id"]: b["judul"] for b in baru})
n_iklan = split.groupby("karier").size()
mk_per_skill = mks.groupby("skill")["nama_mata_kuliah"].apply(lambda s: sorted(set(s))).to_dict()
kat_i, cak_i = kat.set_index("karier"), cak.set_index("karier")

profil = []
for i, c in enumerate(KARIER, 1):
    g = R[R["karier"] == c].sort_values("persen", ascending=False)
    t = tri[tri["karier"] == c]
    fids = sorted(f for f, cs in FILE_KARIER.items() if c in cs)
    sk = [label_skkni(f) for f in fids if manifest_i.loc[f, "kelompok"] == "skkni"]
    ok = [okup_judul[f].replace("[Okupasi KOMDIGI] ", "") for f in fids if f in okup_judul]
    diajar = [s for s in g["skill"] if s in mk_per_skill]
    # Bagian 1: kebutuhan industri
    b1 = [f"Karier: {c}.",
          ("Termasuk profil lulusan Prodi D4 Teknik Informatika PNJ." if kat_i.loc[c, "profil_lulusan"] == 1
           else "Karier alternatif di bidang TI (bukan profil lulusan prodi)."),
          f"Area minat: {kat_i.loc[c, 'area_minat'].replace(';', ', ')}.",
          f"Kebutuhan industri: {int(n_iklan.get(c, 0)):,} iklan lowongan TI Indonesia berdeskripsi lengkap pada data penelitian.",
          "Skill teknis yang disyaratkan, urut dari yang paling sering diminta (persentase iklan): "
          + "; ".join(f"{r.skill} ({r.persen}%)" for r in g.itertuples()) + "."]
    # Bagian 2: kurikulum dan standar
    b2 = [f"Karier: {c}. Cakupan kurikulum PNJ terhadap skill syarat (berbobot demand): {cak_i.loc[c, 'cakupan_%']}%."]
    if diajar:
        b2.append("Skill syarat yang dibekali mata kuliah: "
                  + "; ".join(f"{s} (MK: {', '.join(mk_per_skill[s])})" for s in diajar) + ".")
    if cak_i.loc[c, "gap_skill"]:
        b2.append(f"Skill syarat yang belum dibekali kurikulum (perlu belajar mandiri atau sertifikasi): {cak_i.loc[c, 'gap_skill']}.")
    if ok:
        b2.append("Okupasi terkait di Peta Okupasi Nasional KOMDIGI: " + "; ".join(ok) + ".")
    if sk:
        b2.append("Standar kompetensi acuan: " + "; ".join(sk) + ".")
    b2.append(f"Padanan O*NET: {kat_i.loc[c, 'onet_soc']}; {round(100 * t['di_onet'].mean())}% skill syarat juga tercantum di O*NET "
              f"dan {round(100 * t['di_salah_satu'].mean())}% di salah satu referensi standar.")
    for huruf, sub, bag in [("a", "kebutuhan industri", b1), ("b", "kurikulum dan standar", b2)]:
        judul = f"[Profil Karier] {c} – {sub}"
        profil.append({"chunk_id": f"profil_karier-{i:02d}{huruf}", "jenis": "profil_karier", "judul": judul,
                       "teks": f"{judul}\n" + " ".join(bag), "profesi": [], "karier": [c], "file_id": "",
                       "sumber": "Disusun dari data penelitian (Tahap 8–9): iklan lowongan, bobot demand, cakupan kurikulum, triangulasi",
                       "halaman": "", "bagian": 1 if huruf == "a" else 2})
panjang = pd.Series({p["chunk_id"]: len(p["teks"]) for p in profil})
print("Chunk profil karier:", len(profil), "| panjang maks:", panjang.max(), "karakter | >1.500:", list(panjang[panjang > MAKS_CHAR].index))
print(profil[2 * KARIER.index("Senior Programmer") + 1]["teks"])

# Cell 5 — Gabung, label akses, simpan KB v2

In [ ]:
kb = pd.concat([kb, pd.DataFrame(baru + profil)], ignore_index=True)
AKSES = {"rps": "privat", "mata_kuliah": "privat", "kurikulum": "privat",
         "skkni": "publik", "okupasi": "publik", "profil_karier": "publik", "roadmap": "publik"}
kb["akses"] = kb["jenis"].map(AKSES)
kb["retrieval"] = kb["jenis"] != "roadmap"                    # roadmap: hanya kosakata, tidak dikutip
assert kb["akses"].notna().all() and kb["chunk_id"].is_unique

# Unit SKKNI: hanya bagian deskripsi + elemen + kriteria unjuk kerja yang menjadi bukti. Bagian BATASAN VARIABEL
# dan PANDUAN PENILAIAN (format standar) atau "Required skills..." (SKKNI 22/2019 adopsi) adalah teks asesmen
# yang sama di banyak unit; chunk sesudah penanda itu tidak diambil, dan chunk yang memuat penanda dipotong.
PENANDA_STD = re.compile(r"BATASAN VARIABEL|PANDUAN PENILAIAN")
PENANDA_22 = re.compile(r"Required skills\b|BATASAN VARIABEL|PANDUAN PENILAIAN")    # huruf R kapital = judul bagian
n_sk_awal = int(kb["jenis"].eq("skkni").sum())
for _, g in kb[kb["jenis"] == "skkni"].groupby("judul"):
    pola, lewat = (PENANDA_22 if g["file_id"].iloc[0] == FID22 else PENANDA_STD), False
    for i in g.sort_values("bagian").index:
        if lewat:
            kb.at[i, "retrieval"] = False
            continue
        judul_i, isi = kb.at[i, "teks"].split("\n", 1)
        m = pola.search(isi)
        if m:
            lewat, isi = True, isi[:m.start()].strip()
            kb.at[i, "teks"] = f"{judul_i}\n{isi}"
            if len(isi) < 250:
                kb.at[i, "retrieval"] = False
sk = kb[kb["jenis"] == "skkni"]
assert sk.groupby("judul")["retrieval"].any().all(), "Ada unit SKKNI tanpa chunk yang bisa diambil"
print(f"SKKNI: {int(sk['retrieval'].sum())} dari {n_sk_awal} chunk diambil; {sk['judul'].nunique()} unit tetap terwakili")
kb["n_char"] = kb["teks"].str.len()
KOLOM = ["chunk_id", "jenis", "judul", "teks", "karier", "akses", "retrieval", "file_id", "sumber", "halaman", "bagian", "n_char"]
kb = kb[KOLOM]
kb.to_json(KB_DIR / "kb_v2_chunks.jsonl", orient="records", lines=True, force_ascii=False)
ring = kb.groupby(["jenis", "akses"]).agg(chunk=("chunk_id", "size"), ikut_retrieval=("retrieval", "sum")).reset_index()
print("Total chunk KB v2:", len(kb))
ring

# Cell 6 — Embedding dan index FAISS

In [ ]:
MODEL_NAME = "intfloat/multilingual-e5-base"
if os.environ.get("EMBED_MOCK") == "1":                       # hanya untuk uji lokal tanpa model; tidak dipakai di Colab
    from sklearn.feature_extraction.text import HashingVectorizer
    _hv = HashingVectorizer(n_features=768, alternate_sign=False, norm="l2")
    def embed(texts, prefix):
        return _hv.transform(texts).toarray().astype("float32")
else:
    import torch
    from sentence_transformers import SentenceTransformer
    model = SentenceTransformer(MODEL_NAME, device="cuda" if torch.cuda.is_available() else "cpu")
    model.max_seq_length = 512
    def embed(texts, prefix):                                 # E5: "passage: " untuk dokumen, "query: " untuk kueri
        return model.encode([prefix + t for t in texts], batch_size=64, normalize_embeddings=True,
                            show_progress_bar=len(texts) > 100, convert_to_numpy=True).astype("float32")
import faiss
emb = embed(kb["teks"].tolist(), "passage: ")
index = faiss.IndexFlatIP(emb.shape[1])                       # inner product = cosine karena vektor dinormalisasi
index.add(emb)
faiss.write_index(index, str(KB_DIR / "kb_v2_index.faiss"))
np.save(KB_DIR / "kb_v2_embeddings.npy", emb)
print("Vektor:", index.ntotal, "| dimensi:", emb.shape[1])

# Cell 7 — Fungsi retrieval (sama dengan yang akan dipakai backend)

In [ ]:
KARIER_SET = kb["karier"].map(set).tolist()
def kueri_karier(karier, cocok, gap):
    """Kueri Bab III 3.5.4: nama karier + skill yang cocok + tiga gap berbobot terbesar."""
    return (f"{karier}. Kompetensi yang dimiliki: {', '.join(cocok) or '-'}. "
            f"Kompetensi yang perlu ditingkatkan: {', '.join(gap[:3]) or '-'}.")

def ambil(karier, cocok, gap, mode="internal", k=TOP_K, tau=None):
    """Top-k passage untuk satu karier. mode='publik' membuang chunk privat (FR11)."""
    q = embed([kueri_karier(karier, cocok, gap)], "query: ")[0]
    mask = kb["retrieval"].to_numpy() & np.array([karier in s for s in KARIER_SET])
    if mode == "publik":
        mask &= (kb["akses"] == "publik").to_numpy()
    idx = np.where(mask)[0]
    skor = emb[idx] @ q
    urut = idx[np.argsort(-skor)]
    hasil = kb.iloc[urut][["chunk_id", "jenis", "akses", "judul", "teks"]].copy()
    hasil.insert(0, "skor", np.sort(skor)[::-1].round(4))
    hasil = hasil.drop_duplicates("judul").head(k).reset_index(drop=True)
    if tau is not None:
        hasil = hasil[hasil["skor"] >= tau].reset_index(drop=True)
    return hasil

print("Chunk yang bisa diambil per karier:")
print(pd.Series({c: int(sum(c in s for s, r in zip(KARIER_SET, kb["retrieval"]) if r)) for c in KARIER}).to_string())

# Cell 8 — 30 kueri pengembangan (2 per karier) dan kandidat label

In [ ]:
# Profil A: tiga skill berbobot terbesar dimiliki, tiga berikutnya menjadi gap.
# Profil B: tiga skill yang diajarkan kurikulum dimiliki, tiga skill yang tidak diajarkan menjadi gap.
RANK_PEMBANDING = (8, 15, 30)
dev, kandidat = [], []
for c in KARIER:
    g = R[R["karier"] == c].sort_values("w", ascending=False)["skill"].tolist()
    ajar = [s for s in g if s in mk_per_skill]
    tidak = [s for s in g if s not in mk_per_skill]
    profil_dev = {"A": (g[:3], g[3:6]), "B": ((ajar or g)[:3], (tidak or g[3:])[:3])}
    for v, (cocok, gap) in profil_dev.items():
        qid = f"{KARIER.index(c) + 1:02d}{v}"
        dev.append({"qid": qid, "karier": c, "cocok": "; ".join(cocok), "gap": "; ".join(gap),
                    "kueri": kueri_karier(c, cocok, gap)})
        # Top-5 (yang benar-benar dikirim ke LLM) + passage peringkat 8, 15, 30 sebagai pembanding berskor rendah.
        # Tanpa pembanding ini hampir semua kandidat relevan (kumpulan sudah difilter per karier) sehingga τ tak bisa ditentukan.
        for r, h in enumerate(ambil(c, cocok, gap, k=30).itertuples(), 1):
            if r > TOP_K and r not in RANK_PEMBANDING:
                continue
            kandidat.append({"qid": qid, "karier": c, "kueri": dev[-1]["kueri"], "rank": r, "skor": h.skor,
                             "chunk_id": h.chunk_id, "jenis": h.jenis, "judul": h.judul,
                             "isi_passage": h.teks.split("\n", 1)[-1]})
dev, kandidat = pd.DataFrame(dev), pd.DataFrame(kandidat)
dev.to_csv(EVAL_DIR / "tau_dev_kueri.csv", index=False)
kandidat.to_csv(EVAL_DIR / "tau_dev_kandidat.csv", index=False)
# Kueri A dan B satu karier sering mengambil passage yang sama: cukup dilabeli sekali per (karier, chunk).
kunci = ["karier", "chunk_id"]
unik = (kandidat.groupby(kunci, sort=False)
        .agg(jenis=("jenis", "first"), judul=("judul", "first"), isi_passage=("isi_passage", "first"),
             qid=("qid", lambda s: ", ".join(sorted(set(s)))),
             peringkat=("rank", lambda s: ", ".join(str(x) for x in sorted(set(s)))))
        .reset_index())
unik = unik.merge(dev.groupby("karier").agg(skill_kueri=("cocok", lambda s: " | ".join(s)),
                                            gap_kueri=("gap", lambda s: " | ".join(s))), on="karier")
unik["relevan_1_0"] = ""
unik = unik[["karier", "qid", "peringkat", "skill_kueri", "gap_kueri", "chunk_id", "jenis", "judul", "isi_passage", "relevan_1_0"]]
print("Kueri:", len(dev), "| pasangan kueri–passage:", len(kandidat), "| baris unik untuk dilabeli:", len(unik))
unik.groupby("jenis").size()

# Cell 9 — Buat Google Sheet untuk diberi label

In [ ]:
# Sheet "tau_dev_label_v2" dibuat di My Drive. Baca kolom isi_passage (teks lengkap yang akan diterima LLM),
# lalu isi kolom relevan_1_0. Sheet versi pertama (tau_dev_label) tidak dipakai lagi dan boleh dihapus.
SHEET_INFO = LOG_DIR / "tahap11_sheet_v2.json"
if DI_COLAB:
    from google.colab import auth
    auth.authenticate_user()
    import gspread
    from google.auth import default
    gc = gspread.authorize(default()[0])
    if SHEET_INFO.exists():
        print("Sheet sudah ada, tidak dibuat ulang:", json.load(open(SHEET_INFO))["url"])
    else:
        sh = gc.create("tau_dev_label_v2")
        ws = sh.sheet1
        ws.update([unik.columns.tolist()] + unik.astype(str).values.tolist())
        ws.freeze(rows=1)
        json.dump({"id": sh.id, "url": sh.url}, open(SHEET_INFO, "w"))
        print("Isi label di:", sh.url)
else:
    unik.to_csv(EVAL_DIR / "tau_dev_label_unik.csv", index=False)
    print("Lokal: isi 04_eval/tau_dev_label_unik.csv lalu simpan sebagai 04_eval/tau_dev_label_unik_terlabel.csv")

## Kriteria label (isi `relevan_1_0`)
Beri **1** bila passage dapat dipakai sebagai bukti untuk penjelasan karier tersebut, yaitu memuat salah satu:
- tugas, unit kompetensi, atau kriteria unjuk kerja yang berkaitan dengan skill yang dimiliki atau gap pada kueri;
- deskripsi okupasi atau profil karier yang sama dengan karier pada kueri;
- isi mata kuliah/RPS yang mengajarkan salah satu skill pada kueri.

Beri **0** bila passage hanya berisi teks administratif (daftar isi, tanda tangan, penetapan), membahas bidang lain,
atau terlalu umum sehingga tidak mendukung klaim apa pun tentang skill pada kueri.
Nilai berdasarkan kolom `isi_passage` (teks yang akan diterima LLM), **bukan judulnya**. Judul unit hampir selalu
cocok dengan kariernya, jadi menilai dari judul membuat semua baris menjadi 1 dan τ tidak bisa ditentukan.
Chunk `profil_karier` adalah ringkasan data sistem untuk karier itu sendiri: beri 1.
Kolom `peringkat` 8, 15, atau 30 menandai passage pembanding berskor lebih rendah; nilai dengan kriteria yang sama,
jangan otomatis 0 atau 1. Passage pembanding yang memang relevan tetap diberi 1.
Setelah selesai, jalankan Bagian B.

# Cell 10 — BAGIAN B: baca label

In [ ]:
if DI_COLAB and SHEET_INFO.exists():
    if "gc" not in globals():
        from google.colab import auth
        auth.authenticate_user()
        import gspread
        from google.auth import default
        gc = gspread.authorize(default()[0])
    lab_u = pd.DataFrame(gc.open_by_key(json.load(open(SHEET_INFO))["id"]).sheet1.get_all_records())
else:
    lab_u = pd.read_csv(EVAL_DIR / "tau_dev_label_unik_terlabel.csv")
lab_u["relevan_1_0"] = pd.to_numeric(lab_u["relevan_1_0"], errors="coerce")
kosong = int(lab_u["relevan_1_0"].isna().sum())
assert kosong == 0, f"Masih ada {kosong} baris belum dilabeli"
lab_u.to_csv(EVAL_DIR / "tau_dev_label_unik_terlabel.csv", index=False)
if "kandidat" not in globals():                               # runtime sempat restart: muat ulang hasil Cell 8
    kandidat = pd.read_csv(EVAL_DIR / "tau_dev_kandidat.csv")
    dev = pd.read_csv(EVAL_DIR / "tau_dev_kueri.csv")
lab = kandidat.drop(columns="isi_passage").merge(lab_u[["karier", "chunk_id", "relevan_1_0"]], on=["karier", "chunk_id"], how="left")
assert lab["relevan_1_0"].notna().all(), "Ada pasangan kueri–passage yang labelnya tidak ditemukan"
lab["skor"] = lab["skor"].astype(float)
print("Baris unik berlabel:", len(lab_u), "| relevan:", int(lab_u["relevan_1_0"].sum()))
lab.to_csv(EVAL_DIR / "tau_dev_terlabel.csv", index=False)
print("Label:", len(lab), "| relevan:", int(lab["relevan_1_0"].sum()), f"({lab['relevan_1_0'].mean():.0%})")
top5 = lab[lab["rank"] <= TOP_K]
print("Precision@5 rata-rata per kueri:", round(top5.groupby("qid")["relevan_1_0"].mean().mean(), 3),
      "| relevan di passage pembanding (rank 8/15/30):", f"{lab.loc[lab['rank'] > TOP_K, 'relevan_1_0'].mean():.0%}")
lab.groupby(["jenis"])["relevan_1_0"].agg(["size", "mean"]).round(2)

# Cell 11 — Pilih τ (Youden J: pemisah terbaik passage relevan vs tidak relevan)

In [ ]:
y, s = lab["relevan_1_0"].astype(int).to_numpy(), lab["skor"].to_numpy()
assert 0 < y.sum() < len(y), ("Semua label sama. Periksa lagi passage pembanding (rank 8/15/30) dan passage SKKNI/RPS "
                              "berdasarkan isi_passage; τ hanya bisa ditentukan bila ada passage yang tidak relevan.")
rows = []
for t in np.unique(s):
    pred = s >= t
    tpr, fpr = (pred & (y == 1)).sum() / (y == 1).sum(), (pred & (y == 0)).sum() / (y == 0).sum()
    prec = (pred & (y == 1)).sum() / pred.sum()
    rows.append({"tau": float(t), "TPR": tpr, "FPR": fpr, "J": tpr - fpr, "presisi": prec})
kurva = pd.DataFrame(rows)
best = kurva.sort_values(["J", "tau"], ascending=[False, False]).iloc[0]
TAU = round(float(best["tau"]), 4)
lab["lolos_tau"] = (lab["skor"] >= TAU) & (lab["rank"] <= TOP_K)        # yang dikirim ke LLM: top-5 yang ≥ τ
per_q = lab[lab["rank"] <= TOP_K].groupby("qid").agg(lolos=("lolos_tau", "sum"), relevan_lolos=("relevan_1_0", lambda r: int((r[lab.loc[r.index, "lolos_tau"]] == 1).sum())))
abstain = int((per_q["lolos"] == 0).sum())
kurva.to_csv(EVAL_DIR / "tau_dev_kurva.csv", index=False)
print(f"τ = {TAU} | TPR {best.TPR:.2f} | FPR {best.FPR:.2f} | presisi {best.presisi:.2f}")
print(f"Kueri tanpa passage ≥ τ (sistem menyatakan bukti tidak cukup): {abstain} dari {len(per_q)}")
print("Rata-rata passage yang lolos per kueri:", round(per_q["lolos"].mean(), 2))

# Cell 12 — Uji mode publik (TC12) pada 30 kueri

In [ ]:
bocor = 0
for r in dev.itertuples():
    h = ambil(r.karier, r.cocok.split("; "), r.gap.split("; "), mode="publik", tau=TAU)
    bocor += int((h["akses"] == "privat").sum())
assert bocor == 0, "Ada passage privat yang terambil di mode publik"
n_publik = [len(ambil(r.karier, r.cocok.split("; "), r.gap.split("; "), mode="publik", tau=TAU)) for r in dev.itertuples()]
print("Mode publik: 0 passage privat | kueri tanpa bukti di mode publik:", sum(n == 0 for n in n_publik), "dari", len(dev))

# Cell 13 — Konfigurasi, artefak untuk web, dan log

In [ ]:
konfig = {"kb_versi": KB_VERSI, "model_embedding": MODEL_NAME, "prefix_query": "query: ", "prefix_passage": "passage: ",
          "index": "FAISS IndexFlatIP (cosine, vektor dinormalisasi)", "top_k": TOP_K, "tau": TAU,
          "jenis_tidak_diambil": ["roadmap"], "akses_mode_publik": ["publik"],
          "kueri": "{karier}. Kompetensi yang dimiliki: {cocok}. Kompetensi yang perlu ditingkatkan: {gap[:3]}."}
json.dump(konfig, open(KB_DIR / "retrieval_config.json", "w"), indent=2, ensure_ascii=False)

# 05_artifacts/ = semua yang dibutuhkan backend web. Folder ini memuat chunk PRIVAT: jangan diunggah ke repo publik.
for f in ["kb_v2_chunks.jsonl", "kb_v2_index.faiss", "retrieval_config.json"]:
    shutil.copy(KB_DIR / f, ART_DIR / f)
for f in ["katalog_karier.csv", "bobot_skill_karier.csv", "kamus_skill_v3.csv", "mk_skill_terverifikasi.csv",
          "cakupan_kurikulum_karier_v2.csv"]:
    shutil.copy(PROC / f, ART_DIR / f)
kb[kb["akses"] == "publik"].to_json(ART_DIR / "kb_v2_chunks_publik.jsonl", orient="records", lines=True, force_ascii=False)

log = {"tanggal": datetime.datetime.now().isoformat(timespec="seconds"), "kb_versi": KB_VERSI, "total_chunk": int(len(kb)),
       "per_jenis": kb["jenis"].value_counts().to_dict(), "per_akses": kb["akses"].value_counts().to_dict(),
       "ikut_retrieval": int(kb["retrieval"].sum()), "model": MODEL_NAME, "dimensi": int(emb.shape[1]),
       "dev": {"kueri": int(len(dev)), "label": int(len(lab)), "relevan_%": round(100 * lab["relevan_1_0"].mean(), 1),
               "precision@5": round(top5.groupby("qid")["relevan_1_0"].mean().mean(), 3),
               "rank_pembanding": list(RANK_PEMBANDING),
               "tau": TAU, "TPR": round(float(best.TPR), 3), "FPR": round(float(best.FPR), 3),
               "kueri_tanpa_bukti": abstain},
       "mode_publik_bocor": bocor, "artefak": sorted(p.name for p in ART_DIR.iterdir())}
json.dump(log, open(LOG_DIR / "tahap11_log.json", "w"), indent=2, ensure_ascii=False)
print(json.dumps({k: v for k, v in log.items() if k != "artefak"}, indent=2, ensure_ascii=False))

---
# Tahap 11b — Penentuan tau (CPU)
*Sumber: `11b_tau_cpu.ipynb`*

# Tahap 11B — Penentuan τ, uji mode publik, dan artefak (CPU, tanpa GPU)
Lanjutan `11_kb_index_v3` Cell 1–9 yang sudah dijalankan. Notebook ini **tidak** membangun ulang index:
memuat `kb_v2_chunks.jsonl` dan `kb_v2_embeddings.npy` yang sudah tersimpan, lalu membaca label
`04_eval/tau_dev_label_unik_terlabel.csv`. Runtime **CPU** cukup (model hanya mengubah 30 kueri menjadi vektor).

Label relevansi (158 pasangan karier–passage) diisi dengan bantuan asisten AI berdasarkan kriteria tertulis di
notebook 11 dan ditinjau peneliti; hal ini dicatat di log (`label_oleh`).

# Cell 1 — Lokasi proyek dan instal

In [ ]:
import os, re, json, datetime, shutil
from pathlib import Path
import numpy as np, pandas as pd

try:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_DIR = Path("/content/drive/MyDrive/Skripsi_RAG")
    os.system("pip install -q sentence-transformers faiss-cpu")
except ImportError:
    PROJECT_DIR = Path(os.environ.get("SKRIPSI_DIR", "."))
PROC, KB_DIR = PROJECT_DIR / "02_processed", PROJECT_DIR / "03_kb"
EVAL_DIR, LOG_DIR, ART_DIR = PROJECT_DIR / "04_eval", PROJECT_DIR / "logs", PROJECT_DIR / "05_artifacts"
ART_DIR.mkdir(exist_ok=True)
KB_VERSI, TOP_K = "v2", 5
RANK_PEMBANDING = (8, 15, 30)

# Cell 2 — Muat KB, embedding, kueri, dan kandidat hasil notebook 11

In [ ]:
kb = pd.read_json(KB_DIR / "kb_v2_chunks.jsonl", lines=True)
emb = np.load(KB_DIR / "kb_v2_embeddings.npy")
assert len(kb) == len(emb), "kb_v2_chunks dan kb_v2_embeddings tidak sinkron: jalankan ulang notebook 11 Cell 1–6"
dev = pd.read_csv(EVAL_DIR / "tau_dev_kueri.csv", keep_default_na=False)
kandidat = pd.read_csv(EVAL_DIR / "tau_dev_kandidat.csv")
KARIER = pd.read_csv(PROC / "katalog_karier.csv")["karier"].tolist()
MODEL_NAME = "intfloat/multilingual-e5-base"
if os.environ.get("EMBED_MOCK") == "1":                       # hanya untuk uji lokal
    from sklearn.feature_extraction.text import HashingVectorizer
    _hv = HashingVectorizer(n_features=emb.shape[1], alternate_sign=False, norm="l2")
    def embed(texts, prefix):
        return _hv.transform(texts).toarray().astype("float32")
else:
    from sentence_transformers import SentenceTransformer
    model = SentenceTransformer(MODEL_NAME, device="cpu")
    def embed(texts, prefix):
        return model.encode([prefix + t for t in texts], normalize_embeddings=True, convert_to_numpy=True).astype("float32")
print("Chunk:", len(kb), "| kueri:", len(dev), "| kandidat:", len(kandidat))

# Cell 3 — Fungsi retrieval (sama dengan notebook 11 Cell 7)

In [ ]:
KARIER_SET = kb["karier"].map(set).tolist()
def kueri_karier(karier, cocok, gap):
    """Kueri Bab III 3.5.4: nama karier + skill yang cocok + tiga gap berbobot terbesar."""
    return (f"{karier}. Kompetensi yang dimiliki: {', '.join(cocok) or '-'}. "
            f"Kompetensi yang perlu ditingkatkan: {', '.join(gap[:3]) or '-'}.")

def ambil(karier, cocok, gap, mode="internal", k=TOP_K, tau=None):
    """Top-k passage untuk satu karier. mode='publik' membuang chunk privat (FR11)."""
    q = embed([kueri_karier(karier, cocok, gap)], "query: ")[0]
    mask = kb["retrieval"].to_numpy() & np.array([karier in s for s in KARIER_SET])
    if mode == "publik":
        mask &= (kb["akses"] == "publik").to_numpy()
    idx = np.where(mask)[0]
    skor = emb[idx] @ q
    urut = idx[np.argsort(-skor)]
    hasil = kb.iloc[urut][["chunk_id", "jenis", "akses", "judul", "teks"]].copy()
    hasil.insert(0, "skor", np.sort(skor)[::-1].round(4))
    hasil = hasil.drop_duplicates("judul").head(k).reset_index(drop=True)
    if tau is not None:
        hasil = hasil[hasil["skor"] >= tau].reset_index(drop=True)
    return hasil

print("Chunk yang bisa diambil per karier:")
print(pd.Series({c: int(sum(c in s for s, r in zip(KARIER_SET, kb["retrieval"]) if r)) for c in KARIER}).to_string())

# Cell 4 — Baca label

In [ ]:
lab_u = pd.read_csv(EVAL_DIR / "tau_dev_label_unik_terlabel.csv")
lab_u["relevan_1_0"] = pd.to_numeric(lab_u["relevan_1_0"], errors="coerce")
kosong = int(lab_u["relevan_1_0"].isna().sum())
assert kosong == 0, f"Masih ada {kosong} baris belum dilabeli"
lab_u.to_csv(EVAL_DIR / "tau_dev_label_unik_terlabel.csv", index=False)
lab = kandidat.drop(columns="isi_passage").merge(lab_u[["karier", "chunk_id", "relevan_1_0"]], on=["karier", "chunk_id"], how="left")
assert lab["relevan_1_0"].notna().all(), "Ada pasangan kueri–passage yang labelnya tidak ditemukan"
lab["skor"] = lab["skor"].astype(float)
print("Baris unik berlabel:", len(lab_u), "| relevan:", int(lab_u["relevan_1_0"].sum()))
lab.to_csv(EVAL_DIR / "tau_dev_terlabel.csv", index=False)
print("Label:", len(lab), "| relevan:", int(lab["relevan_1_0"].sum()), f"({lab['relevan_1_0'].mean():.0%})")
top5 = lab[lab["rank"] <= TOP_K]
print("Precision@5 rata-rata per kueri:", round(top5.groupby("qid")["relevan_1_0"].mean().mean(), 3),
      "| relevan di passage pembanding (rank 8/15/30):", f"{lab.loc[lab['rank'] > TOP_K, 'relevan_1_0'].mean():.0%}")
lab.groupby(["jenis"])["relevan_1_0"].agg(["size", "mean"]).round(2)

# Cell 5 — Pilih τ (Youden J)

In [ ]:
y, s = lab["relevan_1_0"].astype(int).to_numpy(), lab["skor"].to_numpy()
assert 0 < y.sum() < len(y), ("Semua label sama. Periksa lagi passage pembanding (rank 8/15/30) dan passage SKKNI/RPS "
                              "berdasarkan isi_passage; τ hanya bisa ditentukan bila ada passage yang tidak relevan.")
rows = []
for t in np.unique(s):
    pred = s >= t
    tpr, fpr = (pred & (y == 1)).sum() / (y == 1).sum(), (pred & (y == 0)).sum() / (y == 0).sum()
    prec = (pred & (y == 1)).sum() / pred.sum()
    rows.append({"tau": float(t), "TPR": tpr, "FPR": fpr, "J": tpr - fpr, "presisi": prec})
kurva = pd.DataFrame(rows)
best = kurva.sort_values(["J", "tau"], ascending=[False, False]).iloc[0]
TAU = round(float(best["tau"]), 4)
lab["lolos_tau"] = (lab["skor"] >= TAU) & (lab["rank"] <= TOP_K)        # yang dikirim ke LLM: top-5 yang ≥ τ
per_q = lab[lab["rank"] <= TOP_K].groupby("qid").agg(lolos=("lolos_tau", "sum"), relevan_lolos=("relevan_1_0", lambda r: int((r[lab.loc[r.index, "lolos_tau"]] == 1).sum())))
abstain = int((per_q["lolos"] == 0).sum())
kurva.to_csv(EVAL_DIR / "tau_dev_kurva.csv", index=False)
print(f"τ = {TAU} | TPR {best.TPR:.2f} | FPR {best.FPR:.2f} | presisi {best.presisi:.2f}")
print(f"Kueri tanpa passage ≥ τ (sistem menyatakan bukti tidak cukup): {abstain} dari {len(per_q)}")
print("Rata-rata passage yang lolos per kueri:", round(per_q["lolos"].mean(), 2))

# Cell 6 — Uji mode publik (TC12)

In [ ]:
bocor = 0
for r in dev.itertuples():
    h = ambil(r.karier, r.cocok.split("; "), r.gap.split("; "), mode="publik", tau=TAU)
    bocor += int((h["akses"] == "privat").sum())
assert bocor == 0, "Ada passage privat yang terambil di mode publik"
n_publik = [len(ambil(r.karier, r.cocok.split("; "), r.gap.split("; "), mode="publik", tau=TAU)) for r in dev.itertuples()]
print("Mode publik: 0 passage privat | kueri tanpa bukti di mode publik:", sum(n == 0 for n in n_publik), "dari", len(dev))

# Cell 7 — Konfigurasi, artefak untuk web, dan log

In [ ]:
konfig = {"kb_versi": KB_VERSI, "model_embedding": MODEL_NAME, "prefix_query": "query: ", "prefix_passage": "passage: ",
          "index": "FAISS IndexFlatIP (cosine, vektor dinormalisasi)", "top_k": TOP_K, "tau": TAU,
          "jenis_tidak_diambil": ["roadmap"], "akses_mode_publik": ["publik"],
          "kueri": "{karier}. Kompetensi yang dimiliki: {cocok}. Kompetensi yang perlu ditingkatkan: {gap[:3]}."}
json.dump(konfig, open(KB_DIR / "retrieval_config.json", "w"), indent=2, ensure_ascii=False)

# 05_artifacts/ = semua yang dibutuhkan backend web. Folder ini memuat chunk PRIVAT: jangan diunggah ke repo publik.
for f in ["kb_v2_chunks.jsonl", "kb_v2_index.faiss", "retrieval_config.json"]:
    shutil.copy(KB_DIR / f, ART_DIR / f)
for f in ["katalog_karier.csv", "bobot_skill_karier.csv", "kamus_skill_v3.csv", "mk_skill_terverifikasi.csv",
          "cakupan_kurikulum_karier_v2.csv"]:
    shutil.copy(PROC / f, ART_DIR / f)
kb[kb["akses"] == "publik"].to_json(ART_DIR / "kb_v2_chunks_publik.jsonl", orient="records", lines=True, force_ascii=False)

log = {"tanggal": datetime.datetime.now().isoformat(timespec="seconds"), "kb_versi": KB_VERSI, "total_chunk": int(len(kb)),
       "per_jenis": kb["jenis"].value_counts().to_dict(), "per_akses": kb["akses"].value_counts().to_dict(),
       "ikut_retrieval": int(kb["retrieval"].sum()), "model": MODEL_NAME, "dimensi": int(emb.shape[1]),
       "dev": {"kueri": int(len(dev)), "label": int(len(lab)), "relevan_%": round(100 * lab["relevan_1_0"].mean(), 1),
               "precision@5": round(top5.groupby("qid")["relevan_1_0"].mean().mean(), 3),
               "rank_pembanding": list(RANK_PEMBANDING),
               "tau": TAU, "TPR": round(float(best.TPR), 3), "FPR": round(float(best.FPR), 3),
               "kueri_tanpa_bukti": abstain},
       "label_oleh": "asisten AI dengan kriteria tertulis; ditinjau peneliti", "mode_publik_bocor": bocor, "artefak": sorted(p.name for p in ART_DIR.iterdir())}
json.dump(log, open(LOG_DIR / "tahap11_log.json", "w"), indent=2, ensure_ascii=False)
print(json.dumps({k: v for k, v in log.items() if k != "artefak"}, indent=2, ensure_ascii=False))

---
# Tahap 12 — Evaluasi LLM RAG vs non-RAG (v2)
*Sumber: `12_llm_rag_eval_v2.ipynb`*

# Tahap 12 — Dua tugas LLM, pemeriksa klaim, dan uji RAG vs non-RAG (RQ5)
Bab III subbab 3.5.3–3.5.4 (Tabel 3.11) dan 3.6.5. LLM **tidak** menghitung skor; LLM hanya:
1. mengekstrak skill kamus dari teks pengalaman bebas, dan
2. menulis penjelasan empat bagian dari data sistem (+ passage hasil retrieval pada kondisi RAG).

Uji: 60 profil uji (4 per karier, acak dari iklan hold-out) × 2 kondisi = 120 penjelasan. Kedua kondisi memakai
model, prompt, skor, daftar skill, format, dan bahasa (Inggris) yang sama; bedanya hanya ada/tidaknya passage.

Persiapan: buat API key gratis di console.groq.com, simpan di **Secrets Colab** (ikon kunci di panel kiri) dengan nama `GROQ_API_KEY`
(atau `GEMINI_API_KEY` bila PENYEDIA = "gemini"), aktifkan "Notebook access". Runtime **CPU** cukup. Semua jawaban LLM disimpan di cache, jadi kalau terputus
atau kena batas kuota, jalankan ulang saja: panggilan yang sudah berhasil tidak diulang.
**Jangan** mencetak API key; hapus output (Edit → Clear all outputs) sebelum notebook dibagikan.

# Cell 1 — Lokasi proyek dan instal

In [ ]:
import os, re, json, time, hashlib, datetime, random
from pathlib import Path
import numpy as np, pandas as pd

try:
    from google.colab import drive, userdata
    drive.mount('/content/drive')
    PROJECT_DIR = Path("/content/drive/MyDrive/Skripsi_RAG")
    os.system("pip install -q groq==1.7.0 google-genai==2.31.0 sentence-transformers faiss-cpu")
    DI_COLAB = True
except ImportError:
    PROJECT_DIR = Path(os.environ.get("SKRIPSI_DIR", "."))
    DI_COLAB = False
INTERIM, PROC, KB_DIR = PROJECT_DIR / "01_interim", PROJECT_DIR / "02_processed", PROJECT_DIR / "03_kb"
EVAL_DIR, LOG_DIR, ART_DIR = PROJECT_DIR / "04_eval", PROJECT_DIR / "logs", PROJECT_DIR / "05_artifacts"
SEED, N_PER_KARIER = 42, 4
PROMPT_VERSI = "v1"
# Lapisan dokumen yang boleh dikirim ke layanan LLM. "publik" = hanya SKKNI, okupasi, profil karier (tidak ada teks
# RPS/kurikulum PNJ yang keluar ke Google). Ubah ke "internal" hanya bila izin prodi untuk mengirim RPS sudah ada.
MODE_RETRIEVAL = "internal"
# Penyedia LLM: "groq" (kuota gratis lebih longgar) atau "gemini". Satu penyedia dan satu model untuk seluruh evaluasi.
PENYEDIA = "groq"
print("Proyek:", PROJECT_DIR, "| mode retrieval:", MODE_RETRIEVAL)

# Cell 2 — Data hasil Tahap 8–11

In [ ]:
cfg = json.load(open(KB_DIR / "retrieval_config.json"))
TAU, TOP_K = cfg["tau"], cfg["top_k"]
kb = pd.read_json(KB_DIR / "kb_v2_chunks.jsonl", lines=True)
emb = np.load(KB_DIR / "kb_v2_embeddings.npy")
kat = pd.read_csv(PROC / "katalog_karier.csv")
R = pd.read_csv(PROC / "bobot_skill_karier.csv")
kamus = pd.read_csv(PROC / "kamus_skill_v3.csv", keep_default_na=False)
cak = pd.read_csv(PROC / "cakupan_kurikulum_karier_v2.csv", keep_default_na=False).set_index("karier")
mks = pd.read_csv(PROC / "mk_skill_terverifikasi.csv", dtype=str)
mks = mks[mks["valid"].str.replace(".0", "", regex=False) == "1"]
mk_list = pd.read_csv(INTERIM / "manual" / "F031_mata_kuliah_kurikulum2021.csv", dtype=str)
NAMA_MK = sorted(set(mk_list["nama_mata_kuliah"].dropna().str.strip()))
uji = pd.read_csv(PROC / "holdout_uji.csv", keep_default_na=False)
uji = uji[uji["layak_uji"].astype(str) == "True"].copy()
n_iklan = pd.read_csv(PROC / "split_iklan.csv").groupby("karier").size()
KARIER = kat["karier"].tolist()
SKILL = kamus["skill"].tolist()
SOFT = set(kamus.loc[kamus["kategori"] == "Soft skill", "skill"])
POLA = {r["skill"]: re.compile(r["pola"], re.I) for _, r in kamus.iterrows()}
W = {c: dict(zip(g["skill"], g["w"])) for c, g in R.groupby("karier")}
PERSEN = {(r.karier, r.skill): float(r.persen) for r in R.itertuples()}
MK_SKILL = mks.groupby("nama_mata_kuliah")["skill"].apply(set).to_dict()
SKILL_MK = mks.groupby("skill")["nama_mata_kuliah"].apply(lambda s: sorted(set(s))).to_dict()
print(f"τ = {TAU} | chunk: {len(kb)} | iklan uji layak: {len(uji)} | MK: {len(NAMA_MK)}")

# Cell 3 — Retriever (sama dengan Tahap 11)

In [ ]:
if os.environ.get("EMBED_MOCK") == "1":                       # hanya uji lokal
    from sklearn.feature_extraction.text import HashingVectorizer
    _hv = HashingVectorizer(n_features=emb.shape[1], alternate_sign=False, norm="l2")
    def embed(texts, prefix):
        return _hv.transform(texts).toarray().astype("float32")
else:
    from sentence_transformers import SentenceTransformer
    _st = SentenceTransformer(cfg["model_embedding"], device="cpu")
    def embed(texts, prefix):
        return _st.encode([prefix + t for t in texts], normalize_embeddings=True, convert_to_numpy=True).astype("float32")
KARIER_SET = kb["karier"].map(set).tolist()

def ambil(karier, cocok, gap, mode=MODE_RETRIEVAL, k=TOP_K, tau=TAU):
    q = embed([f"{karier}. Kompetensi yang dimiliki: {', '.join(cocok) or '-'}. "
               f"Kompetensi yang perlu ditingkatkan: {', '.join(gap[:3]) or '-'}."], "query: ")[0]
    mask = kb["retrieval"].to_numpy() & np.array([karier in s for s in KARIER_SET])
    if mode == "publik":
        mask &= (kb["akses"] == "publik").to_numpy()
    idx = np.where(mask)[0]
    skor = emb[idx] @ q
    o = np.argsort(-skor)
    h = kb.iloc[idx[o]][["chunk_id", "jenis", "akses", "judul", "teks"]].copy()
    h.insert(0, "skor", skor[o].round(4))
    h = h.drop_duplicates("judul").head(k)
    return h[h["skor"] >= tau].reset_index(drop=True)

# Cell 4 — Klien LLM: model dikunci, suhu 0, cache, dan coba ulang saat kuota habis

In [ ]:
CACHE = EVAL_DIR / "llm_cache.jsonl"
MODEL_FILE = LOG_DIR / f"tahap12_model_{PENYEDIA}.json"
MOCK = os.environ.get("LLM_MOCK") == "1"                     # hanya uji lokal tanpa internet
def rahasia(nama):
    return userdata.get(nama) if DI_COLAB else os.environ[nama]

# Urutan pilihan model; yang pertama tersedia di akun dipakai lalu dikunci di MODEL_FILE.
PILIHAN = {"groq": ["openai/gpt-oss-120b", "llama-3.3-70b-versatile", "moonshotai/kimi-k2-instruct", "openai/gpt-oss-20b"],
           "gemini": ["gemini-2.5-flash", "gemini-2.0-flash", "gemini-flash-latest"]}
if MOCK:
    MODEL = "mock-llm"
else:
    if PENYEDIA == "groq":
        from groq import Groq, RateLimitError, APIStatusError, APIConnectionError
        client = Groq(api_key=rahasia("GROQ_API_KEY"))
        tersedia = [m.id for m in client.models.list().data]
    else:
        from google import genai
        from google.genai import types, errors
        client = genai.Client(api_key=rahasia("GEMINI_API_KEY"))
        tersedia = [m.name.split("/")[-1] for m in client.models.list() if "generateContent" in (m.supported_actions or [])]
    if MODEL_FILE.exists():                                   # model dikunci sejak run pertama (Bab III 3.5.3)
        MODEL = json.load(open(MODEL_FILE))["model"]
    else:
        cocok_model = [m for m in PILIHAN[PENYEDIA] if m in tersedia]
        assert cocok_model, f"Tidak ada model pilihan yang tersedia. Model di akun: {tersedia}"
        MODEL = cocok_model[0]
        json.dump({"penyedia": PENYEDIA, "model": MODEL, "dipilih": datetime.datetime.now().isoformat(timespec="seconds")},
                  open(MODEL_FILE, "w"), indent=2)
print("Penyedia:", PENYEDIA, "| model (dikunci untuk seluruh evaluasi):", MODEL)

_cache = {}
if CACHE.exists():
    for line in open(CACHE, encoding="utf-8"):
        r = json.loads(line); _cache[r["kunci"]] = r
JEDA = 2.5 if PENYEDIA == "groq" else 6.5                     # detik antarpanggilan agar tidak melewati batas per menit

def _sekali(sistem, pengguna, skema):
    if PENYEDIA == "groq":                                    # mode JSON; skema ditulis di instruksi lalu divalidasi
        resp = client.chat.completions.create(
            model=MODEL, temperature=0, seed=SEED, response_format={"type": "json_object"},
            messages=[{"role": "system", "content": sistem + " Reply with one JSON object that follows this JSON schema: "
                       + json.dumps(skema)}, {"role": "user", "content": pengguna}])
        return json.loads(resp.choices[0].message.content)
    resp = client.models.generate_content(model=MODEL, contents=pengguna, config=types.GenerateContentConfig(
        system_instruction=sistem, temperature=0, seed=SEED, response_mime_type="application/json", response_json_schema=skema))
    return json.loads(resp.text)

def _tunggu(e, coba):
    if PENYEDIA == "groq" and isinstance(e, RateLimitError):
        try:
            return float(e.response.headers.get("retry-after", 0)) + 1
        except Exception:
            pass
    return 20 * (coba + 1)

def panggil_llm(tugas, sistem, pengguna, skema, mock=None):
    """Satu panggilan LLM berformat JSON. Hasil disimpan di cache per (model, prompt)."""
    kunci = hashlib.sha256(json.dumps([MODEL, PROMPT_VERSI, tugas, sistem, pengguna], ensure_ascii=False).encode()).hexdigest()
    if kunci in _cache:
        return _cache[kunci]["jawaban"]
    if MOCK:
        jawab = mock()
    else:
        for coba in range(8):
            try:
                jawab = _sekali(sistem, pengguna, skema)
                if all(k in jawab for k in skema.get("required", [])):
                    break
                raise json.JSONDecodeError("kunci JSON tidak lengkap", "", 0)
            except json.JSONDecodeError:
                time.sleep(3)
            except Exception as e:
                kuota = ("RateLimit" in type(e).__name__) or getattr(e, "code", None) == 429 or "429" in str(e)[:20]
                if "limit: 0" in str(e):
                    raise RuntimeError(f"Model {MODEL} tidak punya kuota gratis di akun ini. Hapus {MODEL_FILE.name} "
                                       "di folder logs lalu ganti PENYEDIA/PILIHAN.") from e
                if not kuota and coba >= 2:
                    raise
                time.sleep(_tunggu(e, coba))
        else:
            raise RuntimeError("LLM gagal menjawab setelah 8 percobaan (kemungkinan kuota harian habis; lanjutkan besok, cache aman)")
        time.sleep(JEDA)
    rec = {"kunci": kunci, "tugas": tugas, "penyedia": PENYEDIA, "model": MODEL, "prompt_versi": PROMPT_VERSI,
           "waktu": datetime.datetime.now().isoformat(timespec="seconds"), "jawaban": jawab}
    with open(CACHE, "a", encoding="utf-8") as f:
        f.write(json.dumps(rec, ensure_ascii=False) + "\n")
    _cache[kunci] = rec
    return jawab

# Cell 5 — Tugas 1: ekstraksi skill dari teks pengalaman (Tabel 3.11 baris 1)

In [ ]:
SISTEM_EKSTRAK = (
    "You extract technical and soft skills from a student's free-text description of their experience. "
    "Use ONLY skill names from the provided dictionary, written exactly as in the dictionary. "
    "For each skill, quote the exact phrase from the text that shows it. Do not infer skills that are not stated. "
    "If no dictionary skill is stated, return an empty list.")
SKEMA_EKSTRAK = {"type": "object", "properties": {"skills": {"type": "array", "items": {"type": "object",
                 "properties": {"skill": {"type": "string"}, "phrase": {"type": "string"}},
                 "required": ["skill", "phrase"]}}}, "required": ["skills"]}

def ekstrak_skill(teks):
    pengguna = f"Skill dictionary: {json.dumps(SKILL, ensure_ascii=False)}\n\nExperience text:\n{teks}"
    mock = lambda: {"skills": [{"skill": s, "phrase": POLA[s].search(teks).group(0)} for s in SKILL if POLA[s].search(teks)]}
    hasil = panggil_llm("ekstraksi", SISTEM_EKSTRAK, pengguna, SKEMA_EKSTRAK, mock)
    diterima, ditolak = [], []
    for it in hasil.get("skills", []):
        ok = it.get("skill") in SKILL and it.get("phrase", "").lower() in teks.lower()
        (diterima if ok else ditolak).append(it)
    return diterima, ditolak                                  # di web: mahasiswa mengonfirmasi/menghapus tiap skill

CONTOH = ["Saya pernah magang 3 bulan membuat REST API dengan Laravel dan MySQL, lalu deploy pakai Docker.",
          "During my internship I built Power BI dashboards from Excel and SQL data for the finance team.",
          "Saya aktif di organisasi kampus sebagai ketua divisi acara."]
for t in CONTOH:
    d, x = ekstrak_skill(t)
    print([i["skill"] for i in d], "| ditolak:", [i["skill"] for i in x])

# Cell 6 — 60 profil uji (4 per karier dari iklan hold-out) dan karier teratasnya

In [ ]:
def skor_K(S):
    return {c: sum(w for s, w in W[c].items() if s in S) / sum(W[c].values()) for c in KARIER}

prof = (uji.groupby("karier", group_keys=False).sample(n=N_PER_KARIER, random_state=SEED).reset_index(drop=True))
rows = []
for r in prof.itertuples():
    S = set(r.skill_teknis.split(";"))
    K = skor_K(S)
    c = sorted(KARIER, key=lambda x: (-K[x], x))[0]           # yang dijelaskan = karier peringkat 1 (seperti di web)
    req = sorted(W[c], key=lambda s: -W[c][s])
    cocok = [s for s in req if s in S]
    gap = [s for s in req if s not in S]
    rows.append({"pid": f"P{len(rows) + 1:02d}", "iid": r.iid, "karier_iklan": r.karier, "karier_top1": c,
                 "fit": round(100 * K[c], 1), "skill_profil": ";".join(sorted(S)),
                 "cocok": ";".join(cocok), "gap": ";".join(gap)})
P = pd.DataFrame(rows)
P.to_csv(EVAL_DIR / "penjelasan_profil_uji.csv", index=False)
print("Profil uji:", len(P), "| top-1 = karier iklan:", f"{(P.karier_top1 == P.karier_iklan).mean():.0%}")
P.head()

# Cell 7 — Tugas 2: prompt penjelasan (sama persis untuk RAG dan non-RAG)

In [ ]:
BAGIAN = ["basis_of_fit", "matched_competencies", "competencies_to_improve", "development_advice"]
SISTEM_JELAS = (
    "You explain a career recommendation to an Informatics student of Politeknik Negeri Jakarta (PNJ). "
    "The ranking and the fit score were computed by the system; never change or recompute them. "
    "Use ONLY the facts in SYSTEM DATA and in PASSAGES. Do not add courses, standards, units, numbers or skills "
    "that are not given there. Write short statements in the requested language. "
    "Each statement lists its sources: the passage identifiers it uses, and/or \"DATA\" for SYSTEM DATA. "
    "If no passages are given, rely on SYSTEM DATA only and do not mention any standard or occupation document. "
    "Sections: basis_of_fit (why this career fits), matched_competencies (skills the student already has), "
    "competencies_to_improve (the skill gaps, most important first), development_advice (how to close the gaps, "
    "including PNJ courses only when SYSTEM DATA lists them).")
SKEMA_JELAS = {"type": "object", "properties": {b: {"type": "array", "items": {"type": "object",
               "properties": {"statement": {"type": "string"}, "sources": {"type": "array", "items": {"type": "string"}}},
               "required": ["statement", "sources"]}} for b in BAGIAN}, "required": BAGIAN}

def data_sistem(p):
    c, cocok, gap = p["karier_top1"], [s for s in p["cocok"].split(";") if s], [s for s in p["gap"].split(";") if s]
    return {"career": c, "graduate_profile_career": bool(kat.set_index("karier").loc[c, "profil_lulusan"]),
            "fit_score_0_100": p["fit"], "job_ads_analysed_for_career": int(n_iklan.get(c, 0)),
            "curriculum_coverage_percent": float(cak.loc[c, "cakupan_%"]),
            "matched_skills": [{"skill": s, "percent_of_ads_requesting": PERSEN[(c, s)]} for s in cocok],
            "skill_gaps": [{"skill": s, "percent_of_ads_requesting": PERSEN[(c, s)],
                            "taught_in_pnj_courses": SKILL_MK.get(s, [])} for s in gap[:5]]}

def pesan(p, passages, bahasa="English"):
    teks_p = ("\n".join(f"[{r.chunk_id}] {r.teks[:1500]}" for r in passages.itertuples())
              if len(passages) else "(no passages)")
    return (f"Language: {bahasa}\n\nSYSTEM DATA:\n{json.dumps(data_sistem(p), ensure_ascii=False, indent=1)}"
            f"\n\nPASSAGES:\n{teks_p}")

def mock_jelas(p, passages):                                  # hanya uji lokal: jawaban buatan dengan 1 klaim keliru
    d = data_sistem(p); ids = list(passages["chunk_id"])
    m = d["matched_skills"][:2]; g = d["skill_gaps"][:3]
    return {"basis_of_fit": [{"statement": f"{d['career']} fits with a score of {d['fit_score_0_100']}.", "sources": ["DATA"]}]
            + ([{"statement": "This is described in SKKNI 282/2016 unit J.620100.004.01.", "sources": ids[:1] or ["DATA"]}]),
            "matched_competencies": [{"statement": f"You already have {x['skill']}, requested in {x['percent_of_ads_requesting']}% of ads.", "sources": ["DATA"]} for x in m],
            "competencies_to_improve": [{"statement": f"Improve {x['skill']} ({x['percent_of_ads_requesting'] + 3}% of ads).", "sources": ["DATA"]} for x in g],
            "development_advice": [{"statement": f"Take the course {x['taught_in_pnj_courses'][0]} for {x['skill']}.", "sources": ["DATA"]}
                                   for x in g if x["taught_in_pnj_courses"]][:1]}

def jelaskan(p, kondisi):
    cocok, gap = [s for s in p["cocok"].split(";") if s], [s for s in p["gap"].split(";") if s]
    passages = ambil(p["karier_top1"], cocok, gap) if kondisi == "rag" else kb.iloc[0:0]
    out = panggil_llm("penjelasan", SISTEM_JELAS, pesan(p, passages), SKEMA_JELAS, lambda: mock_jelas(p, passages))
    return out, list(passages["chunk_id"])

# Cell 8 — Hasilkan 120 penjelasan (±15–20 menit karena jeda kuota; bisa dilanjutkan bila terputus)

In [ ]:
if "_tunggu_asli" not in globals():
    _tunggu_asli = _tunggu

def _tunggu(e, coba):
    d = _tunggu_asli(e, coba)
    print(f"   [coba {coba}] tunggu {d:.0f} dtk ← {str(e)[:250]}", flush=True)
    return d

In [ ]:
HASIL = []
for i, p in P.iterrows():
    for kondisi in ["rag", "non_rag"]:
        out, ids = jelaskan(p, kondisi)
        HASIL.append({"pid": p["pid"], "kondisi": kondisi, "passage": ids, "jawaban": out})
    print(f"{i + 1}/{len(P)} profil selesai (pid={p['pid']})", flush=True)
with open(EVAL_DIR / "penjelasan_hasil.jsonl", "w", encoding="utf-8") as f:
    for h in HASIL:
        f.write(json.dumps({**h, "model": MODEL, "prompt_versi": PROMPT_VERSI, "mode_retrieval": MODE_RETRIEVAL},
                           ensure_ascii=False) + "\n")
print("Penjelasan:", len(HASIL), "| passage rata-rata (RAG):",
      round(np.mean([len(h["passage"]) for h in HASIL if h["kondisi"] == "rag"]), 2))

In [ ]:
import requests
from google.colab import userdata

r = requests.post(
    "https://api.groq.com/openai/v1/chat/completions",
    headers={"Authorization": f"Bearer {userdata.get('GROQ_API_KEY')}"},
    json={"model": MODEL, "messages": [{"role": "user", "content": "ping"}], "max_tokens": 5},
    timeout=30,
)
print("Status:", r.status_code)
if r.status_code == 429:
    print(r.json()["error"]["message"])
else:
    for k, v in r.headers.items():
        if k.startswith("x-ratelimit"):
            print(k, "=", v)

In [ ]:
HASIL = [json.loads(l) for l in open(EVAL_DIR / "penjelasan_hasil.jsonl", encoding="utf-8")]
assert len(HASIL) == 120 and {h["model"] for h in HASIL} == {MODEL}
HASIL = [{k: h[k] for k in ("pid", "kondisi", "passage", "jawaban")} for h in HASIL]
print("Dimuat:", len(HASIL), "penjelasan | model:", MODEL)

In [ ]:
import shutil
for f in [EVAL_DIR / "penjelasan_cek_klaim.csv", LOG_DIR / "tahap12_log.json"]:
    shutil.copy(f, f.with_name(f.stem + "_v1_sebelum_koreksi" + f.suffix))
print("Cadangan hasil sebelum koreksi tersimpan.")

# Cell 9 — Pemeriksa klaim otomatis (Bab III 3.6.5)

In [ ]:
# Klaim yang diperiksa: (a) status skill, (b) angka persentase/jumlah iklan, (c) mata kuliah, (d) standar/unit,
# (e) sitasi passage. Pernyataan tanpa klaim yang dapat diperiksa dicatat sebagai "tak terperiksa".
KODE_UNIT = re.compile(r"\b(J\.\d{6}\.\d{3}\.\d{2}|[JK]\.\d{2}[A-Z0-9]{3}\d{2}\.\d{2,3}\.\d|ICA[A-Z]{3}\d{3}[A-Z]?|ICT[A-Z]{3}\d{3}[A-Z]?|TIK\.[A-Z]{3}\d{4}|\d{6}(?= -))")
SKKNI_NO = re.compile(r"SKKNI\s*(?:No\.?\s*|Number\s*)?(\d{1,3})\s*/\s*(\d{4})", re.I)
PERSEN_RE = re.compile(r"(\d{1,3}(?:[.,]\d)?)\s?%")
JUMLAH_IKLAN = re.compile(r"(\d[\d,.]*)\s+(?:job\s+)?(?:advertisements|ads|iklan|lowongan)", re.I)
MK_SEBUT = re.compile(r"(?:course|courses|mata kuliah)\s+(?:on |in |called |titled )?[\"'“‘]?([A-Z][A-Za-z&/+\- ]{3,50})", re.U)
TEKS_KB = " ".join(kb["judul"]) + " " + " ".join(kb["teks"])
NOMOR_SKKNI = {(int(a), b) for a, b in re.findall(r"SKKNI\s+(\d{1,3})/(\d{4})", " ".join(kb["judul"]))}
mk_norm = {m.lower(): m for m in NAMA_MK}

NAMA_SKILL = {s: re.compile(r"(?<![A-Za-z0-9])" + re.escape(s) + r"(?![A-Za-z0-9])", 0 if len(s) <= 3 else re.I) for s in SKILL}
def skill_disebut(teks):
    # Nama karier bukan klaim skill; toleran terhadap spasi di sekitar "/" dan huruf besar-kecil
    for c in KARIER:
        teks = re.sub(re.escape(c).replace(r"\ /\ ", r"\s*/\s*"), " ", teks, flags=re.I)
    # Skill bernama panjang diperiksa dulu; skill pendek di dalam teks skill panjang
    # (React di "React Native", SQL di "SQL Server", REST API di "API Testing") tidak dihitung lagi
    terpakai, hasil = [], set()
    for s in sorted(SKILL, key=len, reverse=True):
        for m in [*POLA[s].finditer(teks), *NAMA_SKILL[s].finditer(teks)]:
            if not any(a <= m.start() and m.end() <= b for a, b in terpakai):
                terpakai.append((m.start(), m.end()))
                hasil.add(s)
                break
    return [s for s in SKILL if s in hasil]

def periksa(h, p):
    c = p["karier_top1"]
    S, cocok, gap = set(p["skill_profil"].split(";")), set(p["cocok"].split(";")) - {""}, set(p["gap"].split(";")) - {""}
    nilai_ok = {round(v, 1) for (cc, s), v in PERSEN.items() if cc == c} | {float(cak.loc[c, "cakupan_%"]), float(p["fit"])}
    klaim, tak, sitasi_salah = [], 0, 0
    for bag in BAGIAN:
        for it in h["jawaban"].get(bag, []):
            t, n0 = it.get("statement", ""), len(klaim)
            for src in it.get("sources", []):
                if src != "DATA" and src not in h["passage"]:
                    sitasi_salah += 1
            sk = skill_disebut(t)
            for s in sk:                                                     # (a) status skill
                ok = (s in cocok or s in S) if bag == "matched_competencies" else \
                     (s in gap) if bag == "competencies_to_improve" else (s in S or s in W[c] or s in SOFT)
                klaim.append(("skill", bag, s, ok))
            for x in PERSEN_RE.findall(t):                                   # (b) persentase
                v = float(x.replace(",", "."))
                klaim.append(("angka", bag, f"{v}%", any(abs(v - o) <= 0.6 for o in nilai_ok)))
            for x in JUMLAH_IKLAN.findall(t):
                v = int(re.sub(r"[,.]", "", x))
                klaim.append(("angka", bag, f"{v} iklan", abs(v - int(n_iklan.get(c, -9))) <= 1))
            for x in MK_SEBUT.findall(t):                                    # (c) mata kuliah
                nama = next((mk_norm[m] for m in mk_norm if x.lower().startswith(m)), None)
                ok = nama is not None and (not sk or bool(MK_SKILL.get(nama, set()) & set(sk)))
                klaim.append(("mata_kuliah", bag, x.strip(), ok))
            for m in mk_norm:                                                # MK disebut tanpa kata "course"
                if m in t.lower() and not any(k[0] == "mata_kuliah" and k[2].lower().startswith(m) for k in klaim[n0:]):
                    klaim.append(("mata_kuliah", bag, mk_norm[m], not sk or bool(MK_SKILL.get(mk_norm[m], set()) & set(sk))))
            for a, b in SKKNI_NO.findall(t):                                 # (d) standar dan unit
                klaim.append(("standar", bag, f"SKKNI {a}/{b}", (int(a), b) in NOMOR_SKKNI))
            for kd in KODE_UNIT.findall(t):
                klaim.append(("standar", bag, kd, kd in TEKS_KB))
            if len(klaim) == n0:
                tak += 1
    n = len(klaim); ok = sum(k[3] for k in klaim)
    tulis_gap = {k[2] for k in klaim if k[0] == "skill" and k[1] == "competencies_to_improve"}
    top3 = [s for s in p["gap"].split(";") if s][:3]
    return {"pid": h["pid"], "kondisi": h["kondisi"], "n_klaim": n, "klaim_didukung": ok,
            "dukungan": ok / n if n else np.nan, "tak_terperiksa": tak, "sitasi_tidak_valid": sitasi_salah,
            "lengkap_4_bagian": all(len(h["jawaban"].get(b, [])) > 0 for b in BAGIAN),
            "gap_sesuai": tulis_gap <= gap and set(top3) <= tulis_gap,
            "klaim_salah": "; ".join(f"{k[0]}:{k[2]}" for k in klaim if not k[3])}

Pi = P.set_index("pid")
cek = pd.DataFrame([periksa(h, Pi.loc[h["pid"]]) for h in HASIL])
cek.to_csv(EVAL_DIR / "penjelasan_cek_klaim.csv", index=False)
cek.groupby("kondisi")[["n_klaim", "dukungan", "tak_terperiksa", "sitasi_tidak_valid", "lengkap_4_bagian", "gap_sesuai"]].mean().round(3)

# Cell 10 — RAG vs non-RAG: uji Wilcoxon dan kriteria (Tabel 3.13 baris 5)

In [ ]:
from scipy.stats import wilcoxon
pv = cek.pivot(index="pid", columns="kondisi", values="dukungan").dropna()
selisih = pv["rag"] - pv["non_rag"]
p_w = float(wilcoxon(pv["rag"], pv["non_rag"], zero_method="wilcox").pvalue) if (selisih != 0).any() else 1.0
ring = {"pasangan": int(len(pv)), "dukungan_rag": round(float(pv["rag"].mean()), 3),
        "dukungan_non_rag": round(float(pv["non_rag"].mean()), 3), "wilcoxon_p": p_w,
        "lengkap_rag": round(float(cek.loc[cek.kondisi == "rag", "lengkap_4_bagian"].mean()), 3),
        "gap_sesuai_rag": round(float(cek.loc[cek.kondisi == "rag", "gap_sesuai"].mean()), 3)}
ring["kriteria_RAG>=0.90_dan_>non_RAG_p<0.05"] = bool(ring["dukungan_rag"] >= 0.90 and ring["dukungan_rag"] > ring["dukungan_non_rag"] and p_w < 0.05)
ring["kriteria_lengkap>=90%"] = bool(ring["lengkap_rag"] >= 0.90)
print(json.dumps(ring, indent=2))

# Contoh klaim yang tidak didukung (untuk dibahas di Bab IV)
cek[cek["klaim_salah"] != ""][["pid", "kondisi", "klaim_salah"]].head(15)

# Cell 11 — Sampel 30 pasangan untuk penilaian manual tersamar (Bab III 3.6.5)

In [ ]:
rng = random.Random(SEED)
sampel = rng.sample(sorted(P["pid"]), 30)
H = {(h["pid"], h["kondisi"]): h for h in HASIL}
baris, kunci = [], []
for pid in sampel:
    urutan = ["rag", "non_rag"]; rng.shuffle(urutan)                 # A/B diacak agar penilai tidak tahu kondisinya
    for label, kond in zip("AB", urutan):
        teks = "\n".join(f"{b}: " + " | ".join(i["statement"] for i in H[(pid, kond)]["jawaban"].get(b, [])) for b in BAGIAN)
        baris.append({"pid": pid, "versi": label, "karier": Pi.loc[pid, "karier_top1"], "penjelasan": teks,
                      "klaim_tak_terperiksa_didukung_0_1": ""})
        kunci.append({"pid": pid, "versi": label, "kondisi": kond})
pd.DataFrame(baris).to_csv(EVAL_DIR / "penjelasan_sampel_manual.csv", index=False)
pd.DataFrame(kunci).to_csv(LOG_DIR / "penjelasan_sampel_manual_kunci.csv", index=False)   # dibuka setelah penilaian
print("Sampel manual:", len(baris), "penjelasan (30 pasangan) → 04_eval/penjelasan_sampel_manual.csv")

# Cell 12 — Simpan prompt untuk backend dan log

In [ ]:
json.dump({"prompt_versi": PROMPT_VERSI, "penyedia": PENYEDIA, "model": MODEL, "suhu": 0, "sistem_ekstraksi": SISTEM_EKSTRAK,
           "skema_ekstraksi": SKEMA_EKSTRAK, "sistem_penjelasan": SISTEM_JELAS, "skema_penjelasan": SKEMA_JELAS,
           "bagian": BAGIAN, "mode_retrieval_evaluasi": MODE_RETRIEVAL},
          open(ART_DIR / "prompts_v1.json", "w"), indent=2, ensure_ascii=False)
log = {"tanggal": datetime.datetime.now().isoformat(timespec="seconds"), "penyedia": PENYEDIA, "model": MODEL, "prompt_versi": PROMPT_VERSI,
       "mode_retrieval": MODE_RETRIEVAL, "tau": TAU, "profil_uji": int(len(P)), "penjelasan": int(len(HASIL)),
       "hasil": ring, "per_kondisi": cek.groupby("kondisi")[["n_klaim", "dukungan", "tak_terperiksa",
                                                            "sitasi_tidak_valid"]].mean().round(3).to_dict(orient="index")}
json.dump(log, open(LOG_DIR / "tahap12_log.json", "w"), indent=2, ensure_ascii=False, default=float)
print(json.dumps(log, indent=2, ensure_ascii=False, default=float))

---
# Tahap 12b — Rating manual tersamar + LLM penilai
*Sumber: `12b_rating_manual.ipynb`*

# Tahap 12b — Rating manual tersamar RAG vs non-RAG + LLM penilai kedua (20 Okt 2026)
Logika di `src/tahap12b_rating.py`. Notebook ini aman di-**Run all** kapan saja:
1. **Run pertama**: membuat Google Sheet `rating_manual_rag_v1` di `04_eval` (tab Petunjuk, Rating, Ulangan) dan menjalankan LLM penilai (±15 menit, hasil di-cache). Colab akan meminta izin akses Google (pilih akun PNJ).
2. Isi tab **Rating** (30 pasangan). Minimal 3 hari kemudian isi tab **Ulangan** (10 pasangan) tanpa melihat tab Rating.
3. **Run all lagi**: analisis berjalan hanya jika kedua tab lengkap. Kunci A/B baru dibuka di langkah ini.

Lembar yang sudah ada tidak pernah ditimpa. Skor LLM per versi sengaja tidak ditampilkan sebelum analisis agar penilaian tetap tersamar.

In [ ]:
import os, sys, json, importlib
from pathlib import Path
try:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_DIR, LOKAL = Path("/content/drive/MyDrive/Skripsi_RAG"), False
    os.system("pip install -q google-genai==2.31.0 groq==1.7.0")
except ImportError:
    PROJECT_DIR, LOKAL = Path(os.environ.get("SKRIPSI_DIR", ".")), True
FOLDER_ID_EVAL = "1KfOakWCtCul7msVhDV2BqNk5vMvMckjU"   # folder Skripsi_RAG/04_eval di Google Drive
PENYEDIA_JUDGE = "groq"    # 21 Okt: pindah dari Gemini (kuota harian kecil). Model dipilih otomatis, keluarga berbeda dari pembuat (gpt-oss-120b)
GANTI_MODEL_JUDGE = None   # biarkan None (isi nama model hanya bila ingin memaksa model tertentu)
sys.path.insert(0, str(PROJECT_DIR / "src"))
import tahap12b_rating as M
importlib.reload(M)
T = M.Tahap12b(PROJECT_DIR, folder_id_eval=FOLDER_ID_EVAL, lokal=LOKAL)
print(T.muat())

## 1. Lembar rating (Google Sheet)

In [ ]:
status = T.buat_lembar()
print("Status isian:", json.dumps(status, indent=1))

## 2. LLM sebagai penilai kedua (rubrik dan Fakta Sistem yang sama)

In [ ]:
try:
    print(T.penilai_llm(PENYEDIA_JUDGE, GANTI_MODEL_JUDGE))
except M.KuotaHabis as e:
    print("BERHENTI SEMENTARA:", e)

## 3. Analisis (hanya jika Rating dan Ulangan lengkap)

In [ ]:
log = T.analisis()
if log:
    print("KESIMPULAN:", log["kesimpulan_utama"])
    print(json.dumps({k: log[k] for k in ["manusia", "intra_penilai", "ar_vs_llm", "llm", "model_judge"] if k in log}, indent=1, ensure_ascii=False))

---
# Tahap 13b — Soft skill (O*NET, v2)
*Sumber: `13b_softskill_onet_v2.ipynb`*

# Tahap 13b — Soft skill O*NET v2b (20 Okt 2026)
Perubahan dari run 19 Okt (disetujui AR): "interpersonal skills" → *Social Perceptiveness*; "cross-functional team" → *Cooperation*; 5 perbaikan pola (v2b).
Logika ada di `src/tahap13_softskill.py`; label validasi pola di `04_eval/validasi_softskill_label_v2.csv`.
Jalankan **Runtime → Run all**. Log lama disimpan sebagai `logs/tahap13_log_v1b.json`.

In [ ]:
import os, sys, json, importlib
from pathlib import Path
try:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_DIR = Path("/content/drive/MyDrive/Skripsi_RAG")
except ImportError:
    PROJECT_DIR = Path(os.environ.get("SKRIPSI_DIR", "."))
sys.path.insert(0, str(PROJECT_DIR / "src"))
import tahap13_softskill as M
importlib.reload(M)
T = M.Tahap13(PROJECT_DIR)
print("Proyek:", PROJECT_DIR, "| kamus soft skill:", M.VERSI_KAMUS)

## 1. Data lowongan dan split Tahap 8

In [ ]:
print(T.muat_data())

## 2. Deteksi skill (teknis, bahasa, 29 soft skill O*NET)

In [ ]:
info, frek = T.deteksi()
print(info)
frek.rename("% iklan").to_frame().T

## 3. Validasi pola soft skill (presisi)

In [ ]:
print("Sampel:", T.sampel_validasi(), "cuplikan")
presisi, salah = T.presisi()
if presisi is None:
    print("File label belum ada di 04_eval.")
else:
    print("Presisi:", presisi["total"], "| n:", presisi["n"], "| tanpa label:", presisi["tanpa_label"])
    print("Per skill < 1:", {k: v for k, v in presisi["per_skill"].items() if v < 1})
salah

## 4. Demand dan bobot w(c,s) dengan soft skill (ambang 10%)

In [ ]:
ringkas, soft_karier = T.bobot()
for s, k in soft_karier.items():
    print(f"{s}: {len(k)} karier")
ringkas

## 5. Uji ranking pada data uji Tahap 8/10 (n = 1.766)

In [ ]:
makro, mikro, per_karier = T.uji_ranking()
print("Makro:", json.dumps(makro, indent=1))
print("Mikro:", json.dumps(mikro, indent=1))
per_karier

## 6. Uji statistik berpasangan terhadap v1

In [ ]:
print(json.dumps(T.uji_statistik(), indent=2))

## 7. Simpan log dan ekspor untuk web

In [ ]:
ringkas_log, web = T.simpan()
print(json.dumps(ringkas_log, indent=2))
print("Ekspor web:", web)

---
# Tahap 14 — Pembekuan model dan sensitivitas
*Sumber: `14_freeze_sensitivitas.ipynb`*

# Tahap 14 — Kunci model, sensitivitas ambang, integritas, cek label, manifest (20 Okt 2026)
Logika di `src/tahap14_freeze.py`. Tidak mengubah hasil Tahap 8–13. Aman di-**Run all** berulang:
- Run pertama membuat Google Sheet `cek_label_claude_v1` di `04_eval` (40 cuplikan soft skill + 30 passage τ, label Claude disembunyikan). Isi kolom `label_AR`, lalu Run all lagi untuk menghitung kesepakatan.
- 21 Okt: cek τ v1 tidak memuat skill kueri (kriteria tidak setara dengan pelabelan Tahap 11); dilaporkan sebagai keterbatasan. Lembar τ v2 tidak dibuat (keputusan AR).
- Jalankan ulang setelah Tahap 12b selesai agar manifest mencakup hasil rating.

In [ ]:
import os, sys, json, importlib
from pathlib import Path
try:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_DIR, LOKAL = Path("/content/drive/MyDrive/Skripsi_RAG"), False
except ImportError:
    PROJECT_DIR, LOKAL = Path(os.environ.get("SKRIPSI_DIR", ".")), True
FOLDER_ID_EVAL = "1KfOakWCtCul7msVhDV2BqNk5vMvMckjU"   # folder Skripsi_RAG/04_eval
sys.path.insert(0, str(PROJECT_DIR / "src"))
import tahap13_softskill, tahap12b_rating, tahap14_freeze as M
for m in (tahap13_softskill, tahap12b_rating, M):
    importlib.reload(m)
T = M.Tahap14(PROJECT_DIR, folder_id_eval=FOLDER_ID_EVAL, lokal=LOKAL)

## 1. Kunci model LLM final

In [ ]:
print(json.dumps(T.kunci_model(), indent=2, ensure_ascii=False))

## 2. Sensitivitas ambang demand (5–20%) dan ablasi IDF
Baris ambang 10% harus sama persis dengan hasil Tahap 13 (uji reproduksi).

In [ ]:
sens, repro = T.sensitivitas()
print("Reproduksi Tahap 13:", repro)
sens

## 2b. Sensitivitas normalisasi camelCase (21 Okt)
Tahap 8/13 memecah camelCase ("MySQL" → "My SQL") sebelum mencocokkan pola. Skill teknis dideteksi ulang pada teks asli, bobot dan profil uji dihitung ulang, lalu Hit@k dibandingkan. Bobot utama tidak diubah (keterbatasan, opsi A).

In [ ]:
cc, cc_skill = T.sensitivitas_camelcase()
print(cc.to_string(index=False))
cc_skill.head(12)

## 3. Pemeriksaan integritas data

In [ ]:
cek = T.integritas()
print("Semua lulus:", bool(cek["lulus"].all()))
cek

## 4. Cek sampel label buatan Claude (diisi AR)

In [ ]:
print(T.lembar_cek_label())
hasil_cek = T.analisis_cek_label()
if hasil_cek:
    print(json.dumps({k: {kk: vv for kk, vv in v.items() if kk != "beda"} for k, v in hasil_cek.items() if isinstance(v, dict)}, indent=1))

## 5. Manifest pembekuan data (SHA-256)

In [ ]:
print(json.dumps(T.manifest(), indent=1, ensure_ascii=False))